# ECSoC Autonomic Aging Analysis
## Autonomic Aging Database (PhysioNet, Schumann & Bär 2021) — N=1,121

### 研究設計
- **データ**: 健常ボランティア1,121名の安静時ECG + 連続無侵襲血圧 (単回記録、8–35分)
- **比較**: Cross-sectional, 15年齢群 (18–19歳 〜 85–92歳)
- **主要検定**: Kruskal-Wallis (群間差) + Spearman (線形トレンド) + 二次回帰 (U字/J字型トレンド)
- **理論的背景**: Okabe (2026) ECSoC Paper 2 の発達モデル改訂 (Section 6.4.1, 8.1) を
  より大規模・高解像度な年齢層構造で検証する

### 本notebookの位置づけ
本パイプラインは Sleep-EDF Temazepam within-subject notebook (EEG, 30秒エポック,
睡眠段階別比較) をベースに、心臓ECG・拍動スケールDFAに合わせて全面的に書き換えたものです。

**主な変更点 (元notebookとの対比)**

| 項目 | Temazepam notebook (元) | 本notebook (adapted) |
|---|---|---|
| 信号 | EEG (単一/2チャンネル) | ECG (単一チャンネル、R波検出→RR間隔) |
| ファイル形式 | EDF (mne) | WFDB (.dat/.hea, wfdbパッケージ) |
| エポック単位 | 時間 (30秒 = 3000サンプル @100Hz) | 拍動 (beat-indexed RR間隔系列) |
| DFAスケール | 短10–40サンプル/長80–400サンプル (時間ベース) | 短4–16拍/長16–64拍 (拍動ベース、ECSoC Paper 2と同一) |
| グループ変数 | 睡眠段階 (W/N1/N2/N3/REM) | 年齢群 (15群、18–92歳) |
| デザイン | Within-subject paired (Placebo vs Temazepam) | Cross-sectional (被験者間比較、単回記録) |
| 主要統計 | 対応ありWilcoxon | Kruskal-Wallis + Spearman + 二次回帰 |
| D_eff (多チャンネル次元) | EEG 2ch (Fpz-Cz + Pz-Oz) | **本notebookでは未実装** (ECG単チャンネルのため。BP同時記録を用いた拡張は今後の課題) |

### ECSoC指標 (定義はEEG版・ECSoC Paper 2と共通)
- **CHI** = 2(α₁ − α₂): スケール非対称性 (拍動スケール版)
- **α₁**: 短距離DFA指数 (4–16拍; 洞房結節の拍動間ゆらぎ、迷走神経性)
- **α₂**: 長距離DFA指数 (16–64拍; 圧受容器反射・液性調節)
- **R²**: DFAスケーリング適合度 (全域プールfit)
- **PhaseV**: R² < 0.93 (スケーリング崩壊)

### ファイル構成の前提
- Google Drive上に `download_autonomic_aging.ipynb` で取得したデータが存在すること
  (`MyDrive/PhysioNet/autonomic-aging-cardiovascular/` 以下に `.dat`/`.hea` + `subject-info.csv`)
- レコードはpseudonymized ID (0001–1121) で命名されたWFDB形式


In [ ]:
# ============================================================
# Step 1: 環境セットアップ & Google Drive マウント
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install wfdb scipy numpy pandas matplotlib seaborn openpyxl statsmodels -q

import os, glob, json, time, warnings, traceback
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress, spearmanr, kruskal, mannwhitneyu
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import wfdb
from wfdb import processing as wfdb_processing
warnings.filterwarnings('ignore')

print('ライブラリ読み込み完了')
print(f'wfdb version: {wfdb.__version__}')


In [ ]:
# ============================================================
# Step 2: パス設定
# ============================================================
DRIVE_ROOT       = '/content/drive/MyDrive'
AGING_ROOT        = os.path.join(DRIVE_ROOT,
    'PhysioNet/autonomic-aging-cardiovascular')  # download_autonomic_aging.ipynb の既定保存先
SUBJECT_INFO_CSV  = os.path.join(AGING_ROOT, 'subject-info.csv')
OUTPUT_DIR        = os.path.join(DRIVE_ROOT, 'ECSoC_AutonomicAging_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINT_CSV    = os.path.join(OUTPUT_DIR, 'checkpoint_subject_results.csv')
FINAL_CSV         = os.path.join(OUTPUT_DIR, 'all_subjects_ecsoc.csv')

# ECSoCパラメータ (拍動スケール; ECSoC Paper 2 と同一定義)
DFA_S_LO, DFA_S_HI = 4, 16     # 短距離: 4-16拍
DFA_L_LO, DFA_L_HI = 16, 64    # 長距離: 16-64拍
DFA_N_SCALES        = 12
PHASE_V_THR         = 0.93
MIN_BEATS_REQUIRED  = DFA_L_HI * 2   # 128拍 (dfa関数の内部要件と一致させる)

# RRフィルタリング (生理的レンジ + 期外収縮除外)
RR_LO_MS, RR_HI_MS   = 300, 2000
RR_MAX_PCT_CHANGE    = 0.20

# 年齢群ラベル (subject-info.csv の Age_group 1-15 に対応; データセット説明文より)
AGE_GROUP_LABELS = {
    1: '18-19', 2: '20-24', 3: '25-29', 4: '30-34', 5: '35-39',
    6: '40-44', 7: '45-49', 8: '50-54', 9: '55-59', 10: '60-64',
    11: '65-69', 12: '70-74', 13: '75-79', 14: '80-84', 15: '85-92',
}
AGE_GROUP_MIDPOINT = {
    1: 18.5, 2: 22, 3: 27, 4: 32, 5: 37, 6: 42, 7: 47, 8: 52, 9: 57,
    10: 62, 11: 67, 12: 72, 13: 77, 14: 82, 15: 88.5,
}

# パイロット実行用サブサンプル設定
# None = 全1,121名を処理 (数時間かかる可能性あり)
# 整数を指定すると各年齢群からその人数だけランダム抽出してパイロット実行
SUBSAMPLE_N_PER_GROUP = None   # 例: 動作確認したいときは 10 などに変更

print(f'AGING_ROOT       : {AGING_ROOT}')
print(f'  存在            : {os.path.exists(AGING_ROOT)}')
print(f'SUBJECT_INFO_CSV  : {SUBJECT_INFO_CSV}')
print(f'  存在            : {os.path.exists(SUBJECT_INFO_CSV)}')
print(f'OUTPUT_DIR        : {OUTPUT_DIR}')
print(f'MIN_BEATS_REQUIRED: {MIN_BEATS_REQUIRED} 拍 (約{MIN_BEATS_REQUIRED*0.8/60:.1f}分 @ 心拍75bpm換算)')


In [ ]:
# ============================================================
# Step 3: subject-info.csv 読み込み (列名の揺れに頑健に対応)
# ============================================================
df_info_raw = pd.read_csv(SUBJECT_INFO_CSV)
print('subject-info.csv 列名 (元):', list(df_info_raw.columns))

def find_col(columns, keywords):
    '''列名候補からキーワードに合致する最初の列名を返す (大文字小文字無視)'''
    for kw in keywords:
        for c in columns:
            if kw.lower() in c.lower():
                return c
    return None

cols = list(df_info_raw.columns)
col_id     = find_col(cols, ['id'])
col_age    = find_col(cols, ['age_group', 'agegroup', 'age group', 'age'])
col_sex    = find_col(cols, ['sex', 'gender'])
col_bmi    = find_col(cols, ['bmi'])
col_device = find_col(cols, ['device'])

print(f'マッピング: ID={col_id}, AgeGroup={col_age}, Sex={col_sex}, BMI={col_bmi}, Device={col_device}')

assert col_id is not None and col_age is not None, \
    '❌ ID列 または AgeGroup列が見つかりません。subject-info.csvの列名を確認し、' \
    'find_colのキーワードリストを手動で調整してください。'

rename_map = {col_id: 'subject_id', col_age: 'age_group'}
if col_sex:    rename_map[col_sex]    = 'sex'
if col_bmi:    rename_map[col_bmi]    = 'bmi'
if col_device: rename_map[col_device] = 'device'

df_info = df_info_raw.rename(columns=rename_map).copy()

# subject_id を4桁ゼロ埋め文字列に統一 (0001-1121形式)
df_info['subject_id'] = df_info['subject_id'].astype(str).str.extract(r'(\d+)')[0].str.zfill(4)
# age_group を数値化 (pd.to_numeric なら欠損/非数値文字列も NaN として安全に検出できる)
df_info['age_group'] = pd.to_numeric(df_info['age_group'], errors='coerce')

n_missing_age = df_info['age_group'].isna().sum()
if n_missing_age > 0:
    print(f'⚠️ age_group が欠損/非数値の被験者: {n_missing_age}件')
    print(df_info.loc[df_info['age_group'].isna()])
    print('  → 年齢群が特定できないため、これらの被験者は以降の解析から除外します。')
    df_info = df_info[df_info['age_group'].notna()].reset_index(drop=True)

df_info['age_group'] = df_info['age_group'].astype(int)

# AGE_GROUP_LABELS/MIDPOINT の定義範囲 (1-15) 外の値がないか確認
unexpected_groups = sorted(set(df_info['age_group'].unique()) - set(AGE_GROUP_LABELS.keys()))
if unexpected_groups:
    print(f'⚠️ AGE_GROUP_LABELS の想定範囲 (1-15) 外の age_group 値: {unexpected_groups}')

df_info['age_label']  = df_info['age_group'].map(AGE_GROUP_LABELS)
df_info['age_mid']    = df_info['age_group'].map(AGE_GROUP_MIDPOINT)

print(f'\n被験者数: {len(df_info)}')
print(f'年齢群分布:')
print(df_info['age_group'].value_counts().sort_index())
df_info.head()


In [ ]:
# ============================================================
# Step 4: レコードファイル (.hea/.dat) の対応表を構築
# ============================================================
# PhysioNetの規約に従い、まず RECORDS インデックスファイルを探す。
# 見つからない場合は再帰globでフォールバック。

records_index_path = None
for cand in glob.glob(os.path.join(AGING_ROOT, '**', 'RECORDS'), recursive=True):
    records_index_path = cand
    break

record_id_to_path = {}

if records_index_path:
    print(f'RECORDSインデックス発見: {records_index_path}')
    with open(records_index_path) as f:
        rec_lines = [l.strip() for l in f if l.strip()]
    base_dir = os.path.dirname(records_index_path)
    for rec in rec_lines:
        rid = os.path.basename(rec).replace('.hea', '')
        rid_digits = ''.join([c for c in rid if c.isdigit()]).zfill(4)
        record_id_to_path[rid_digits] = os.path.join(base_dir, rec)
else:
    print('RECORDSインデックスなし。.heaファイルを再帰globで探索します (数分かかる場合あり)...')
    hea_files = glob.glob(os.path.join(AGING_ROOT, '**', '*.hea'), recursive=True)
    print(f'  見つかった.heaファイル数: {len(hea_files)}')
    for hea in hea_files:
        rid = os.path.basename(hea).replace('.hea', '')
        rid_digits = ''.join([c for c in rid if c.isdigit()]).zfill(4)
        # wfdb.rdrecord は拡張子なしのパスを要求する
        record_id_to_path[rid_digits] = hea.replace('.hea', '')

print(f'\nマッピング済みレコード数: {len(record_id_to_path)}')

# subject-info.csv とレコードファイルの対応を確認
df_info['record_path'] = df_info['subject_id'].map(record_id_to_path)
n_matched = df_info['record_path'].notna().sum()
n_missing = df_info['record_path'].isna().sum()
print(f'subject-info.csvとマッチしたレコード: {n_matched} / {len(df_info)}')
if n_missing > 0:
    print(f'⚠️ 対応レコードなし: {n_missing}件 (先頭5件の subject_id):')
    print(df_info[df_info['record_path'].isna()]['subject_id'].head().tolist())

df_info_valid = df_info[df_info['record_path'].notna()].reset_index(drop=True)
print(f'\n解析対象被験者数: {len(df_info_valid)}')


## コア計算関数

以下は元のTemazepam notebookの `dfa_ecsoc()` をベースに、
**時間スケール(サンプル数)→拍動スケール(拍数)** に変更したものです。
アルゴリズム本体(セグメント分割・最小二乗トレンド除去・log-log回帰)は完全に同一で、
ECSoC Paper 2 の心臓解析パイプラインと数値的に整合するよう設計しています。

処理フロー: `WFDBレコード読込 → ECGチャンネル特定 → R波検出(xqrs) → RR間隔抽出 →
期外収縮フィルタ → 拍動スケールDFA → CHI/R²/PhaseV`


In [ ]:
# ============================================================
# Step 5: ECSoCコア計算関数
# ============================================================

def dfa_ecsoc_beats(rr, s_lo=DFA_S_LO, s_hi=DFA_S_HI,
                     l_lo=DFA_L_LO, l_hi=DFA_L_HI, n_scales=DFA_N_SCALES):
    '''
    拍動スケール二領域DFA。
    rr: 1D array, RR間隔 (ms), 長さ=拍数のインデックス系列
    元のdfa_ecsoc() (EEG/サンプルスケール版) とアルゴリズムは完全に同一。
    窓をサンプル数ではなく拍数で定義する点のみが異なる。
    '''
    x = np.asarray(rr, dtype=np.float64)
    if len(x) < l_hi * 2:
        return None

    y = np.cumsum(x - x.mean())

    scales_s   = np.unique(np.round(np.geomspace(s_lo, s_hi, n_scales)).astype(np.int32))
    scales_l   = np.unique(np.round(np.geomspace(l_lo, l_hi, n_scales)).astype(np.int32))
    scales_all = np.unique(np.concatenate([scales_s, scales_l]))

    log_s_list, log_F_list = [], []
    for s in scales_all:
        s = int(s)
        n_seg = len(y) // s
        if n_seg < 4:
            continue
        segs = y[:n_seg * s].reshape(n_seg, s)
        t     = np.arange(s, dtype=np.float64)
        t_c   = t - t.mean()
        t_var = (t_c ** 2).sum()
        seg_c      = segs - segs.mean(axis=1, keepdims=True)
        slopes     = (seg_c * t_c).sum(axis=1) / t_var
        intercepts = segs.mean(axis=1) - slopes * t.mean()
        trend      = slopes[:, None] * t + intercepts[:, None]
        rms = np.sqrt(((segs - trend) ** 2).mean(axis=1).mean())
        if np.isfinite(rms) and rms > 0:
            log_s_list.append(np.log2(s))
            log_F_list.append(np.log2(rms))

    log_s = np.array(log_s_list)
    log_F = np.array(log_F_list)
    if len(log_s) < 6:
        return None

    mask_s = (2**log_s >= s_lo) & (2**log_s <= s_hi)
    mask_l = (2**log_s >= l_lo) & (2**log_s <= l_hi)
    if mask_s.sum() < 3 or mask_l.sum() < 3:
        return None

    a1, _, r_s, _, _ = linregress(log_s[mask_s], log_F[mask_s])
    a2, _, r_l, _, _ = linregress(log_s[mask_l], log_F[mask_l])
    _,  _, r_a, _, _ = linregress(log_s,          log_F)

    alpha1 = abs(a1)
    alpha2 = abs(a2)
    return {
        'alpha1': alpha1,
        'alpha2': alpha2,
        'CHI':    2 * (alpha1 - alpha2),
        'R2':     r_a**2,
        'R2_s':   r_s**2,
        'R2_l':   r_l**2,
    }


def rr_ectopic_filter(rr_ms, lo=RR_LO_MS, hi=RR_HI_MS, max_pct_change=RR_MAX_PCT_CHANGE):
    '''生理的レンジ外・急激な変化 (期外収縮的) のRR間隔を除外'''
    rr = np.asarray(rr_ms, dtype=np.float64)
    keep = (rr >= lo) & (rr <= hi)
    rr_valid = rr[keep]
    if len(rr_valid) < 2:
        return rr_valid
    out = [rr_valid[0]]
    for v in rr_valid[1:]:
        prev = out[-1]
        if abs(v - prev) / prev <= max_pct_change:
            out.append(v)
    return np.array(out)


def find_ecg_channel(sig_names):
    '''信号名リストからECGチャンネル名を推定'''
    for name in sig_names:
        if 'ecg' in name.lower() or 'ekg' in name.lower():
            return name
    # フォールバック: 最初のチャンネル
    return sig_names[0] if sig_names else None


RESULT_FIELDS = ['subject_id', 'error', 'ecg_channel', 'fs', 'qrs_method',
                 'n_beats_raw', 'n_beats_clean', 'mean_RR_ms', 'SDNN_ms', 'RMSSD_ms',
                 'PhaseV', 'alpha1', 'alpha2', 'CHI', 'R2', 'R2_s', 'R2_l']

def _full_result(subject_id, error=None, **kwargs):
    # 修正: 全ての返り値パスで同一キー集合を保証する (チェックポイントCSVの列崩れ防止)
    # process_one_subjectのどのreturn文でも必ずこの関数を経由させ、
    # 未使用のフィールドはNoneのまま残すことで、バッチ処理時のDataFrame結合で
    # チャンクごとに列数が変わる (= CSVが壊れる) 問題を防ぐ。
    row = {f: None for f in RESULT_FIELDS}
    row['subject_id'] = subject_id
    row['error'] = error
    row.update(kwargs)
    return row


def process_one_subject(record_path, subject_id):
    '''
    1被験者分のWFDBレコードを処理してECSoC指標dictを返す。
    エラー時は {'error': str(e)} を返す (呼び出し側でスキップ判定)。
    '''
    try:
        record = wfdb.rdrecord(record_path)
    except Exception as e:
        return _full_result(subject_id, f'read_error: {e}')

    sig_names = record.sig_name
    ecg_ch = find_ecg_channel(sig_names)
    if ecg_ch is None:
        return _full_result(subject_id, 'no_ecg_channel')

    ch_idx = sig_names.index(ecg_ch)
    ecg_sig = record.p_signal[:, ch_idx]
    fs = record.fs

    if np.all(np.isnan(ecg_sig)) or np.nanstd(ecg_sig) == 0:
        return _full_result(subject_id, 'flat_or_nan_signal')

    # 欠損値を線形補間 (WFDBはNaNをそのまま持つことがある)
    if np.any(np.isnan(ecg_sig)):
        nans = np.isnan(ecg_sig)
        ecg_sig[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), ecg_sig[~nans])

    try:
        qrs_inds = wfdb_processing.xqrs_detect(sig=ecg_sig, fs=fs, verbose=False)
    except Exception as e:
        return _full_result(subject_id, f'qrs_detect_error: {e}')

    qrs_method = 'xqrs'
    # 修正: xqrsが不十分な場合、gqrsにフォールバック（学習区間アーティファクトによる検出失敗対策）
    if len(qrs_inds) < MIN_BEATS_REQUIRED:
        try:
            qrs_inds_gqrs = wfdb_processing.gqrs_detect(sig=ecg_sig, fs=fs)
            if len(qrs_inds_gqrs) >= MIN_BEATS_REQUIRED:
                qrs_inds = qrs_inds_gqrs
                qrs_method = 'gqrs_fallback'
        except Exception:
            pass

    if len(qrs_inds) < MIN_BEATS_REQUIRED:
        return _full_result(subject_id, f'too_few_beats ({len(qrs_inds)})',
                             n_beats_raw=len(qrs_inds))

    rr_raw = np.diff(qrs_inds) / fs * 1000.0  # ms
    rr_clean = rr_ectopic_filter(rr_raw)

    if len(rr_clean) < MIN_BEATS_REQUIRED:
        return _full_result(subject_id, f'too_few_beats_after_filter ({len(rr_clean)})',
                             n_beats_raw=len(rr_raw), n_beats_clean=len(rr_clean))

    dfa = dfa_ecsoc_beats(rr_clean)
    if dfa is None:
        return _full_result(subject_id, 'dfa_failed',
                             n_beats_raw=len(rr_raw), n_beats_clean=len(rr_clean))

    row = _full_result(
        subject_id, error=None,
        ecg_channel=ecg_ch, fs=fs, qrs_method=qrs_method,
        n_beats_raw=len(rr_raw), n_beats_clean=len(rr_clean),
        mean_RR_ms=float(np.mean(rr_clean)),
        SDNN_ms=float(np.std(rr_clean, ddof=1)),
        RMSSD_ms=float(np.sqrt(np.mean(np.diff(rr_clean)**2))),
        PhaseV=int(dfa['R2'] < PHASE_V_THR),
        **dfa
    )
    return row


print('コア関数定義完了')
print(f'  DFA短距離: {DFA_S_LO}-{DFA_S_HI}拍')
print(f'  DFA長距離: {DFA_L_LO}-{DFA_L_HI}拍')
print(f'  最小必要拍数: {MIN_BEATS_REQUIRED}拍')


In [ ]:
# ============================================================
# Step 6: 単一被験者テスト (動作確認)
# ============================================================
test_row = df_info_valid.iloc[0]
print(f"テスト被験者: ID={test_row['subject_id']}, "
      f"年齢群={test_row['age_group']} ({test_row['age_label']})")
print(f"レコードパス: {test_row['record_path']}")

t0 = time.time()
result_test = process_one_subject(test_row['record_path'], test_row['subject_id'])
elapsed = time.time() - t0

print(f'\n処理時間: {elapsed:.1f}秒')
print('結果:')
for k, v in result_test.items():
    print(f'  {k}: {v}')

if result_test.get('error') is None:
    print(f"\n推定全体処理時間 (N={len(df_info_valid)}): "
          f"約{elapsed * len(df_info_valid) / 3600:.1f}時間 (単純外挿、実際は信号長により変動)")


In [ ]:
# ============================================================
# Step 6.5: 診断 — n_beats_raw=0 の原因切り分け (チャンネル誤選択 vs 個別レコード不良)
# ============================================================
record_d = wfdb.rdrecord(test_row['record_path'])
print('sig_name:', record_d.sig_name)
print('units   :', record_d.units)
print('fs      :', record_d.fs, ' sig_len:', record_d.sig_len)

for i, name in enumerate(record_d.sig_name):
    s = record_d.p_signal[:, i]
    n_nan = np.isnan(s).sum()
    print(f'\nch{i} ({name}): n_nan={n_nan}/{len(s)}, '
          f'min={np.nanmin(s):.3f}, max={np.nanmax(s):.3f}, std={np.nanstd(s):.3f}')
    s_interp = s.copy()
    if n_nan > 0:
        nans = np.isnan(s_interp)
        s_interp[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), s_interp[~nans])
    try:
        qrs = wfdb_processing.xqrs_detect(sig=s_interp, fs=record_d.fs, verbose=False)
        print(f'  → xqrs検出数: {len(qrs)}拍')
    except Exception as e:
        print(f'  → xqrs失敗: {e}')

# 系統的バグか個別レコードの問題かの切り分け: 他の被験者でも同様に0拍になるか確認
print('\n--- 他の被験者でも確認 ---')
for idx in range(1, min(4, len(df_info_valid))):
    row_i = df_info_valid.iloc[idx]
    res_i = process_one_subject(row_i['record_path'], row_i['subject_id'])
    print(f"  ID={row_i['subject_id']}: error={res_i.get('error')}, "
          f"n_beats_raw={res_i.get('n_beats_raw')}")


In [ ]:
# ============================================================
# Step 7: 単一被験者のRR系列 & DFA log-logプロットで目視確認
# ============================================================
record_t = wfdb.rdrecord(test_row['record_path'])
ecg_ch_t = find_ecg_channel(record_t.sig_name)
ecg_sig_t = record_t.p_signal[:, record_t.sig_name.index(ecg_ch_t)]
fs_t = record_t.fs

qrs_t = wfdb_processing.xqrs_detect(sig=np.nan_to_num(ecg_sig_t), fs=fs_t, verbose=False)
qrs_t = np.asarray(qrs_t, dtype=int)  # 修正: 空配列返却時のfloat64化に対応
qrs_method_t = 'xqrs'
# 修正: xqrsが不十分な場合、gqrsにフォールバック
if qrs_t.size < MIN_BEATS_REQUIRED:
    qrs_t_gqrs = np.asarray(wfdb_processing.gqrs_detect(sig=np.nan_to_num(ecg_sig_t), fs=fs_t), dtype=int)
    if qrs_t_gqrs.size >= MIN_BEATS_REQUIRED:
        qrs_t = qrs_t_gqrs
        qrs_method_t = 'gqrs_fallback'
if qrs_t.size == 0:
    print(f'⚠ R波が検出されませんでした: {test_row["record_path"]}')
else:
    print(f'検出手法: {qrs_method_t}, 検出数: {qrs_t.size}拍')
rr_raw_t = np.diff(qrs_t) / fs_t * 1000.0
rr_clean_t = rr_ectopic_filter(rr_raw_t)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), facecolor='#FFFFFF')

# (a) ECG波形の一部
ax = axes[0]
seg = slice(0, min(int(fs_t * 10), len(ecg_sig_t)))  # 先頭10秒
ax.plot(np.arange(seg.stop) / fs_t, ecg_sig_t[seg], color='#3A6EA5', lw=0.8)
qrs_in_seg = qrs_t[qrs_t < seg.stop].astype(int)  # 修正: インデックス用にintを保証
ax.scatter(qrs_in_seg / fs_t, ecg_sig_t[qrs_in_seg], color='#D9534F', s=15, zorder=5, label='検出R波')
ax.set_title(f'ECG波形 (先頭10秒) — ch={ecg_ch_t}')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude')
ax.legend(fontsize=8)

# (b) RR間隔タコグラム
ax = axes[1]
ax.plot(rr_clean_t, color='#5CB85C', lw=0.8)
ax.set_title(f'RRタコグラム (フィルタ後, n={len(rr_clean_t)}拍)')
ax.set_xlabel('拍インデックス'); ax.set_ylabel('RR (ms)')

# (c) DFA log-log プロット
ax = axes[2]
x = rr_clean_t - rr_clean_t.mean()
y_cum = np.cumsum(x)
scales_s = np.unique(np.round(np.geomspace(DFA_S_LO, DFA_S_HI, DFA_N_SCALES)).astype(int))
scales_l = np.unique(np.round(np.geomspace(DFA_L_LO, DFA_L_HI, DFA_N_SCALES)).astype(int))
scales_all = np.unique(np.concatenate([scales_s, scales_l]))
log_s_plot, log_F_plot = [], []
for s in scales_all:
    n_seg = len(y_cum) // s
    if n_seg < 4:
        continue
    segs = y_cum[:n_seg*s].reshape(n_seg, s)
    t = np.arange(s, dtype=np.float64)
    t_c = t - t.mean(); t_var = (t_c**2).sum()
    seg_c = segs - segs.mean(axis=1, keepdims=True)
    slopes = (seg_c * t_c).sum(axis=1) / t_var
    intercepts = segs.mean(axis=1) - slopes * t.mean()
    trend = slopes[:,None]*t + intercepts[:,None]
    rms = np.sqrt(((segs-trend)**2).mean(axis=1).mean())
    if np.isfinite(rms) and rms > 0:
        log_s_plot.append(np.log2(s)); log_F_plot.append(np.log2(rms))
log_s_plot, log_F_plot = np.array(log_s_plot), np.array(log_F_plot)
mask_s = (2**log_s_plot >= DFA_S_LO) & (2**log_s_plot <= DFA_S_HI)
mask_l = (2**log_s_plot >= DFA_L_LO) & (2**log_s_plot <= DFA_L_HI)
ax.scatter(log_s_plot, log_F_plot, color='#5A6470', s=20, zorder=3)
if mask_s.sum() >= 2:
    a1,b1,_,_,_ = linregress(log_s_plot[mask_s], log_F_plot[mask_s])
    ax.plot(log_s_plot[mask_s], a1*log_s_plot[mask_s]+b1, color='#3A6EA5', lw=2,
            label=f'α₁={abs(a1):.3f} (短距離4-16拍)')
if mask_l.sum() >= 2:
    a2,b2,_,_,_ = linregress(log_s_plot[mask_l], log_F_plot[mask_l])
    ax.plot(log_s_plot[mask_l], a2*log_s_plot[mask_l]+b2, color='#D9534F', lw=2,
            label=f'α₂={abs(a2):.3f} (長距離16-64拍)')
ax.set_xlabel('log₂(s) [拍]'); ax.set_ylabel('log₂F(s)')
ax.set_title('DFA log-log fit')
ax.legend(fontsize=8)

for ax in axes:
    ax.set_facecolor('#FFFFFF')
plt.tight_layout()
sanity_fig_path = os.path.join(OUTPUT_DIR, 'sanity_check_single_subject.png')
fig.savefig(sanity_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'図保存: {sanity_fig_path}')


## バッチ処理

以下のセルは全被験者(または `SUBSAMPLE_N_PER_GROUP` で指定した人数)を順に処理します。

**チェックポイント機構**: 処理済み結果は一定件数ごとに `checkpoint_subject_results.csv` に
追記保存されます。Colabのセッションが切断されても、再実行すれば処理済みIDをスキップして
続きから再開できます。


In [ ]:
# ============================================================
# Step 8: バッチ処理 (チェックポイント/再開対応 + 並列化)
# ============================================================
from concurrent.futures import ThreadPoolExecutor, as_completed

CHECKPOINT_EVERY = 20  # 何人処理するごとにCSVへ追記保存するか
N_WORKERS = 4  # 並列ワーカー数。Colab無料枠は概ね2vCPUだが、
               # ボトルネックはDrive I/O待ち(ファイル読込)であるため、
               # コア数より多めでも有効。Drive側のレート制限を避けるため4程度を推奨。

# 処理対象リストの決定 (サブサンプル設定に応じる)
if SUBSAMPLE_N_PER_GROUP is not None:
    target_df = (df_info_valid.groupby('age_group', group_keys=False)
                 .apply(lambda g: g.sample(min(len(g), SUBSAMPLE_N_PER_GROUP), random_state=42)))
    print(f'パイロットモード: 各年齢群から最大{SUBSAMPLE_N_PER_GROUP}名 → 対象{len(target_df)}名')
else:
    target_df = df_info_valid
    print(f'全数処理モード: 対象{len(target_df)}名')

# 既存チェックポイントの読み込み (再開用)
# 修正: 列数不整合等でCSVが破損している場合、自動でバックアップして新規開始する
if os.path.exists(CHECKPOINT_CSV):
    try:
        df_done = pd.read_csv(CHECKPOINT_CSV, dtype={'subject_id': str})
        done_ids = set(df_done['subject_id'])
        print(f'チェックポイント発見: 処理済み {len(done_ids)}名 → スキップして再開します')
    except pd.errors.ParserError as e:
        backup_path = CHECKPOINT_CSV.replace('.csv', f'_corrupt_{int(time.time())}.csv')
        os.rename(CHECKPOINT_CSV, backup_path)
        print(f'⚠ チェックポイントCSVが破損していたためバックアップして新規開始します: {backup_path}')
        print(f'   (原因: {e})')
        df_done = pd.DataFrame()
        done_ids = set()
else:
    df_done = pd.DataFrame()
    done_ids = set()
    print('チェックポイントなし。新規に開始します')

remaining = target_df[~target_df['subject_id'].isin(done_ids)]
print(f'残り処理対象: {len(remaining)}名 (並列ワーカー数: {N_WORKERS})\n')

def _safe_process(sid, record_path):
    # ワーカースレッド内で例外を吸収し、辞書を返す (1件の失敗で全体を止めない)
    try:
        return process_one_subject(record_path, sid)
    except Exception as e:
        return {'subject_id': sid, 'error': f'unexpected_exception: {e}'}

buffer = []
n_done = 0
t_start = time.time()

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {
        executor.submit(_safe_process, row['subject_id'], row['record_path']): row['subject_id']
        for _, row in remaining.iterrows()
    }

    for future in as_completed(futures):
        res = future.result()
        buffer.append(res)
        n_done += 1

        if n_done % CHECKPOINT_EVERY == 0 or n_done == len(remaining):
            # 修正: 列集合・列順序をRESULT_FIELDSに固定してチャンク間の不整合を防ぐ
            df_buffer = pd.DataFrame(buffer).reindex(columns=RESULT_FIELDS)
            write_header = not os.path.exists(CHECKPOINT_CSV)
            df_buffer.to_csv(CHECKPOINT_CSV, mode='a', header=write_header, index=False)
            buffer = []
            elapsed = time.time() - t_start
            rate = n_done / elapsed if elapsed > 0 else 0
            eta_min = (len(remaining) - n_done) / rate / 60 if rate > 0 else float('nan')
            print(f'  [{n_done}/{len(remaining)}] '
                  f'経過={elapsed/60:.1f}分 処理速度={rate*60:.1f}件/分 残り推定={eta_min:.1f}分')

print(f'\nバッチ処理完了。チェックポイントファイル: {CHECKPOINT_CSV}')


In [ ]:
# ============================================================
# Step 9: 結果集約 & subject-infoとのマージ
# ============================================================
df_results = pd.read_csv(CHECKPOINT_CSV, dtype={'subject_id': str})
df_results = df_results.drop_duplicates(subset='subject_id', keep='last')

n_total   = len(df_results)
n_error   = df_results['error'].notna().sum()
n_success = n_total - n_error

print(f'処理済み総数: {n_total}')
print(f'  成功: {n_success}')
print(f'  エラー: {n_error}')
if n_error > 0:
    print('\nエラー内訳:')
    print(df_results[df_results['error'].notna()]['error']
          .str.replace(r'\(.*\)', '', regex=True).value_counts())

df_merged = df_results.merge(
    df_info_valid[['subject_id', 'age_group', 'age_label', 'age_mid', 'sex', 'bmi', 'device']],
    on='subject_id', how='left'
)
df_merged.to_csv(FINAL_CSV, index=False)
print(f'\n結果保存: {FINAL_CSV}')

df_valid_results = df_merged[df_merged['error'].isna()].copy()
print(f'\n有効CHI推定数: {len(df_valid_results)}')
print(f'年齢群別サンプル数:')
print(df_valid_results.groupby('age_group').size())
df_valid_results.head()


In [ ]:
# ============================================================
# Step 9.5: 品質管理 (QC) — 実測記録時間に基づくビート検出率チェック
# ============================================================
# 重要な修正: 当初、raw_hr_bpm の計算に全被験者共通の固定記録時間(約15分)を
# 仮定していたが、実際の記録時間は被験者ごとに約8〜36分と大きく異なることが
# 判明した。固定値のままだと、長時間記録の被験者を誤って「頻脈」として除外し、
# 短時間記録の被験者の異常な低心拍を見逃す。ここでは各被験者の実際の記録時間
# (ヘッダのsig_len/fs)を都度読み込んで正しく計算する。
# (ヘッダ読み込みは信号本体を読まないため高速。1000名超でも数分程度)

record_path_map = df_info_valid.set_index('subject_id')['record_path'].to_dict()

durations_min = {}
for sid in df_valid_results['subject_id']:
    rp = record_path_map.get(sid)
    try:
        hdr = wfdb.rdheader(rp)
        durations_min[sid] = hdr.sig_len / hdr.fs / 60
    except Exception:
        durations_min[sid] = np.nan

df_valid_results['recording_min'] = df_valid_results['subject_id'].map(durations_min)
df_valid_results['raw_hr_bpm']  = df_valid_results['n_beats_raw'] / df_valid_results['recording_min']
df_valid_results['reject_rate'] = 1 - df_valid_results['n_beats_clean'] / df_valid_results['n_beats_raw']

# QC基準: 生理的にありえない検出心拍数、または期外収縮フィルタでの高すぎる除去率
QC_HR_LO_BPM, QC_HR_HI_BPM, QC_REJECT_MAX = 40, 150, 0.40
qc_flag = (
    (df_valid_results['raw_hr_bpm'] < QC_HR_LO_BPM) |
    (df_valid_results['raw_hr_bpm'] > QC_HR_HI_BPM) |
    (df_valid_results['reject_rate'] > QC_REJECT_MAX)
)
df_valid_results['qc_excluded'] = qc_flag

df_clean = df_valid_results[~qc_flag].copy()

print('記録時間(分)の分布:')
print(df_valid_results['recording_min'].describe())
print(f'\nQC除外: {int(qc_flag.sum())}/{len(df_valid_results)} ({100*qc_flag.mean():.2f}%)')
print(f'QC後の解析対象 (df_clean): n={len(df_clean)}')

qc_by_group = (df_valid_results.groupby('age_group')
    .agg(n_pre_qc=('subject_id', 'count'), n_excluded=('qc_excluded', 'sum'))
    .reset_index())
qc_by_group['excluded_pct'] = 100 * qc_by_group['n_excluded'] / qc_by_group['n_pre_qc']
print('\n年齢群別QC除外率:')
print(qc_by_group.to_string(index=False))

qc_clean_path = os.path.join(OUTPUT_DIR, 'qc_clean_subjects.csv')
df_clean.to_csv(qc_clean_path, index=False)
print(f'\nQC後データ保存: {qc_clean_path}')
print('\n⚠ 以降のStep 10〜14はすべてdf_clean (QC後) を使用します。df_valid_resultsは')
print('  QC前の生データとして残していますが、記述統計・検定には使用しないでください。')


In [ ]:
# ============================================================
# Step 10: 年齢群別サマリーテーブル
# ============================================================
summary_rows = []
for ag in sorted(df_clean['age_group'].unique()):
    sub = df_clean[df_clean['age_group'] == ag]
    summary_rows.append({
        'age_group':     ag,
        'age_label':     AGE_GROUP_LABELS[ag],
        'n':             len(sub),
        'CHI_mean':      sub['CHI'].mean(),
        'CHI_sd':        sub['CHI'].std(),
        'alpha1_mean':   sub['alpha1'].mean(),
        'alpha2_mean':   sub['alpha2'].mean(),
        'R2_mean':       sub['R2'].mean(),
        'PhaseV_rate':   sub['PhaseV'].mean(),
        'SDNN_mean_ms':  sub['SDNN_ms'].mean(),
    })

df_age_summary = pd.DataFrame(summary_rows)
df_age_summary['CHI_se'] = df_age_summary['CHI_sd'] / np.sqrt(df_age_summary['n'])
df_age_summary_path = os.path.join(OUTPUT_DIR, 'age_group_summary.csv')
df_age_summary.to_csv(df_age_summary_path, index=False)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(df_age_summary.to_string(index=False))
print(f'\n保存: {df_age_summary_path}')


## 統計検定

ECSoC Paper 2 (Section 6.4.1, 8.1) の「改訂発達モデル」は、CHIの加齢プロファイルが
**単調ではなくU字/J字型**であると予測しています(若年で臨界点近傍=低CHI、
中高年で超臨界的安定化=CHI上昇)。Fantasia データベース(N=40, 2群のみ)では
この形状を検証する解像度がありませんでした。

本データセット(15年齢群、N=1,121)では、以下の3種類の検定でこの予測を検証します。

1. **Kruskal-Wallis検定**: 15年齢群間に有意な差があるか (ノンパラメトリック群間比較)
2. **Spearman順位相関 (線形トレンド)**: 年齢群と CHI の単調な関係の強さ
3. **二次回帰 (U字/J字検定)**: CHI ~ age + age² を当てはめ、二次項の符号と有意性を確認
   - 二次項が正で有意 → U字型 (Paper 2の予測と整合)
   - 二次項が有意でない → 単調、またはより複雑な形状


In [ ]:
# ============================================================
# Step 11: 統計検定 (群間差・線形トレンド・U字/J字型検定)
# ============================================================
groups_for_kw = [df_clean[df_clean['age_group']==ag]['CHI'].dropna().values
                  for ag in sorted(df_clean['age_group'].unique())]
groups_for_kw = [g for g in groups_for_kw if len(g) > 0]

kw_stat, kw_p = kruskal(*groups_for_kw)
print('=== 1. Kruskal-Wallis検定 (15年齢群間のCHI差) ===')
print(f'  H = {kw_stat:.3f}, p = {kw_p:.3e}, k = {len(groups_for_kw)}群')

age_num = df_clean['age_mid'].values
chi_val = df_clean['CHI'].values
valid_mask = ~(np.isnan(age_num) | np.isnan(chi_val))
age_num, chi_val = age_num[valid_mask], chi_val[valid_mask]

rho, p_rho = spearmanr(age_num, chi_val)
print(f'\n=== 2. Spearman順位相関 (年齢 vs CHI, 線形トレンド) ===')
print(f'  rho = {rho:+.3f}, p = {p_rho:.3e}, n = {len(age_num)}')

# 二次回帰 (中心化してから当てはめ、多重共線性を回避)
age_c = age_num - age_num.mean()
coeffs = np.polyfit(age_c, chi_val, 2)     # [quad, linear, intercept]
pred = np.polyval(coeffs, age_c)
ss_res = np.sum((chi_val - pred)**2)
ss_tot = np.sum((chi_val - chi_val.mean())**2)
r2_quad = 1 - ss_res / ss_tot

# 二次項の有意性は線形モデルとのF検定で評価
from scipy.stats import f as f_dist
coeffs_lin = np.polyfit(age_c, chi_val, 1)
pred_lin = np.polyval(coeffs_lin, age_c)
ss_res_lin = np.sum((chi_val - pred_lin)**2)
n = len(chi_val)
df1, df2 = 1, n - 3
F_stat = ((ss_res_lin - ss_res) / df1) / (ss_res / df2)
p_quad = 1 - f_dist.cdf(F_stat, df1, df2)

print(f'\n=== 3. 二次回帰 (U字/J字型検定) ===')
print(f'  CHI ~ a*(age-mean)^2 + b*(age-mean) + c')
print(f'  a (二次項係数) = {coeffs[0]:+.6f}  {"[U字型: 正]" if coeffs[0]>0 else "[逆U字型: 負]"}')
print(f'  b (線形項係数) = {coeffs[1]:+.6f}')
print(f'  c (切片)       = {coeffs[2]:+.6f}')
print(f'  R² (二次モデル) = {r2_quad:.4f}')
print(f'  二次項の有意性 (線形モデルに対するF検定): F({df1},{df2})={F_stat:.3f}, p={p_quad:.3e}')

vertex_age = age_num.mean() - coeffs[1] / (2 * coeffs[0]) if coeffs[0] != 0 else np.nan
if coeffs[0] > 0:
    print(f'  推定される谷(最小CHI)の年齢: 約{vertex_age:.1f}歳')
elif coeffs[0] < 0:
    print(f'  推定される山(最大CHI)の年齢: 約{vertex_age:.1f}歳')

print(f'\n=== 解釈 ===')
# 修正: 元のコードは `p_rho > 0` / `p_rho < 0` で分岐していたが、p値は常に非負のため
# 前者は常に真、後者は到達不能というバグだった。相関係数rhoの符号で判定するよう修正し、
# 有意な逆U字 (coeffs[0]<0, 本解析での実際の結果) の分岐も明示的に追加した。
# 注: 「U字型 = Paper2予測と整合的」という対応付けは元のコードの想定を踏襲しているが、
#     Paper2の改訂発達モデルが実際に予測する形状(U字か逆U字か)は要確認。
if p_quad < 0.05 and coeffs[0] > 0:
    print('  → U字型トレンドを支持 (要確認: Paper 2 改訂発達モデルの予測形状と一致するか)')
elif p_quad < 0.05 and coeffs[0] < 0:
    print('  → 逆U字型(山型)トレンドを支持: 若年〜中年で上昇し高齢で低下するパターン')
elif rho > 0 and p_rho < 0.05:
    print('  → 単調な正のトレンドを支持 (元の単純加齢仮説寄り)')
elif rho < 0 and p_rho < 0.05:
    print('  → 単調な負のトレンドを支持')
else:
    print('  → 明確なトレンドは検出されず。年齢群別サマリーテーブルを目視確認してください')

df_stat_summary = pd.DataFrame([
    {'test': 'Kruskal-Wallis (15群)', 'statistic': kw_stat, 'p_value': kw_p},
    {'test': 'Spearman (線形トレンド)', 'statistic': rho, 'p_value': p_rho},
    {'test': '二次項 F検定 (U字/J字)', 'statistic': F_stat, 'p_value': p_quad},
])
df_stat_summary.to_csv(os.path.join(OUTPUT_DIR, 'age_trend_statistics.csv'), index=False)


## Step 11a: 全コホートにおける三次項の要否検定 (CHI, alpha1, alpha2)

Step 11 の二次回帰 (CHI ~ age+age²) は「線形 vs 二次」のF検定のみを行っていた。
一方、女性サブサンプル (N=636) のStep 12iでは、alpha1について**二次→三次でAICが
大きく改善し (-234.84→-240.38)、三次項が統計的に必要 (p=0.0062)** という結果が出た。

これが女性サブサンプル特有の現象なのか、それとも全コホート (N=1,121, 男女込み) でも
再現される、より一般的な加齢曲線の特徴なのかを確認する。もし全コホートでも三次項が
必要なら、論文の「逆U字」という記述を「より複雑な非対称な加齢軌跡」に更新する根拠になる。

Step 12iと同一の定式化 (中心化age、statsmodels OLS + ネストF検定) を用いる。


In [ ]:
# ============================================================
# Step 11a: 全コホートでの二次 vs 三次モデル比較 (CHI, alpha1, alpha2)
# ============================================================
if 'df_clean' in globals():
    full_cubic_results = {}

    for dv in ['CHI', 'alpha1', 'alpha2']:
        df_dv = df_clean.dropna(subset=[dv, 'age_mid']).copy()
        df_dv['age_c'] = df_dv['age_mid'] - df_dv['age_mid'].mean()

        f_quad  = f'{dv} ~ age_c + I(age_c**2)'
        f_cubic = f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)'
        m_quad  = smf.ols(f_quad, data=df_dv).fit()
        m_cubic = smf.ols(f_cubic, data=df_dv).fit()
        nested = anova_lm(m_quad, m_cubic)
        p_cubic_term = nested['Pr(>F)'].iloc[1]

        print(f'\n\n=== {dv}: 全コホート 二次 vs 三次 (N={len(df_dv)}) ===')
        print(f'  二次モデル  AIC={m_quad.aic:.2f}, adj-R2={m_quad.rsquared_adj:.4f}')
        print(f'  三次モデル  AIC={m_cubic.aic:.2f}, adj-R2={m_cubic.rsquared_adj:.4f}')
        print(f'  三次項(age**3)の必要性 (ネストF検定): p={p_cubic_term:.4f}')
        if p_cubic_term < 0.05 and m_cubic.aic < m_quad.aic:
            print(f'  → {dv}: 三次項は統計的に必要。二次(単純な逆U字)モデルは')
            print('    加齢軌跡の形状を捉えきれていない可能性がある。')
        else:
            print(f'  → {dv}: 三次項の追加的な必要性は確認できない (二次モデルで十分)。')

        # 係数と外挿の目安として、頂点/変曲点情報も記録 (二次モデルの頂点年齢)
        vertex_age = df_dv['age_mid'].mean() - m_quad.params['age_c'] / (2 * m_quad.params['I(age_c ** 2)']) \
            if m_quad.params['I(age_c ** 2)'] != 0 else float('nan')

        full_cubic_results[dv] = {
            'N':                   len(df_dv),
            'p_cubic_term_needed': p_cubic_term,
            'quad_AIC':            m_quad.aic,
            'cubic_AIC':           m_cubic.aic,
            'quad_adjR2':          m_quad.rsquared_adj,
            'cubic_adjR2':         m_cubic.rsquared_adj,
            'quad_vertex_age':     vertex_age,
        }

    df_full_cubic_summary = pd.DataFrame(full_cubic_results).T
    df_full_cubic_summary.index.name = 'dv'
    print('\n\n=== サマリー: 全コホート 三次項要否検定 (CHI, alpha1, alpha2) ===')
    print(df_full_cubic_summary.to_string(float_format=lambda x: f'{x:.4f}'))

    print('\n\n=== 女性サブサンプル (Step 12i) との対比 ===')
    if 'df_cubic_summary' in globals():
        compare_rows = []
        for dv in ['CHI', 'alpha1', 'alpha2']:
            compare_rows.append({
                'dv': dv,
                'p_cubic_full_cohort': full_cubic_results[dv]['p_cubic_term_needed'],
                'p_cubic_female_only': df_cubic_summary.loc[dv, 'p_cubic_term_needed'],
            })
        df_compare = pd.DataFrame(compare_rows).set_index('dv')
        df_compare['replicates_in_full_cohort'] = (
            (df_compare['p_cubic_full_cohort'] < 0.05) & (df_compare['p_cubic_female_only'] < 0.05)
        )
        print(df_compare.to_string(float_format=lambda x: f'{x:.4f}'))
        compare_path = os.path.join(OUTPUT_DIR, 'full_cohort_vs_female_cubic_term_comparison.csv')
        df_compare.to_csv(compare_path)
        print(f'\n保存: {compare_path}')
    else:
        print('  df_cubic_summary (Step 12i) が未実行のため、対比表はスキップします。')
        print('  先にStep 12iを実行してから、このセルを再実行してください。')

    full_cubic_path = os.path.join(OUTPUT_DIR, 'full_cohort_cubic_term_test.csv')
    df_full_cubic_summary.to_csv(full_cubic_path)
    print(f'\n保存: {full_cubic_path}')
else:
    print('df_clean が見つからないため、全コホート三次項検定はスキップします (Step 9.5を先に実行してください)')


## Step 11b: 三次モデルの形状特定 (極値・変曲点) と可視化、群平均フィットとの対比

Step 11a で、全コホート・女性サブサンプルの両方において alpha1・alpha2 は三次項が
統計的に必要 (それぞれ p=0.0027, p=0.0001 [全コホート]) であることが再現された。
CHI は両サンプルで三次項不要 (p=0.95 [全コホート])。

ここでは、

1. 三次モデルの係数から極値 (local max/min) と変曲点を計算し、実際の年齢に変換する
   (「非対称な軌跡」を定量的に記述するため)。
2. 個体レベル回帰 (Step 11a の方式) と、15年齢群の**群平均への回帰**を並べて可視化する。
   これは Step 11a で見つかった CHI の頂点年齢 (44.9歳) が、Step 12e のマークダウンに
   記載された既知値 (~55-59歳) とズレている点を確認するため
   (個体レベルOLSは年齢群間のサンプルサイズ不均等の影響を受けるが、群平均への回帰は
   15群を均等に重み付けする)。


In [ ]:
# ============================================================
# Step 11b: 三次モデルの極値・変曲点、および個体レベルvs群平均フィットの対比
# ============================================================
if 'df_clean' in globals():
    shape_results = {}
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), facecolor='#FFFFFF')

    for ax, dv in zip(axes, ['CHI', 'alpha1', 'alpha2']):
        df_dv = df_clean.dropna(subset=[dv, 'age_mid']).copy()
        age_mean = df_dv['age_mid'].mean()
        df_dv['age_c'] = df_dv['age_mid'] - age_mean

        # --- 個体レベルの三次モデル (Step 11aと同一) ---
        m_cubic_ind = smf.ols(f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)', data=df_dv).fit()
        a = m_cubic_ind.params['I(age_c ** 3)']
        b = m_cubic_ind.params['I(age_c ** 2)']
        c = m_cubic_ind.params['age_c']
        d = m_cubic_ind.params['Intercept']

        # 極値: p'(x) = 3a x^2 + 2b x + c = 0
        extrema_c = np.roots([3*a, 2*b, c]) if a != 0 else np.array([])
        extrema_c = np.real(extrema_c[np.isreal(extrema_c)])
        age_min, age_max = df_dv['age_mid'].min(), df_dv['age_mid'].max()
        extrema_age = [age_mean + xc for xc in extrema_c if age_min <= age_mean + xc <= age_max]

        # 変曲点: p''(x) = 6a x + 2b = 0  ->  x = -b/(3a)
        inflection_age_raw = age_mean - b / (3*a) if a != 0 else np.nan
        # a(三次係数)がゼロに近い場合(CHIなど)、変曲点は観測範囲外へ発散しうる。
        # 発散した値をそのままaxvline/表示に使うとx軸が壊れるため、観測範囲外はNaNにする。
        if inflection_age_raw is not None and not np.isnan(inflection_age_raw) and (age_min <= inflection_age_raw <= age_max):
            inflection_age = inflection_age_raw
        else:
            inflection_age = np.nan

        # --- 群平均への回帰 (15年齢群を均等に重み付け) ---
        grp = df_dv.groupby('age_mid', as_index=False)[dv].mean()
        grp['age_c'] = grp['age_mid'] - grp['age_mid'].mean()
        coeffs_grp = np.polyfit(grp['age_c'], grp[dv], 2)  # [quad, lin, intercept]
        vertex_age_grp = grp['age_mid'].mean() - coeffs_grp[1] / (2*coeffs_grp[0]) if coeffs_grp[0] != 0 else np.nan

        print(f'\n=== {dv}: 形状特定 (N={len(df_dv)}) ===')
        print(f'  三次係数: a={a:.3e}, b={b:.3e}, c={c:.3e}, d={d:.4f}')
        print(f'  極値の年齢 (観測範囲内, 個体レベル三次モデル): {[f"{x:.1f}" for x in extrema_age]}')
        print(f'  変曲点の年齢: {inflection_age:.1f}')
        print(f'  [対比] 個体レベル二次モデルの頂点年齢:   {full_cubic_results[dv]["quad_vertex_age"]:.1f}歳' \
              if 'full_cubic_results' in globals() and dv in full_cubic_results else '')
        print(f'  [対比] 群平均(15群均等重み)二次モデルの頂点年齢: {vertex_age_grp:.1f}歳')

        shape_results[dv] = {
            'cubic_a': a, 'cubic_b': b, 'cubic_c': c, 'cubic_d': d,
            'inflection_age': inflection_age,
            'extrema_age_str': ','.join(f'{x:.1f}' for x in extrema_age),
            'quad_vertex_age_individual': full_cubic_results[dv]['quad_vertex_age']
                if 'full_cubic_results' in globals() and dv in full_cubic_results else np.nan,
            'quad_vertex_age_groupmean': vertex_age_grp,
        }

        # --- 可視化 ---
        ax.set_facecolor('#FFFFFF')
        ax.scatter(df_dv['age_mid'], df_dv[dv], s=8, alpha=0.15, color='#8899AA', label='個体データ')
        ax.scatter(grp['age_mid'], grp[dv], s=45, color='#3A6EA5', zorder=4, label='年齢群平均 (15群均等)')

        x_fit = np.linspace(age_min, age_max, 200)
        xc_fit = x_fit - age_mean
        y_cubic = a*xc_fit**3 + b*xc_fit**2 + c*xc_fit + d
        ax.plot(x_fit, y_cubic, color='#D9534F', lw=2, label='三次(個体レベル)')

        m_quad_ind = smf.ols(f'{dv} ~ age_c + I(age_c**2)', data=df_dv).fit()
        y_quad = np.polyval([m_quad_ind.params['I(age_c ** 2)'], m_quad_ind.params['age_c'],
                              m_quad_ind.params['Intercept']], xc_fit)
        ax.plot(x_fit, y_quad, color='#5CB85C', lw=1.6, ls='--', label='二次(個体レベル)')

        for xa in extrema_age:
            ax.axvline(xa, color='#D9534F', ls=':', lw=1, alpha=0.6)
        if not np.isnan(inflection_age):
            ax.axvline(inflection_age, color='#F0AD4E', ls=':', lw=1.2, alpha=0.8)
        ax.set_xlim(age_min - 1, age_max + 1)  # 発散した値がaxvline経由でx軸を壊さないための保険

        ax.set_title(dv, fontsize=11, color='#1A1A1A')
        ax.set_xlabel('年齢 (群中央値)', color='#1A1A1A')
        ax.grid(True, alpha=0.15)
        ax.legend(fontsize=6.5, facecolor='#FFFFFF', labelcolor='#1A1A1A')

    plt.tight_layout()
    shape_fig_path = os.path.join(OUTPUT_DIR, 'ecsoc_full_cohort_cubic_shape.png')
    plt.savefig(shape_fig_path, dpi=150, facecolor='#FFFFFF')
    plt.show()
    print(f'\n保存: {shape_fig_path}')

    df_shape_summary = pd.DataFrame(shape_results).T
    df_shape_summary.index.name = 'dv'
    print('\n\n=== サマリー: 三次モデルの形状特定 (CHI, alpha1, alpha2) ===')
    print(df_shape_summary.to_string(float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))
    shape_path = os.path.join(OUTPUT_DIR, 'full_cohort_cubic_shape_summary.csv')
    df_shape_summary.to_csv(shape_path)
    print(f'\n保存: {shape_path}')

    print('\n\n=== 個体レベル vs 群平均: 頂点年齢の一致度チェック ===')
    for dv in ['CHI', 'alpha1', 'alpha2']:
        vi = shape_results[dv]['quad_vertex_age_individual']
        vg = shape_results[dv]['quad_vertex_age_groupmean']
        if not (np.isnan(vi) or np.isnan(vg)):
            diff = abs(vi - vg)
            flag = ' ⚠ 5歳以上の乖離' if diff >= 5 else ''
            print(f'  {dv}: 個体レベル={vi:.1f}歳, 群平均={vg:.1f}歳, 差={diff:.1f}歳{flag}')
else:
    print('df_clean が見つからないため、形状特定はスキップします (Step 9.5, Step 11aを先に実行してください)')


## Step 11c: 制限三次スプラインによる alpha2 二峰構造の確認 (多項式アーティファクト排除)

Step 11b で、alpha2 の三次モデルは観測範囲内に**2つの極値** (極小 ~24.6歳, 極大 ~67.1歳) を
持つことがわかった。これは単純な「inverted-U」ではなく、非単調な軌跡を示唆する重要な所見だが、
三次多項式は係数がわずかでも極値の位置が敏感に動く("wiggle")ことがあるため、
別の関数形 (制限三次スプライン) でも同じ二峰パターンが再現されるかを確認する必要がある。

女性サブサンプルでは Step 12j で df=3 のスプラインを使ったが、全コホート (N=1032, `age_mid`
固有値15) では自由度に余裕があるため df=4 とする。

やること:
1. quadratic-only との比較でスプライン全体の説明力を確認 (参考情報)
2. スプライン予測を細かい年齢グリッドで評価し、符号変化から極値を数値的に検出
3. 三次モデルの極値 (Step 11b) と数値的に比較
4. 3指標をまとめて可視化 (二次・三次・スプラインを重ねる)


In [ ]:
# ============================================================
# Step 11c: 全コホートでの制限三次スプラインによる形状確認
# ============================================================
if 'df_clean' in globals():
    from patsy import dmatrix, build_design_matrices

    SPLINE_DF_FULL = 4  # 全コホート(N=1032, age_mid固有値15)向けにやや柔軟性を上げる
    spline_shape_results = {}
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), facecolor='#FFFFFF')

    for ax, dv in zip(axes, ['CHI', 'alpha1', 'alpha2']):
        df_dv = df_clean.dropna(subset=[dv, 'age_mid']).copy()
        age_mean = df_dv['age_mid'].mean()
        age_min, age_max = df_dv['age_mid'].min(), df_dv['age_mid'].max()
        df_dv['age_c'] = df_dv['age_mid'] - age_mean

        # --- スプライン基底の構築 (学習データのknotを保存し、予測時に再利用) ---
        spline_basis = dmatrix(f'cr(age_c, df={SPLINE_DF_FULL}) - 1', data=df_dv, return_type='dataframe')
        spline_cols = [f'age_spline_{i+1}' for i in range(spline_basis.shape[1])]
        spline_basis.columns = spline_cols
        df_dv = pd.concat([df_dv.reset_index(drop=True), spline_basis.reset_index(drop=True)], axis=1)

        f_spline = f'{dv} ~ ' + ' + '.join(spline_cols)
        m_spline = smf.ols(f_spline, data=df_dv).fit()

        f_quad = f'{dv} ~ age_c + I(age_c**2)'
        m_quad = smf.ols(f_quad, data=df_dv).fit()
        nested = anova_lm(m_quad, m_spline)
        p_spline_vs_quad = nested['Pr(>F)'].iloc[1]

        # --- 細かいグリッドでスプライン予測 → 極値を数値的に検出 ---
        x_grid = np.linspace(age_min, age_max, 400)
        xc_grid = x_grid - age_mean
        grid_basis = build_design_matrices([spline_basis.design_info], {'age_c': xc_grid})[0]
        grid_basis = np.asarray(grid_basis)
        y_spline_grid = m_spline.params['Intercept'] + grid_basis @ m_spline.params[spline_cols].values \
            if 'Intercept' in m_spline.params.index else grid_basis @ m_spline.params[spline_cols].values

        dy = np.diff(y_spline_grid)
        sign_change = np.where(np.diff(np.sign(dy)) != 0)[0]
        spline_extrema_age = [float(x_grid[i+1]) for i in sign_change]

        # --- 三次モデル(Step 11b)の極値と数値比較 ---
        m_cubic = smf.ols(f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)', data=df_dv).fit()
        a3, b3, c3 = m_cubic.params['I(age_c ** 3)'], m_cubic.params['I(age_c ** 2)'], m_cubic.params['age_c']
        cubic_extrema_c = np.roots([3*a3, 2*b3, c3]) if a3 != 0 else np.array([])
        cubic_extrema_c = np.real(cubic_extrema_c[np.isreal(cubic_extrema_c)])
        cubic_extrema_age = sorted([age_mean + xc for xc in cubic_extrema_c if age_min <= age_mean+xc <= age_max])

        print(f'\n=== {dv}: スプライン形状確認 (N={len(df_dv)}, df={SPLINE_DF_FULL}) ===')
        print(f'  スプライン vs 二次のみ: p={p_spline_vs_quad:.4f} (参考; AIC quad={m_quad.aic:.2f}, spline={m_spline.aic:.2f})')
        print(f'  スプライン極値 (数値検出, 観測範囲内): {[f"{x:.1f}" for x in spline_extrema_age]}')
        print(f'  三次モデル極値 (Step 11bと同一手法):     {[f"{x:.1f}" for x in cubic_extrema_age]}')
        n_spline_extrema = len(spline_extrema_age)
        replicated = n_spline_extrema == len(cubic_extrema_age) and n_spline_extrema >= 1
        print(f'  → 極値の個数が三次モデルと一致: {replicated}')

        spline_shape_results[dv] = {
            'spline_df': SPLINE_DF_FULL,
            'p_spline_vs_quad': p_spline_vs_quad,
            'n_spline_extrema': n_spline_extrema,
            'spline_extrema_age_str': ','.join(f'{x:.1f}' for x in spline_extrema_age),
            'cubic_extrema_age_str': ','.join(f'{x:.1f}' for x in cubic_extrema_age),
            'extrema_count_matches_cubic': replicated,
        }

        # --- 可視化 ---
        ax.set_facecolor('#FFFFFF')
        ax.scatter(df_dv['age_mid'], df_dv[dv], s=8, alpha=0.15, color='#8899AA')
        grp = df_dv.groupby('age_mid', as_index=False)[dv].mean()
        ax.scatter(grp['age_mid'], grp[dv], s=45, color='#3A6EA5', zorder=4, label='年齢群平均')

        y_quad_grid = np.polyval([m_quad.params['I(age_c ** 2)'], m_quad.params['age_c'],
                                   m_quad.params['Intercept']], xc_grid)
        y_cubic_grid = a3*xc_grid**3 + b3*xc_grid**2 + c3*xc_grid + m_cubic.params['Intercept']
        ax.plot(x_grid, y_quad_grid, color='#5CB85C', lw=1.4, ls='--', label='二次')
        ax.plot(x_grid, y_cubic_grid, color='#D9534F', lw=1.6, ls=':', label='三次')
        ax.plot(x_grid, y_spline_grid, color='#1A1A1A', lw=2.0, label=f'スプライン(df={SPLINE_DF_FULL})')
        for xa in spline_extrema_age:
            ax.axvline(xa, color='#1A1A1A', ls=':', lw=1, alpha=0.5)

        ax.set_title(f'{dv} (spline vs quad, p={p_spline_vs_quad:.3f})', fontsize=10, color='#1A1A1A')
        ax.set_xlabel('年齢 (群中央値)', color='#1A1A1A')
        ax.grid(True, alpha=0.15)
        ax.legend(fontsize=6.5, facecolor='#FFFFFF', labelcolor='#1A1A1A')

    plt.tight_layout()
    spline_fig_path = os.path.join(OUTPUT_DIR, 'ecsoc_full_cohort_spline_shape.png')
    plt.savefig(spline_fig_path, dpi=150, facecolor='#FFFFFF')
    plt.show()
    print(f'\n保存: {spline_fig_path}')

    df_spline_shape_summary = pd.DataFrame(spline_shape_results).T
    df_spline_shape_summary.index.name = 'dv'
    print('\n\n=== サマリー: スプライン極値 vs 三次モデル極値 (CHI, alpha1, alpha2) ===')
    print(df_spline_shape_summary.to_string())
    spline_shape_path = os.path.join(OUTPUT_DIR, 'full_cohort_spline_vs_cubic_extrema.csv')
    df_spline_shape_summary.to_csv(spline_shape_path)
    print(f'\n保存: {spline_shape_path}')

    print('\n\n=== 解釈上の注意 ===')
    print('・CHIは三次項自体が非有意 (Step 11a, p=0.95) なので、極値・変曲点の数値は')
    print('  三次/スプラインどちらであっても実質的な意味を持たない (ノイズへの過剰適合)。')
    print('・alpha2で「極値2つ (谷→山)」が三次・スプライン双方で一致すれば、これは多項式特有の')
    print('  wiggleではなく、データ自体が持つ非単調な構造である可能性が高くなる。')
else:
    print('df_clean が見つからないため、スプライン形状確認はスキップします (Step 11a, 11bを先に実行してください)')


## Step 11d: ブートストラップによる alpha2 極値の頑健性検証

Step 11b (三次モデル) は alpha2 に2つの極値 (谷 ~24.6歳, 山 ~67.1歳) を検出したが、
Step 11c (制限三次スプライン, df=4) は山 (~64.5歳) のみを再現し、谷は再現しなかった。

ここでは対象者リサンプリング (subject-level bootstrap, 復元抽出) により、

1. **谷 (20–35歳)** が三次・スプラインそれぞれ何%のブートストラップ標本で再現されるか
2. **山 (60–75歳)** が何%再現されるか、およびそのピーク年齢の分布 (95%信頼区間)

を定量化する。想定される解釈:

- 山の再現率が高く (目安80%以上)、95%CIが60代に収まる → 山は頑健な構造
- 谷の再現率が低い (目安10%未満) → 谷は三次多項式固有のアーティファクトである可能性が高い

**注意**: スプラインが谷を検出しないことには2通りの解釈がありうる — (a) 谷自体がアーティファクト、
(b) 谷は実在するが df=4 の制限三次スプラインでは局所的すぎて解像できない。今回の結果だけでは
この2つを完全には切り分けられないため、本ブートストラップでも両モデルを併記する。


In [ ]:
# ============================================================
# Step 11d: alpha2 極値のブートストラップ頑健性検証 (三次 & 制限三次スプライン)
# ============================================================
if 'df_clean' in globals():
    from patsy import dmatrix, build_design_matrices

    N_BOOT = 1000          # 論文用の最終版では2000-5000に増やすことを推奨 (時間がかかる)
    RNG_SEED = 42
    SPLINE_DF_BOOT = 4      # Step 11cと同一
    TROUGH_WINDOW = (20, 35)
    PEAK_WINDOW   = (60, 75)

    rng = np.random.default_rng(RNG_SEED)
    dv = 'alpha2'
    df_dv = df_clean.dropna(subset=[dv, 'age_mid']).reset_index(drop=True).copy()
    n_subj = len(df_dv)
    age_min, age_max = df_dv['age_mid'].min(), df_dv['age_mid'].max()

    print(f'alpha2 ブートストラップ開始: N_BOOT={N_BOOT}, n_subj={n_subj}, 年齢範囲=[{age_min:.1f}, {age_max:.1f}]')

    boot_records = []

    for b in range(N_BOOT):
        idx = rng.integers(0, n_subj, n_subj)
        boot = df_dv.iloc[idx].reset_index(drop=True)
        age_mean_b = boot['age_mid'].mean()
        boot['age_c'] = boot['age_mid'] - age_mean_b

        rec = {'boot_id': b}

        # --- 三次モデル ---
        try:
            m_cubic_b = smf.ols(f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)', data=boot).fit()
            a  = m_cubic_b.params['I(age_c ** 3)']
            b2 = m_cubic_b.params['I(age_c ** 2)']
            c  = m_cubic_b.params['age_c']

            extrema_c = np.roots([3*a, 2*b2, c]) if a != 0 else np.array([])
            extrema_c = np.real(extrema_c[np.isreal(extrema_c)])
            extrema_age_b = sorted([age_mean_b + xc for xc in extrema_c if age_min <= age_mean_b + xc <= age_max])

            # 2階微分 p''(x) = 6a*xc + 2b で極小/極大を判定
            minima = [x for x in extrema_age_b if (6*a*(x - age_mean_b) + 2*b2) > 0]
            maxima = [x for x in extrema_age_b if (6*a*(x - age_mean_b) + 2*b2) < 0]

            trough_hits = [x for x in minima if TROUGH_WINDOW[0] <= x <= TROUGH_WINDOW[1]]
            peak_hits   = [x for x in maxima if PEAK_WINDOW[0]   <= x <= PEAK_WINDOW[1]]

            rec['cubic_n_extrema']      = len(extrema_age_b)
            rec['cubic_trough_present'] = len(trough_hits) > 0
            rec['cubic_trough_age']     = trough_hits[0] if trough_hits else np.nan
            rec['cubic_peak_present']   = len(peak_hits) > 0
            rec['cubic_peak_age']       = peak_hits[0] if peak_hits else np.nan
        except Exception:
            rec.update({'cubic_n_extrema': np.nan, 'cubic_trough_present': False, 'cubic_trough_age': np.nan,
                        'cubic_peak_present': False, 'cubic_peak_age': np.nan})

        # --- 制限三次スプライン ---
        try:
            spline_basis_b = dmatrix(f'cr(age_c, df={SPLINE_DF_BOOT}) - 1', data=boot, return_type='dataframe')
            spline_cols_b = [f'age_spline_{i+1}' for i in range(spline_basis_b.shape[1])]
            spline_basis_b.columns = spline_cols_b
            boot2 = pd.concat([boot.reset_index(drop=True), spline_basis_b.reset_index(drop=True)], axis=1)
            m_spline_b = smf.ols(f'{dv} ~ ' + ' + '.join(spline_cols_b), data=boot2).fit()

            x_grid = np.linspace(age_min, age_max, 400)
            xc_grid = x_grid - age_mean_b
            grid_basis = np.asarray(build_design_matrices([spline_basis_b.design_info], {'age_c': xc_grid})[0])
            y_grid = m_spline_b.params['Intercept'] + grid_basis @ m_spline_b.params[spline_cols_b].values \
                if 'Intercept' in m_spline_b.params.index else grid_basis @ m_spline_b.params[spline_cols_b].values

            dy = np.diff(y_grid)
            sign_change = np.where(np.diff(np.sign(dy)) != 0)[0]

            spline_minima, spline_maxima = [], []
            for i in sign_change:
                if dy[i] < 0 and dy[i+1] > 0:
                    spline_minima.append(float(x_grid[i+1]))
                elif dy[i] > 0 and dy[i+1] < 0:
                    spline_maxima.append(float(x_grid[i+1]))

            s_trough = [x for x in spline_minima if TROUGH_WINDOW[0] <= x <= TROUGH_WINDOW[1]]
            s_peak   = [x for x in spline_maxima if PEAK_WINDOW[0]   <= x <= PEAK_WINDOW[1]]

            rec['spline_n_extrema']      = len(sign_change)
            rec['spline_trough_present'] = len(s_trough) > 0
            rec['spline_trough_age']     = s_trough[0] if s_trough else np.nan
            rec['spline_peak_present']   = len(s_peak) > 0
            rec['spline_peak_age']       = s_peak[0] if s_peak else np.nan
        except Exception:
            rec.update({'spline_n_extrema': np.nan, 'spline_trough_present': False, 'spline_trough_age': np.nan,
                        'spline_peak_present': False, 'spline_peak_age': np.nan})

        boot_records.append(rec)

        if (b + 1) % 100 == 0:
            print(f'  進捗: {b+1}/{N_BOOT}')

    df_boot = pd.DataFrame(boot_records)
    boot_path = os.path.join(OUTPUT_DIR, 'alpha2_bootstrap_extrema.csv')
    df_boot.to_csv(boot_path, index=False)

    print(f'\n\n=== alpha2 ブートストラップ結果まとめ (N_BOOT={N_BOOT}) ===')
    for model, label in [('cubic', '三次モデル'), ('spline', f'制限三次スプライン(df={SPLINE_DF_BOOT})')]:
        trough_rate = df_boot[f'{model}_trough_present'].mean() * 100
        peak_rate   = df_boot[f'{model}_peak_present'].mean() * 100
        print(f'\n--- {label} ---')
        print(f'  谷 (20-35歳) の再現率: {trough_rate:.1f}%')
        print(f'  山 (60-75歳) の再現率: {peak_rate:.1f}%')

        peak_ages = df_boot.loc[df_boot[f'{model}_peak_present'], f'{model}_peak_age'].dropna()
        if len(peak_ages) > 0:
            lo, hi = np.percentile(peak_ages, [2.5, 97.5])
            print(f'  山のピーク年齢: median={peak_ages.median():.1f}歳, 95%CI=[{lo:.1f}, {hi:.1f}]歳 (n={len(peak_ages)})')

        trough_ages = df_boot.loc[df_boot[f'{model}_trough_present'], f'{model}_trough_age'].dropna()
        if len(trough_ages) > 0:
            lo, hi = np.percentile(trough_ages, [2.5, 97.5])
            print(f'  谷の年齢: median={trough_ages.median():.1f}歳, 95%CI=[{lo:.1f}, {hi:.1f}]歳 (n={len(trough_ages)})')

    print(f'\n保存: {boot_path}')
    print('\n判定の目安: 山の再現率が高く(>80%)谷が低い(<10%)なら、山=頑健な構造・谷=多項式アーティファクト、')
    print('という本notebookでの解釈 (Step 11b/11c) がブートストラップでも支持されたことになる。')
else:
    print('df_clean が見つからないため、ブートストラップはスキップします (Step 9.5, 11a-11cを先に実行してください)')


## Step 11e: 谷の再現率の解釈 — 制限三次スプラインの境界制約仮説の検証

Step 11d の結果は当初の想定と異なった:

| | 谷(20-35歳)再現率 | 山(60-75歳)再現率 |
|---|---|---|
| 三次モデル | **89.8%** | 94.9% |
| 制限三次スプライン(df=4) | **39.7%** | 96.9% |

谷の再現率は三次モデルでは高く(想定していた「5-10%なら疑わしい」の目安を大きく上回る)、
スプラインでは中間的な値 (39.7%) であり、「三次多項式固有のアーティファクト」と単純には言えない。

考えられる説明が2つある:

1. **谷は実在するデータ構造** — 三次モデルが89.8%という高い一貫性で検出しているのは、
   個々のブートストラップ標本のノイズではなく、コホート全体に存在する弱い非単調性を
   捉えている可能性がある。
2. **スプラインの境界制約による過小検出** — `patsy`の`cr()`(自然/制限三次スプライン)は
   境界knotの外側で線形になるよう制約されており、谷の推定位置 (中央値24.9歳、範囲
   [18.5, 88.5]歳の下端に近い) はまさにこの境界制約の影響を強く受ける領域にある。
   つまり谷が実在しても、`cr()`の構造上、検出されにくい可能性がある。

この2つを切り分けるため、境界制約を持たない**制約なし三次Bスプライン (`bs()`)**を同一df・
同一ブートストラップ標本 (乱数シード固定) で比較する。加えて、`cr()`の自由度を上げた場合
(df=6) にも谷検出率が変わるかを確認する (df自体は境界制約とは独立な要因のため、参考情報)。

**判定の目安**: `bs_df4`の谷再現率が`cr_df4`より明確に高ければ、境界制約が谷検出を抑制していた
可能性が高く、谷は「未確定だが有望な特徴」として扱うべきことになる。`bs_df4`でも低いままなら、
谷は弱いシグナルに留まる可能性が高まる。


In [ ]:
# ============================================================
# Step 11e: cr() (自然スプライン) vs bs() (制約なし三次Bスプライン) の谷検出比較
# ============================================================
if 'df_clean' in globals():
    from patsy import dmatrix, build_design_matrices

    N_BOOT_E = 500          # 3モデル分fitするため11dより軽めに設定。時間に余裕があれば増やす
    RNG_SEED_E = 42          # Step 11dと同一シード (対応比較のため; このセル単独で実行してもOK)
    TROUGH_WINDOW = (20, 35)
    PEAK_WINDOW   = (60, 75)

    # 比較するスプライン仕様
    SPLINE_SPECS = {
        'cr_df4': 'cr(age_c, df=4) - 1',            # Step 11c/11dと同一 (自然スプライン: 境界外で線形)
        'cr_df6': 'cr(age_c, df=6) - 1',            # 自然スプライン、より柔軟 (df感度の参考情報)
        'bs_df4': 'bs(age_c, df=4, degree=3) - 1',  # 制約なし三次Bスプライン (境界制約なし)
    }

    rng = np.random.default_rng(RNG_SEED_E)
    dv = 'alpha2'
    df_dv = df_clean.dropna(subset=[dv, 'age_mid']).reset_index(drop=True).copy()
    n_subj = len(df_dv)
    age_min, age_max = df_dv['age_mid'].min(), df_dv['age_mid'].max()

    def fit_and_find_extrema(boot, formula_rhs, age_mean_b):
        basis = dmatrix(formula_rhs, data=boot, return_type='dataframe')
        cols = [f'sp_{i+1}' for i in range(basis.shape[1])]
        basis.columns = cols
        d2 = pd.concat([boot.reset_index(drop=True), basis.reset_index(drop=True)], axis=1)
        m = smf.ols(f'{dv} ~ ' + ' + '.join(cols), data=d2).fit()

        x_grid = np.linspace(age_min, age_max, 400)
        xc_grid = x_grid - age_mean_b
        grid_basis = np.asarray(build_design_matrices([basis.design_info], {'age_c': xc_grid})[0])
        y_grid = m.params['Intercept'] + grid_basis @ m.params[cols].values \
            if 'Intercept' in m.params.index else grid_basis @ m.params[cols].values

        dy = np.diff(y_grid)
        sign_change = np.where(np.diff(np.sign(dy)) != 0)[0]
        minima, maxima = [], []
        for i in sign_change:
            if dy[i] < 0 and dy[i+1] > 0:
                minima.append(float(x_grid[i+1]))
            elif dy[i] > 0 and dy[i+1] < 0:
                maxima.append(float(x_grid[i+1]))
        return minima, maxima

    boot_records_e = []
    print(f'Step 11e ブートストラップ開始: N_BOOT={N_BOOT_E}, 比較モデル={list(SPLINE_SPECS.keys())}')

    for b in range(N_BOOT_E):
        idx = rng.integers(0, n_subj, n_subj)
        boot = df_dv.iloc[idx].reset_index(drop=True)
        age_mean_b = boot['age_mid'].mean()
        boot['age_c'] = boot['age_mid'] - age_mean_b

        rec = {'boot_id': b}
        for label, rhs in SPLINE_SPECS.items():
            try:
                minima, maxima = fit_and_find_extrema(boot, rhs, age_mean_b)
                trough_hits = [x for x in minima if TROUGH_WINDOW[0] <= x <= TROUGH_WINDOW[1]]
                peak_hits   = [x for x in maxima if PEAK_WINDOW[0]   <= x <= PEAK_WINDOW[1]]
                rec[f'{label}_trough_present'] = len(trough_hits) > 0
                rec[f'{label}_trough_age']     = trough_hits[0] if trough_hits else np.nan
                rec[f'{label}_peak_present']   = len(peak_hits) > 0
                rec[f'{label}_peak_age']       = peak_hits[0] if peak_hits else np.nan
            except Exception:
                rec[f'{label}_trough_present'] = False
                rec[f'{label}_trough_age'] = np.nan
                rec[f'{label}_peak_present'] = False
                rec[f'{label}_peak_age'] = np.nan

        boot_records_e.append(rec)
        if (b + 1) % 100 == 0:
            print(f'  進捗: {b+1}/{N_BOOT_E}')

    df_boot_e = pd.DataFrame(boot_records_e)
    boot_e_path = os.path.join(OUTPUT_DIR, 'alpha2_bootstrap_spline_comparison.csv')
    df_boot_e.to_csv(boot_e_path, index=False)

    print(f'\n\n=== Step 11e サマリー: スプライン仕様間の谷・山再現率 (N_BOOT={N_BOOT_E}) ===')
    for label in SPLINE_SPECS:
        trough_rate = df_boot_e[f'{label}_trough_present'].mean() * 100
        peak_rate   = df_boot_e[f'{label}_peak_present'].mean() * 100
        print(f'\n--- {label} ---')
        print(f'  谷 (20-35歳) 再現率: {trough_rate:.1f}%')
        print(f'  山 (60-75歳) 再現率: {peak_rate:.1f}%')
        t_ages = df_boot_e.loc[df_boot_e[f'{label}_trough_present'], f'{label}_trough_age'].dropna()
        if len(t_ages) > 0:
            lo, hi = np.percentile(t_ages, [2.5, 97.5])
            print(f'  谷年齢: median={t_ages.median():.1f}歳, 95%CI=[{lo:.1f},{hi:.1f}]歳 (n={len(t_ages)})')

    print(f'\n保存: {boot_e_path}')
    print('\n判定の目安: bs_df4 (境界制約なし) の谷再現率が cr_df4 より明確に高ければ、')
    print('  自然スプラインの境界線形制約が谷の検出を抑制していた可能性が高い。')
    print('  逆に bs_df4 でも再現率が低いままなら、谷はデータ自体に弱いシグナルしか持たない可能性が高まる。')
else:
    print('df_clean が見つからないため、Step 11eはスキップします (Step 9.5, 11a-11dを先に実行してください)')


## Step 11f: 結論の統合 — alpha2 の谷は df=4 制限三次スプラインの検出力不足で見逃されていた

Step 11e の結果:

| モデル | 谷再現率 | 谷年齢 (median, 95%CI) | 山再現率 |
|---|---|---|---|
| 三次モデル (Step 11d) | 89.8% | 24.9歳 [20.5, 28.2] | 94.9% |
| cr(df=4) 自然スプライン (Step 11c/11d/11e) | 39.7–41.8% | 26.2歳 [21.1, 31.0] | 96.9–97.6% |
| cr(df=6) 自然スプライン | **90.0%** | 27.3歳 [21.8, 29.4] | 93.8% |
| bs(df=4) 制約なしBスプライン | **81.4%** | 27.3歳 [21.8, 30.3] | 95.0% |

当初 Step 11c で用いた `cr(df=4)` **単独**が谷の検出率が低い外れ値であり、自由度を上げた
`cr(df=6)` でも、境界制約を外した `bs(df=4)` (同一df) でも谷は高頻度 (81–90%) で再現された。
これは三次モデルの再現率 (89.8%) とも整合する。谷の年齢も4つの独立したモデル仕様すべてで
24.9–27.3歳のごく狭い範囲に収まっている。

**解釈の転換**: 「谷は三次多項式のアーティファクト」ではなく、「谷はデータに実在する構造だが、
Step 11c で採用した `cr(df=4)` という特定のスプライン仕様が、それを検出するには柔軟性・
knot配置の面で力不足だった」という説明の方がデータ全体を最もよく説明する。

以下のセルで、これまでのブートストラップ結果 (Step 11d, 11e の保存済みCSV) を統合し、
論文用の最終サマリー表と結論文を生成する。


In [ ]:
# ============================================================
# Step 11f: alpha2 ブートストラップ結果の統合 (Step 11d + 11e の保存済みCSVを読み込み)
# ============================================================
boot_d_path = os.path.join(OUTPUT_DIR, 'alpha2_bootstrap_extrema.csv')
boot_e_path = os.path.join(OUTPUT_DIR, 'alpha2_bootstrap_spline_comparison.csv')

if os.path.exists(boot_d_path) and os.path.exists(boot_e_path):
    df_d = pd.read_csv(boot_d_path)
    df_e = pd.read_csv(boot_e_path)

    def summarize(df, trough_col, peak_col, trough_age_col, peak_age_col, label, n_boot):
        trough_rate = df[trough_col].mean() * 100
        peak_rate = df[peak_col].mean() * 100
        t_ages = df.loc[df[trough_col], trough_age_col].dropna()
        p_ages = df.loc[df[peak_col], peak_age_col].dropna()
        row = {'model': label, 'n_boot': n_boot,
               'trough_reproduction_pct': trough_rate, 'peak_reproduction_pct': peak_rate}
        if len(t_ages) > 0:
            lo, hi = np.percentile(t_ages, [2.5, 97.5])
            row.update({'trough_age_median': t_ages.median(), 'trough_age_ci_lo': lo, 'trough_age_ci_hi': hi})
        if len(p_ages) > 0:
            lo, hi = np.percentile(p_ages, [2.5, 97.5])
            row.update({'peak_age_median': p_ages.median(), 'peak_age_ci_lo': lo, 'peak_age_ci_hi': hi})
        return row

    rows = [
        summarize(df_d, 'cubic_trough_present', 'cubic_peak_present',
                  'cubic_trough_age', 'cubic_peak_age', 'cubic (Step 11d)', len(df_d)),
        summarize(df_d, 'spline_trough_present', 'spline_peak_present',
                  'spline_trough_age', 'spline_peak_age', 'cr_df4 (Step 11d)', len(df_d)),
        summarize(df_e, 'cr_df4_trough_present', 'cr_df4_peak_present',
                  'cr_df4_trough_age', 'cr_df4_peak_age', 'cr_df4 (Step 11e)', len(df_e)),
        summarize(df_e, 'cr_df6_trough_present', 'cr_df6_peak_present',
                  'cr_df6_trough_age', 'cr_df6_peak_age', 'cr_df6 (Step 11e)', len(df_e)),
        summarize(df_e, 'bs_df4_trough_present', 'bs_df4_peak_present',
                  'bs_df4_trough_age', 'bs_df4_peak_age', 'bs_df4 (Step 11e)', len(df_e)),
    ]
    df_final = pd.DataFrame(rows).set_index('model')

    print('=== Step 11f: alpha2 谷・山 ブートストラップ再現率 統合サマリー ===\n')
    print(df_final.to_string(float_format=lambda x: f'{x:.1f}' if isinstance(x, float) else str(x)))

    final_path = os.path.join(OUTPUT_DIR, 'alpha2_bootstrap_final_summary.csv')
    df_final.to_csv(final_path)
    print(f'\n保存: {final_path}')

    # 低検出力だった cr_df4 を除く4仕様の谷再現率・年齢の一貫性
    robust_models = ['cubic (Step 11d)', 'cr_df6 (Step 11e)', 'bs_df4 (Step 11e)']
    trough_rates_robust = df_final.loc[robust_models, 'trough_reproduction_pct']
    trough_ages_robust  = df_final.loc[robust_models, 'trough_age_median']

    print('\n\n=== 論文用: 更新された結論文 (ドラフト) ===\n')
    print(textwrap.fill(
        f"The short-timescale scaling exponent alpha2 exhibited a nonlinear age trajectory with a "
        f"late-life maximum around the mid-60s (bootstrap reproduction rate 94.9-97.6% across cubic "
        f"and spline specifications; median peak age 64.5-66.9 years). An additional early-adult "
        f"local minimum, initially detected only by the cubic model, was reproduced at a comparably "
        f"high rate ({trough_rates_robust.min():.1f}-{trough_rates_robust.max():.1f}% across "
        f"{len(robust_models)} independent model specifications: cubic polynomial, a more flexible "
        f"restricted cubic spline (df=6), and an unconstrained cubic B-spline of matched flexibility "
        f"to the originally tested spline), converging on an age of "
        f"{trough_ages_robust.min():.1f}-{trough_ages_robust.max():.1f} years. The originally tested "
        f"restricted cubic spline (df=4) failed to reproduce this minimum in the majority of bootstrap "
        f"replicates (39.7-41.8%), which follow-up analyses suggest reflects insufficient flexibility "
        f"of that specific specification rather than absence of the underlying feature. Both the "
        f"late-life maximum and the early-adult minimum are therefore treated as data-driven features "
        f"of the alpha2 age trajectory, subject to confirmation in an independent cohort.",
        width=100))
else:
    missing = [p for p in [boot_d_path, boot_e_path] if not os.path.exists(p)]
    print(f'必要なCSVが見つかりません (Step 11d, 11eを先に実行してください): {missing}')


In [ ]:
# ============================================================
# Step 12: 性別 x 年齢の交互作用検定 (個体レベル回帰)
# ============================================================
# 従来のプールされたMale vs Female比較は年齢を無視しているため、
# 「biphasicカーブの形(ピーク年齢・曲がり方)が男女で異なるか」には答えられない。
# 最高齢群(n=6-23)を性別でさらに分割すると検出力を失うため、群平均表を
# 性別で分割するのではなく、個体レベル(N=1,032)の回帰モデルに性別の主効果と
# 年齢との交互作用項を投入し、既存のサンプルサイズを最大限活用する。

if 'sex' in df_clean.columns and df_clean['sex'].notna().any():
    sex_map = {0: 'Male', 1: 'Female'}
    df_clean['sex_label'] = df_clean['sex'].map(sex_map)

    # --- (a) 参考: プールされた記述統計 (年齢未調整の粗い比較) ---
    print('=== [参考] 性別 x 年齢群 別 CHI平均 (年齢未調整) ===')
    pivot_sex = df_clean.pivot_table(
        index='age_group', columns='sex_label', values='CHI', aggfunc='mean'
    )
    print(pivot_sex.to_string(float_format=lambda x: f'{x:+.4f}'))

    male_chi = df_clean[df_clean['sex_label']=='Male']['CHI'].dropna()
    female_chi = df_clean[df_clean['sex_label']=='Female']['CHI'].dropna()
    if len(male_chi) > 5 and len(female_chi) > 5:
        u_stat, u_p = mannwhitneyu(male_chi, female_chi)
        print(f'\n全体プールでのMale vs Female CHI比較 (Mann-Whitney U, 年齢未調整): '
              f'U={u_stat:.1f}, p={u_p:.3e}')
        print(f'  Male   CHI: {male_chi.mean():+.4f} +/- {male_chi.std():.4f} (n={len(male_chi)})')
        print(f'  Female CHI: {female_chi.mean():+.4f} +/- {female_chi.std():.4f} (n={len(female_chi)})')

    # --- (b) 本題: 個体レベル回帰で年齢x性別の交互作用を検定 ---
    df_sex_valid = df_clean.dropna(subset=['CHI', 'age_mid', 'sex_label']).copy()
    df_sex_valid['sex_label'] = df_sex_valid['sex_label'].astype('category')

    print(f'\n\n=== 個体レベル回帰 (N={len(df_sex_valid)}): '
          f'CHI ~ age + age^2 + sex + age:sex + age^2:sex ===')
    model_full = smf.ols(
        'CHI ~ age_mid + I(age_mid**2) * C(sex_label, Treatment(reference="Male"))',
        data=df_sex_valid
    ).fit()
    print(model_full.summary())

    # --- (c) 交互作用項をまとめて検定するネストF検定 ---
    model_reduced = smf.ols('CHI ~ age_mid + I(age_mid**2)', data=df_sex_valid).fit()
    print('\n=== 交互作用項(性別x年齢, 性別x年齢^2)のネストF検定 ===')
    print('H0: 性別によってCHIの加齢曲線の形は変わらない')
    anova_result = anova_lm(model_reduced, model_full)
    print(anova_result)

    # --- (d) 性別ごとに別々にフィットした二次係数 (参考・可視化用) ---
    print('\n=== 性別ごとの二次回帰係数 (参考: 別々にフィットした場合) ===')
    for sx in ['Male', 'Female']:
        sub = df_sex_valid[df_sex_valid['sex_label'] == sx]
        if len(sub) > 20:
            m = smf.ols('CHI ~ age_mid + I(age_mid**2)', data=sub).fit()
            beta2 = m.params.get('I(age_mid ** 2)', float('nan'))
            pval2 = m.pvalues.get('I(age_mid ** 2)', float('nan'))
            print(f'  {sx} (n={len(sub)}): quadratic coef = {beta2:.6f}, p(quad) = {pval2:.4f}')
        else:
            print(f'  {sx}: n<=20 のため二次回帰は不安定な可能性があります (参考値として扱ってください)')
else:
    print('sex列が見つからないか全て欠損のため、性別x年齢の交互作用解析はスキップします')


## Step 12 拡張パッケージ (性差の多角的検討)

既存の Step 12 (CHIの性別×年齢 交互作用) に加えて、以下4種類の解析を追加する。

1. **Step 12a**: α₁・α₂ の性差 (Step 12 と同一構造で従属変数を CHI → α₁/α₂ に拡張)
2. **Step 12b**: CHI を従属変数にした共変量調整モデル (BMI・記録デバイス・記録時間を追加投入)
3. **Step 12c**: 年齢×性別×CHI の専用可視化 (年齢群別平均±SE、個体散布+性別ごとの二次フィット/ブートストラップ95%CI)
4. **Step 12d**: 女性サブ解析 (閉経状態プロキシによる層別)

**重要な注記 (Step 12d について)**: 本データセット (PhysioNet Autonomic Aging DB, `subject-info.csv`) が
収録するのは年齢群・性別・BMI・記録デバイスのみであり、実際の閉経状態や月経歴は含まれていない。
そのため Step 12d では、自然閉経の典型的な年齢分布 (平均閉経年齢は概ね51歳前後、移行期は
40代後半〜50代前半とされる) に基づいた**年齢群ベースの閉経状態プロキシ**を用いる。これは
実測の閉経状態ではなく粗い近似であるため、結果はあくまで探索的なものとして扱い、
論文で言及する際はプロキシである旨を明記し、限定的な解釈にとどめること。


In [ ]:
# ============================================================
# Step 12a: 性別 x 年齢群別 α₁・α₂ 比較 (Step 12 のCHI解析と同一構造)
# ============================================================
if 'sex_label' in df_clean.columns and df_clean['sex_label'].notna().any():

    # --- (a) 参考: プールされた記述統計 (年齢未調整の粗い比較) ---
    print('=== [参考] 性別 x 年齢群 別 α₁・α₂平均 (年齢未調整) ===')
    pivot_a1 = df_clean.pivot_table(index='age_group', columns='sex_label', values='alpha1', aggfunc='mean')
    pivot_a2 = df_clean.pivot_table(index='age_group', columns='sex_label', values='alpha2', aggfunc='mean')
    print('\n--- α₁ (短距離, 4-16拍) ---')
    print(pivot_a1.to_string(float_format=lambda x: f'{x:.4f}'))
    print('\n--- α₂ (長距離, 16-64拍) ---')
    print(pivot_a2.to_string(float_format=lambda x: f'{x:.4f}'))

    male_a1   = df_clean[df_clean['sex_label']=='Male']['alpha1'].dropna()
    female_a1 = df_clean[df_clean['sex_label']=='Female']['alpha1'].dropna()
    male_a2   = df_clean[df_clean['sex_label']=='Male']['alpha2'].dropna()
    female_a2 = df_clean[df_clean['sex_label']=='Female']['alpha2'].dropna()

    print('\n=== 全体プールでの Male vs Female 比較 (Mann-Whitney U, 年齢未調整) ===')
    for name, m, f in [('alpha1', male_a1, female_a1), ('alpha2', male_a2, female_a2)]:
        if len(m) > 5 and len(f) > 5:
            u_stat, u_p = mannwhitneyu(m, f)
            print(f'  {name}: Male {m.mean():.4f}+/-{m.std():.4f} (n={len(m)}) vs '
                  f'Female {f.mean():.4f}+/-{f.std():.4f} (n={len(f)})  U={u_stat:.1f}, p={u_p:.3e}')

    # --- (b) 本題: 個体レベル回帰で年齢x性別の交互作用をα₁・α₂それぞれに検定 ---
    # (Step 12 の CHI モデルと完全に同一の定式化。従属変数のみ差し替え)
    df_sex_valid_ab = df_clean.dropna(subset=['alpha1', 'alpha2', 'age_mid', 'sex_label']).copy()
    df_sex_valid_ab['sex_label'] = df_sex_valid_ab['sex_label'].astype('category')

    alpha_models = {}
    for dv in ['alpha1', 'alpha2']:
        print(f'\n\n=== 個体レベル回帰 (N={len(df_sex_valid_ab)}): '
              f'{dv} ~ age + age^2 + sex + age:sex + age^2:sex ===')
        m_full = smf.ols(
            f'{dv} ~ age_mid + I(age_mid**2) * C(sex_label, Treatment(reference="Male"))',
            data=df_sex_valid_ab
        ).fit()
        print(m_full.summary())

        m_reduced = smf.ols(f'{dv} ~ age_mid + I(age_mid**2)', data=df_sex_valid_ab).fit()
        print(f'\n=== 交互作用項(性別x年齢, 性別x年齢^2)のネストF検定: {dv} ===')
        print(f'H0: 性別によって{dv}の加齢曲線の形は変わらない')
        print(anova_lm(m_reduced, m_full))

        alpha_models[dv] = {'full': m_full, 'reduced': m_reduced}

        for sx in ['Male', 'Female']:
            sub = df_sex_valid_ab[df_sex_valid_ab['sex_label'] == sx]
            if len(sub) > 20:
                m = smf.ols(f'{dv} ~ age_mid + I(age_mid**2)', data=sub).fit()
                b2 = m.params.get('I(age_mid ** 2)', float('nan'))
                p2 = m.pvalues.get('I(age_mid ** 2)', float('nan'))
                print(f'  [{dv}] {sx} (n={len(sub)}): quadratic coef = {b2:.6f}, p(quad) = {p2:.4f}')
            else:
                print(f'  [{dv}] {sx}: n<=20 のため二次回帰は不安定な可能性があります (参考値として扱ってください)')
else:
    print('sex_label列が見つからないため、α₁・α₂の性差解析はスキップします (Step 12を先に実行してください)')


In [ ]:
# ============================================================
# Step 12b: CHIを従属変数にした共変量調整モデル (BMI・記録デバイス・記録時間)
# ============================================================
# 既存Step 12のmodel_full (age*sexのみ) に、BMI・記録デバイス・記録時間 (recording_min,
# Step 9.5のQCで算出済み) を共変量として追加投入する。目的は、Step 12で見られた
# age×sex交互作用が、これらの身体的/技術的共変量による交絡で説明されてしまわないかの確認。
from statsmodels.stats.outliers_influence import variance_inflation_factor

if 'sex_label' in df_clean.columns and df_clean['sex_label'].notna().any():
    cov_needed = ['CHI', 'age_mid', 'sex_label', 'bmi', 'recording_min']
    has_device = 'device' in df_clean.columns and df_clean['device'].notna().any()
    if has_device:
        device_map = {0: 'TFM', 1: 'CNAP_MP150'}
        df_clean['device_label'] = df_clean['device'].map(device_map)
        cov_needed.append('device_label')

    df_cov = df_clean.dropna(subset=cov_needed).copy()
    df_cov['sex_label'] = df_cov['sex_label'].astype('category')

    print(f'共変量調整モデル用サンプル数 (BMI・記録時間{"・デバイス" if has_device else ""}すべて非欠損): '
          f'n={len(df_cov)} (df_cleanからの脱落: {len(df_clean) - len(df_cov)}件, 主にBMI欠損)')

    device_term = ' + C(device_label)' if has_device else ''

    # (a) 主効果のみの調整モデル (交互作用なし; 各共変量の独立寄与を見る)
    f_main = ('CHI ~ age_mid + I(age_mid**2) + '
              'C(sex_label, Treatment(reference="Male")) + bmi + recording_min' + device_term)
    print(f'\n=== (a) 共変量調整モデル (主効果のみ) ===\n  {f_main}')
    model_main = smf.ols(f_main, data=df_cov).fit()
    print(model_main.summary())

    # (b) age*sex交互作用も含めたフルモデル (共変量調整 + 交互作用)
    f_int = ('CHI ~ age_mid * C(sex_label, Treatment(reference="Male")) + '
             'I(age_mid**2) * C(sex_label, Treatment(reference="Male")) + '
             'bmi + recording_min' + device_term)
    print(f'\n=== (b) 共変量調整 + age×sex交互作用モデル ===\n  {f_int}')
    model_int = smf.ols(f_int, data=df_cov).fit()
    print(model_int.summary())

    # (c) 交絡チェック: 同一サブセット (df_cov) で共変量なしのage*sexモデルと比較
    f_unadj = ('CHI ~ age_mid * C(sex_label, Treatment(reference="Male")) + '
               'I(age_mid**2) * C(sex_label, Treatment(reference="Male"))')
    model_unadj_samesample = smf.ols(f_unadj, data=df_cov).fit()

    print(f'\n=== (c) 交絡チェック: 同一サブセット (n={len(df_cov)}) での比較 ===')
    print(f'  未調整モデル (age*sexのみ)                    AIC = {model_unadj_samesample.aic:.2f}, adj-R² = {model_unadj_samesample.rsquared_adj:.4f}')
    print(f'  共変量調整モデル (BMI・デバイス・記録時間追加, 交互作用込) AIC = {model_int.aic:.2f}, adj-R² = {model_int.rsquared_adj:.4f}')
    print('  → 共変量投入後もage×sex交互作用項の符号・有意性が大きく変わらなければ、')
    print('    Step 12で見られた性差はBMI/デバイス/記録時間による交絡ではないと解釈できる。')
    print('    (上記(b)モデルsummary中の age_mid:C(sex_label)... 項、および')
    print('     I(age_mid**2):C(sex_label)... 項を、この未調整モデルおよび既存Step12のmodel_fullと')
    print('     見比べて確認すること)')

    print('\n=== (b) vs (a) ネストF検定 (共変量調整後もage×sex交互作用が必要か) ===')
    print(anova_lm(model_main, model_int))

    # --- 多重共線性チェック (VIF; 連続変数のみ。age項はVIF計算時のみ中心化) ---
    print('\n=== VIF (多重共線性チェック; 連続変数のみ、age項は中心化して計算) ===')
    vif_input = df_cov[['age_mid', 'bmi', 'recording_min']].dropna().copy()
    vif_input['age_c'] = vif_input['age_mid'] - vif_input['age_mid'].mean()
    vif_input['age_c_sq'] = vif_input['age_c'] ** 2
    vif_cols = ['age_c', 'age_c_sq', 'bmi', 'recording_min']
    vif_table = pd.DataFrame({
        'variable': vif_cols,
        'VIF': [variance_inflation_factor(vif_input[vif_cols].values, i) for i in range(len(vif_cols))]
    })
    print(vif_table.to_string(index=False))
    print('  (目安: VIF>10で強い多重共線性の懸念。モデル自体はStep12との一貫性のためage_midを')
    print('   中心化せずに投入しているが、その場合age_midとI(age_mid**2)間のVIFは高くなるのが通例であり、')
    print('   本チェックではその純粋な影響を分離するためage項のみ中心化して計算している)')

    model_int.summary2().tables[1].to_csv(os.path.join(OUTPUT_DIR, 'chi_covariate_adjusted_model.csv'))
    print(f'\n保存: {os.path.join(OUTPUT_DIR, "chi_covariate_adjusted_model.csv")}')
else:
    print('sex_label列が見つからないため、共変量調整モデルはスキップします (Step 12を先に実行してください)')


In [ ]:
# ============================================================
# Step 12c: 年齢×性別×CHI の専用可視化
# ============================================================
if 'sex_label' in df_clean.columns and df_clean['sex_label'].notna().any():
    SEX_COLORS = {'Male': '#3A6EA5', 'Female': '#D9534F'}

    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5.5), facecolor='#FFFFFF')

    # --- Panel A: 年齢群別 CHI平均 ± SE (性別ごとの折れ線) ---
    axA = axes2[0]
    for sx in ['Male', 'Female']:
        sub = df_clean[df_clean['sex_label'] == sx]
        grp = sub.groupby('age_group')['CHI'].agg(['mean', 'std', 'count']).reindex(AGE_ORDER)
        se = grp['std'] / np.sqrt(grp['count'])
        axA.plot(range(len(AGE_ORDER)), grp['mean'], marker='o', color=SEX_COLORS[sx],
                 lw=1.8, label=f'{sx} (総n={len(sub)})')
        axA.fill_between(range(len(AGE_ORDER)), grp['mean'] - se, grp['mean'] + se,
                          color=SEX_COLORS[sx], alpha=0.15)
    axA.axhline(0, color='#F0E68C', ls=':', lw=0.8)
    axA.set_xticks(range(len(AGE_ORDER)))
    axA.set_xticklabels(age_labels_ordered, rotation=45, ha='right', fontsize=7, color='#1A1A1A')
    axA.set_ylabel('CHI (mean ± SE)', color='#1A1A1A')
    axA.set_title('A. CHI by Age Group × Sex', color='#1A1A1A', fontsize=10)
    axA.legend(fontsize=8, facecolor='#FFFFFF', labelcolor='#1A1A1A')
    axA.set_facecolor('#FFFFFF')
    axA.grid(True, alpha=0.15)

    # --- Panel B: 個体散布 + 性別ごとの二次回帰フィット (ブートストラップ95%CI) ---
    axB = axes2[1]
    x_pred = np.linspace(df_clean['age_mid'].min(), df_clean['age_mid'].max(), 100)
    rng = np.random.default_rng(42)
    N_BOOT = 500

    for sx in ['Male', 'Female']:
        sub = df_clean[df_clean['sex_label'] == sx].dropna(subset=['age_mid', 'CHI'])
        axB.scatter(sub['age_mid'], sub['CHI'], s=6, alpha=0.15, color=SEX_COLORS[sx])

        x, y = sub['age_mid'].values, sub['CHI'].values
        boot_preds = np.zeros((N_BOOT, len(x_pred)))
        for b in range(N_BOOT):
            idx = rng.integers(0, len(x), len(x))
            c = np.polyfit(x[idx], y[idx], 2)
            boot_preds[b] = np.polyval(c, x_pred)
        c_obs = np.polyfit(x, y, 2)
        y_obs = np.polyval(c_obs, x_pred)
        lo, hi = np.percentile(boot_preds, [2.5, 97.5], axis=0)

        axB.plot(x_pred, y_obs, color=SEX_COLORS[sx], lw=2, label=f'{sx} (二次フィット)')
        axB.fill_between(x_pred, lo, hi, color=SEX_COLORS[sx], alpha=0.2)

    axB.axhline(0, color='#F0E68C', ls=':', lw=0.8)
    axB.set_xlabel('年齢 (群中央値, 歳)', color='#1A1A1A')
    axB.set_ylabel('CHI (individual)', color='#1A1A1A')
    axB.set_title('B. CHI vs Age by Sex (quadratic fit, bootstrap 95%CI)', color='#1A1A1A', fontsize=10)
    axB.legend(fontsize=8, facecolor='#FFFFFF', labelcolor='#1A1A1A')
    axB.set_facecolor('#FFFFFF')
    axB.grid(True, alpha=0.15)

    fig2.suptitle(f'ECSoC Autonomic Aging — Age × Sex × CHI (N={len(df_clean)})',
                  color='#1A1A1A', fontsize=12)
    fig2_path = os.path.join(OUTPUT_DIR, 'ecsoc_aging_sex_dashboard.png')
    fig2.savefig(fig2_path, dpi=150, bbox_inches='tight', facecolor='#FFFFFF')
    plt.show()
    print(f'図保存: {fig2_path}')
else:
    print('sex_label列が見つからないため、年齢×性別×CHIの可視化はスキップします (Step 12を先に実行してください)')


In [ ]:
# ============================================================
# Step 12d: 女性サブ解析 — 閉経状態プロキシによる層別
# ============================================================
# 注意: 本データセット (subject-info.csv) には実際の閉経状態・月経歴は含まれていない
# (収録されているのは年齢群・性別・BMI・記録デバイスのみ)。以下は自然閉経の典型的な
# 年齢分布 (平均閉経年齢は概ね51歳前後、移行期は40代後半〜50代前半とされる) に基づく
# 「年齢群ベースのプロキシ」分類であり、個々人の実際の閉経状態を表すものではない。
# 探索的な補助解析として扱い、結果を論文等で用いる際はプロキシである旨を明記すること。

if 'sex_label' in df_clean.columns and (df_clean['sex_label'] == 'Female').any():
    df_female = df_clean[df_clean['sex_label'] == 'Female'].copy()

    def menopause_proxy(age_group):
        if age_group <= 6:      # <45歳
            return 'pre (proxy, <45)'
        elif age_group <= 8:    # 45-54歳 (移行期典型レンジ)
            return 'peri (proxy, 45-54)'
        else:                   # 55歳以上
            return 'post (proxy, 55+)'

    df_female['menopause_proxy'] = df_female['age_group'].apply(menopause_proxy)
    proxy_order = ['pre (proxy, <45)', 'peri (proxy, 45-54)', 'post (proxy, 55+)']

    print(f'女性被験者数 (df_clean内): n={len(df_female)}')
    print('\n=== 閉経状態プロキシ別サンプル数・CHI/α₁/α₂記述統計 ===')
    summary_meno = df_female.groupby('menopause_proxy')[['CHI', 'alpha1', 'alpha2']].agg(['mean', 'std', 'count'])
    summary_meno = summary_meno.reindex(proxy_order)
    print(summary_meno.to_string(float_format=lambda x: f'{x:.4f}'))

    groups_meno = [df_female[df_female['menopause_proxy']==g]['CHI'].dropna().values for g in proxy_order]
    groups_meno = [g for g in groups_meno if len(g) > 0]
    if len(groups_meno) >= 2 and all(len(g) > 5 for g in groups_meno):
        kw_stat_m, kw_p_m = kruskal(*groups_meno)
        print(f'\n=== Kruskal-Wallis (閉経プロキシ3群間のCHI差, 女性のみ) ===')
        print(f'  H = {kw_stat_m:.3f}, p = {kw_p_m:.3e}')
    else:
        print('\n⚠ いずれかの群のnが小さすぎるため、Kruskal-Wallis検定は省略します')

    # pre-proxy vs post-proxy の直接比較 (移行期を除いた対比)
    pre_chi  = df_female[df_female['menopause_proxy']=='pre (proxy, <45)']['CHI'].dropna()
    post_chi = df_female[df_female['menopause_proxy']=='post (proxy, 55+)']['CHI'].dropna()
    if len(pre_chi) > 5 and len(post_chi) > 5:
        u_m, p_m = mannwhitneyu(pre_chi, post_chi)
        print(f'\n=== Pre-proxy vs Post-proxy 直接比較 (Mann-Whitney U) ===')
        print(f'  Pre  (n={len(pre_chi)}): CHI = {pre_chi.mean():+.4f} ± {pre_chi.std():.4f}')
        print(f'  Post (n={len(post_chi)}): CHI = {post_chi.mean():+.4f} ± {post_chi.std():.4f}')
        print(f'  U={u_m:.1f}, p={p_m:.3e}')

    # 女性のみの二次回帰 (年齢連続量; 移行期周辺の非線形性を目視確認する参考値)
    df_f_valid = df_female.dropna(subset=['age_mid', 'CHI'])
    if len(df_f_valid) > 30:
        xf, yf = df_f_valid['age_mid'].values, df_f_valid['CHI'].values
        xf_c = xf - xf.mean()
        coeffs_f = np.polyfit(xf_c, yf, 2)
        print(f'\n=== 女性のみ 二次回帰 (参考; 全年齢, n={len(df_f_valid)}) ===')
        print(f'  二次項係数 = {coeffs_f[0]:+.6f}')

    # --- 可視化: 女性のみ、閉経プロキシで色分けしたCHI vs 年齢散布図 + 群境界線 ---
    fig3, ax3 = plt.subplots(figsize=(7.5, 5.5), facecolor='#FFFFFF')
    proxy_colors = {'pre (proxy, <45)': '#3A6EA5', 'peri (proxy, 45-54)': '#F0AD4E', 'post (proxy, 55+)': '#D9534F'}
    for g in proxy_order:
        sub = df_female[df_female['menopause_proxy'] == g]
        ax3.scatter(sub['age_mid'], sub['CHI'], s=14, alpha=0.5,
                    color=proxy_colors[g], label=f'{g} (n={len(sub)})')
    ax3.axvline(45, color='#888888', ls='--', lw=0.8)
    ax3.axvline(55, color='#888888', ls='--', lw=0.8)
    ax3.axhline(0, color='#F0E68C', ls=':', lw=0.8)
    ax3.set_xlabel('年齢 (群中央値, 歳)', color='#1A1A1A')
    ax3.set_ylabel('CHI', color='#1A1A1A')
    ax3.set_title(f'女性サブ解析: CHI vs 年齢 (閉経状態プロキシ別, n={len(df_female)})\n'
                  '※ プロキシ分類であり実測の閉経状態ではない', color='#1A1A1A', fontsize=10)
    ax3.legend(fontsize=8, facecolor='#FFFFFF', labelcolor='#1A1A1A')
    ax3.set_facecolor('#FFFFFF')
    ax3.grid(True, alpha=0.15)
    fig3_path = os.path.join(OUTPUT_DIR, 'ecsoc_aging_female_menopause_proxy.png')
    fig3.savefig(fig3_path, dpi=150, bbox_inches='tight', facecolor='#FFFFFF')
    plt.show()
    print(f'図保存: {fig3_path}')

    df_female.to_csv(os.path.join(OUTPUT_DIR, 'female_menopause_proxy_subanalysis.csv'), index=False)
    print(f'保存: {os.path.join(OUTPUT_DIR, "female_menopause_proxy_subanalysis.csv")}')
else:
    print('女性被験者データが見つからないため、閉経状態サブ解析はスキップします (Step 12を先に実行してください)')


## Step 12d 拡張: 加齢軌跡そのものの閉経前後差 (age × menopause_proxy 交互作用)

Step 12d の Kruskal-Wallis 検定は、閉経状態プロキシ3群 (pre/peri/post) 間で CHI の**平均水準**が
異なるかを見ているに過ぎない。これは「閉経後にCHIが平均的にどれだけ違うか」という問いには答えるが、
**「加齢に伴うCHI/α₁/α₂の変化の仕方(軌跡・傾き)そのものが、閉経前後で変わるか」**という、
ECSoC的にはより本質的な問いには答えていない。

以下ではこれを明示的に検定・可視化する。

- **Step 12e**: CHI・α₁・α₂それぞれについて `DV ~ age_mid * menopause_proxy` の交互作用モデルを構築し、
  「傾き(軌跡)は群間で共通」という帰無仮説を、主効果のみモデルとのネストF検定で検証する。
  交互作用が有意なら、閉経前後で加齢曲線の**形そのもの**が変化していると解釈できる。
- **Step 12f**: Step 12eのモデル予測を用いて、CHI・α₁・α₂の年齢軌跡をpre/peri/post群別に
  同一年齢軸上に可視化する (α₁とα₂を並べて示すことで、CHIの軌跡変化がどちらの成分に由来するかを確認できる)。

**注**: 本解析は引き続き Step 12d と同じ「年齢群ベースの閉経状態プロキシ」(実測の閉経状態ではない) に
依存している。特に peri 群 (45-54歳, 2年齢群のみ) はサンプルサイズが相対的に小さくなりやすいため、
交互作用の検出力には限界がある点に留意すること。


In [ ]:
# ============================================================
# Step 12e: 女性における age × 閉経状態プロキシ 交互作用検定
# ============================================================
# Step 12d は「閉経プロキシ群間でCHIの"平均"が違うか」(Kruskal-Wallis) を見ただけだった。
# ここでは age_mid * menopause_proxy の交互作用項を直接投入し、
# 「加齢に伴うCHI/α₁/α₂の"軌跡"(傾き)そのものが、閉経前後で変わるか」を検定する。
# 交互作用が有意 → 単なる水準シフトではなく、加齢曲線の形状自体が閉経前後で異なる。
# 交互作用が非有意 → 傾きは共通で、群間差があるとすれば切片(水準)差のみと解釈するのが妥当。

if 'df_female' in globals() and 'menopause_proxy' in df_female.columns:
    df_female['menopause_proxy'] = df_female['menopause_proxy'].astype('category')
    ref_level = 'pre (proxy, <45)'

    trajectory_models = {}
    for dv in ['CHI', 'alpha1', 'alpha2']:
        df_dv = df_female.dropna(subset=[dv, 'age_mid', 'menopause_proxy']).copy()
        n_per_group = df_dv['menopause_proxy'].value_counts()
        print(f'\n\n=== {dv}: age × menopause_proxy 交互作用モデル (女性のみ, N={len(df_dv)}) ===')
        print(f'  群別n: {dict(n_per_group)}')
        if (n_per_group < 10).any():
            print('  ⚠ いずれかの群のn<10のため、交互作用モデルの推定は不安定な可能性があります (参考値として扱う)')

        # フルモデル: 群ごとに傾き(軌跡)が異なることを許容
        f_full = f'{dv} ~ age_mid * C(menopause_proxy, Treatment(reference="{ref_level}"))'
        model_full_dv = smf.ols(f_full, data=df_dv).fit()

        # 縮小モデル: 群間で水準(切片)は違ってよいが、傾きは共通と仮定
        f_reduced = f'{dv} ~ age_mid + C(menopause_proxy, Treatment(reference="{ref_level}"))'
        model_reduced_dv = smf.ols(f_reduced, data=df_dv).fit()

        print('\n--- フルモデル (age × proxy 交互作用あり) ---')
        print(model_full_dv.summary())

        print('\n--- ネストF検定: H0 = 傾き(軌跡)は群間で共通 (交互作用なし) ---')
        nested = anova_lm(model_reduced_dv, model_full_dv)
        print(nested)
        interaction_p = nested['Pr(>F)'].iloc[1]

        # 個別の交互作用項 (peri, post それぞれの傾きが基準群 pre からどれだけ変化するか)
        base_slope = model_full_dv.params.get('age_mid', float('nan'))
        print(f'\n群別の推定傾き (age_mid 1歳あたりの{dv}変化量):')
        print(f'  pre  (基準群): slope = {base_slope:+.6f}')
        for g in ['peri (proxy, 45-54)', 'post (proxy, 55+)']:
            matches = [p for p in model_full_dv.params.index if p.startswith('age_mid:') and g in p]
            if matches:
                key = matches[0]
                delta = model_full_dv.params[key]
                pval = model_full_dv.pvalues[key]
                print(f'  {g}: slope = {base_slope + delta:+.6f}  '
                      f'(pre群との傾きの差 = {delta:+.6f}, p={pval:.4f})')
            else:
                print(f'  {g}: 交互作用項が見つかりません (サンプル不足の可能性)')

        if interaction_p < 0.05:
            print(f'\n  → {dv}: age×menopause_proxy 交互作用は有意 (p={interaction_p:.4f})。')
            print('    閉経前後で「加齢軌跡そのもの」が変化している可能性を支持。')
        else:
            print(f'\n  → {dv}: age×menopause_proxy 交互作用は非有意 (p={interaction_p:.4f})。')
            print('    軌跡の傾きは共通と考えられ、群間差があるとすれば水準(切片)差のみの可能性が高い。')

        trajectory_models[dv] = {'full': model_full_dv, 'reduced': model_reduced_dv,
                                  'interaction_p': interaction_p, 'n': len(df_dv)}

    df_traj_summary = pd.DataFrame([
        {'dv': dv, 'n': v['n'], 'interaction_p': v['interaction_p'],
         'full_AIC': v['full'].aic, 'reduced_AIC': v['reduced'].aic,
         'full_adjR2': v['full'].rsquared_adj, 'reduced_adjR2': v['reduced'].rsquared_adj}
        for dv, v in trajectory_models.items()
    ])
    print('\n\n=== サマリー: age×menopause_proxy 交互作用検定 (CHI, α₁, α₂) ===')
    print(df_traj_summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    traj_summary_path = os.path.join(OUTPUT_DIR, 'female_menopause_trajectory_interaction.csv')
    df_traj_summary.to_csv(traj_summary_path, index=False)
    print(f'\n保存: {traj_summary_path}')
else:
    print('df_female / menopause_proxy が見つからないため、軌跡交互作用検定はスキップします (Step 12dを先に実行してください)')


In [ ]:
# ============================================================
# Step 12f: 可視化 — 女性内 CHI・α₁・α₂ の年齢軌跡 (pre/peri/post 群別)
# ============================================================
# Step 12eで推定した age×menopause_proxy 交互作用モデルの予測値を用いて、
# CHI・α₁・α₂それぞれの「群別の加齢軌跡」を同一年齢軸上に描く。
# α₁とα₂を並べることで、CHI軌跡の変化がどちらの成分(短距離 or 長距離)に由来するかを確認できる。

if 'trajectory_models' in globals() and 'df_female' in globals():
    proxy_colors_traj = {'pre (proxy, <45)': '#3A6EA5',
                          'peri (proxy, 45-54)': '#F0AD4E',
                          'post (proxy, 55+)': '#D9534F'}
    proxy_order_traj = ['pre (proxy, <45)', 'peri (proxy, 45-54)', 'post (proxy, 55+)']
    dv_titles = {'CHI': 'CHI', 'alpha1': 'α₁ (短距離, 4-16拍)', 'alpha2': 'α₂ (長距離, 16-64拍)'}

    fig4, axes4 = plt.subplots(1, 3, figsize=(18, 5.5), facecolor='#FFFFFF')

    for ax, dv in zip(axes4, ['CHI', 'alpha1', 'alpha2']):
        model_full_dv = trajectory_models[dv]['full']
        df_dv = df_female.dropna(subset=[dv, 'age_mid', 'menopause_proxy'])

        for g in proxy_order_traj:
            sub = df_dv[df_dv['menopause_proxy'] == g]
            if len(sub) == 0:
                continue
            ax.scatter(sub['age_mid'], sub[dv], s=14, alpha=0.45, color=proxy_colors_traj[g],
                       label=f'{g} (n={len(sub)})')

            # モデル予測線 (Step 12eのフルモデル) を、その群が実際に取り得る年齢範囲内でのみ描画
            x_range = np.linspace(sub['age_mid'].min(), sub['age_mid'].max(), 50)
            pred_df = pd.DataFrame({'age_mid': x_range, 'menopause_proxy': g})
            y_pred = model_full_dv.predict(pred_df)
            ax.plot(x_range, y_pred, color=proxy_colors_traj[g], lw=2.4)

        ax.axvline(45, color='#888888', ls='--', lw=0.7)
        ax.axvline(55, color='#888888', ls='--', lw=0.7)
        if dv == 'CHI':
            ax.axhline(0, color='#F0E68C', ls=':', lw=0.8)
        ax.set_xlabel('年齢 (群中央値, 歳)', color='#1A1A1A')
        ax.set_ylabel(dv_titles[dv], color='#1A1A1A')
        p_int = trajectory_models[dv]['interaction_p']
        ax.set_title(f'{dv_titles[dv]} 軌跡\n(age×proxy interaction p={p_int:.3f})',
                     color='#1A1A1A', fontsize=10)
        ax.legend(fontsize=7, facecolor='#FFFFFF', labelcolor='#1A1A1A')
        ax.set_facecolor('#FFFFFF')
        ax.grid(True, alpha=0.15)

    fig4.suptitle(f'女性内 加齢軌跡: CHI / α₁ / α₂ (閉経状態プロキシ別, N={len(df_female)})\n'
                  '折れ線はStep12eの交互作用モデルによる群別予測 (実測の閉経状態ではなくプロキシ分類)',
                  color='#1A1A1A', fontsize=12)
    fig4_path = os.path.join(OUTPUT_DIR, 'ecsoc_aging_female_trajectory_by_proxy.png')
    fig4.savefig(fig4_path, dpi=150, bbox_inches='tight', facecolor='#FFFFFF')
    plt.show()
    print(f'図保存: {fig4_path}')
else:
    print('trajectory_models / df_female が見つからないため、軌跡可視化はスキップします (Step 12eを先に実行してください)')


## Step 12e 頑健性チェック

Step 12e の結果 (CHI, α₁ で有意な age×proxy 交互作用) には、確認すべき点が2つある。

1. **多重共線性**: 全3モデルで condition number 警告が出ている (age_mid が非中心化のため)。
   中心化して再フィットし、交互作用検定のp値が変わらないか確認する
   (OLSでは中心化は線形な再パラメータ化に過ぎないため、ネストF検定のp値自体は理論上不変のはずだが、
   数値的な安定性を確認する意味で実施する)。
2. **より本質的な懸念**: pre/peri/post という3群への離散的な区分は、もし真の関係が連続的な
   二次曲線 (Step 10/11 で全体コホートに見出された inverted-U, peak ~55–59歳) であった場合、
   その滑らかな曲率を3本の折れ線で近似しているだけでも「交互作用あり」という結果になり得る。
   これは閉経移行期に特異的な軌跡変化ではなく、既知の二次的加齢パターンを離散近似の形で
   再検出しているだけの可能性がある。これを区別するため、「連続的な age + age² モデル」に対して
   閉経プロキシ (主効果 + age×proxy交互作用) を追加することで説明力が有意に上がるかを
   ネストF検定で確認する。上がらなければ、Step 12eの結果は連続的な曲率の離散近似の可能性が高い。


In [ ]:
# ============================================================
# Step 12g: Step 12e の頑健性チェック — 中心化 & 連続二次モデルとの比較
# ============================================================
if 'df_female' in globals() and 'menopause_proxy' in df_female.columns:
    ref_level = 'pre (proxy, <45)'
    robustness_results = {}

    for dv in ['CHI', 'alpha1', 'alpha2']:
        df_dv = df_female.dropna(subset=[dv, 'age_mid', 'menopause_proxy']).copy()
        df_dv['age_c'] = df_dv['age_mid'] - df_dv['age_mid'].mean()  # このセル内でのみ中心化

        print(f'\n\n=== {dv}: 頑健性チェック (N={len(df_dv)}) ===')

        # --- (a) 中心化ageでの再フィット (Step 12eと同じ交互作用モデルの定式化) ---
        f_full_c    = f'{dv} ~ age_c * C(menopause_proxy, Treatment(reference="{ref_level}"))'
        f_reduced_c = f'{dv} ~ age_c + C(menopause_proxy, Treatment(reference="{ref_level}"))'
        model_full_c    = smf.ols(f_full_c, data=df_dv).fit()
        model_reduced_c = smf.ols(f_reduced_c, data=df_dv).fit()
        nested_c = anova_lm(model_reduced_c, model_full_c)
        p_int_c = nested_c['Pr(>F)'].iloc[1]

        print(f'  (a) 中心化age での交互作用検定: p={p_int_c:.4f}  '
              f'(中心化後のcondition number: {model_full_c.condition_number:.1f})')
        print('      → Step 12e (非中心化) のp値と一致していれば、多重共線性が結果の')
        print('        頑健性を損なっていないと判断できる (センタリングは線形な再パラメータ化に')
        print('        過ぎないため、理論上ネストF検定のp値は不変)。')

        # --- (b) 連続二次モデル (age+age^2, proxyなし) vs 二次モデル+proxy(主効果+交互作用) ---
        f_quad_only  = f'{dv} ~ age_c + I(age_c**2)'
        f_quad_proxy = (f'{dv} ~ age_c + I(age_c**2) + '
                        f'C(menopause_proxy, Treatment(reference="{ref_level}")) + '
                        f'age_c:C(menopause_proxy, Treatment(reference="{ref_level}"))')
        model_quad_only  = smf.ols(f_quad_only, data=df_dv).fit()
        model_quad_proxy = smf.ols(f_quad_proxy, data=df_dv).fit()
        nested_quad = anova_lm(model_quad_only, model_quad_proxy)
        p_proxy_beyond_quad = nested_quad['Pr(>F)'].iloc[1]

        print(f'\n  (b) 連続二次モデルへの閉経プロキシ追加検定:')
        print(f'      age+age² のみ             AIC={model_quad_only.aic:.2f}, adj-R²={model_quad_only.rsquared_adj:.4f}')
        print(f'      age+age²+proxy(+交互作用)  AIC={model_quad_proxy.aic:.2f}, adj-R²={model_quad_proxy.rsquared_adj:.4f}')
        print(f'      ネストF検定 (proxy追加による説明力向上): p={p_proxy_beyond_quad:.4f}')
        if p_proxy_beyond_quad < 0.05:
            print(f'      → {dv}: 連続二次曲線だけでは説明できない、閉経プロキシ群固有の')
            print('        追加的な軌跡差が示唆される (Step 12eの結果は既知の曲率の離散的な')
            print('        再検出だけでは説明できない)。')
        else:
            print(f'      → {dv}: 連続二次モデルでほぼ説明可能であり、Step 12eで見られた')
            print('        「交互作用」は、既知のinverted-U型曲率を3群の折れ線で離散近似した')
            print('        結果である可能性が高い (閉経に特異的な追加的エビデンスとは言い切れない)。')

        robustness_results[dv] = {
            'p_interaction_centered':     p_int_c,
            'p_proxy_beyond_quadratic':   p_proxy_beyond_quad,
            'quad_only_AIC':              model_quad_only.aic,
            'quad_proxy_AIC':             model_quad_proxy.aic,
            'quad_only_adjR2':            model_quad_only.rsquared_adj,
            'quad_proxy_adjR2':           model_quad_proxy.rsquared_adj,
        }

    df_robust_summary = pd.DataFrame(robustness_results).T
    df_robust_summary.index.name = 'dv'
    print('\n\n=== サマリー: Step 12e 頑健性チェック (CHI, α₁, α₂) ===')
    print(df_robust_summary.to_string(float_format=lambda x: f'{x:.4f}'))
    robust_path = os.path.join(OUTPUT_DIR, 'female_menopause_trajectory_robustness.csv')
    df_robust_summary.to_csv(robust_path)
    print(f'\n保存: {robust_path}')

    print('\n\n=== 解釈上の注意 ===')
    print('・peri群 (45-54歳) は n=48 と小さく、係数の標準誤差が大きい (Step 12eの結果を参照)。')
    print('  そのため「移行期そのもの」の軌跡についての結論は現時点では出せない。')
    print('  現状で頑健に言えるのは「45歳未満 vs 55歳以上」の対比のみ。')
    print('・pre群 (n=532) が女性サンプルの大半を占めるため、post/peri群の推定は相対的に不安定になりやすい。')
else:
    print('df_female / menopause_proxy が見つからないため、頑健性チェックはスキップします (Step 12eを先に実行してください)')


## Step 12h–12k: VIF分解・三次モデル・制限三次スプラインによる追加感度分析

Step 12g の頑健性チェックで、交互作用検定のp値は中心化前後で不変であることを確認した
(理論通り、ネストF検定は再パラメータ化に対して不変)。一方で、二次モデルへのproxy追加検定では
CHI・α₂は非有意 (p=0.36, 0.10)、α₁のみ境界的 (p=0.09) という結果だった。

以下では、この結果がベースライン曲線の柔軟性 (二次 vs 三次 vs 制限三次スプライン) に
依存しないかを確認する。

**重要な前提の確認**: 本notebookの `age_mid` は `age_group` (1–15) を
`AGE_GROUP_MIDPOINT` で写像した値であり、真の連続変数ではなく **15水準の離散変数**
(18.5, 22, 27, ..., 88.5歳) である。また `menopause_proxy` は、この `age_mid` 自体の
閾値 (45歳・55歳) から決定論的に導出された区分関数であり、独立に測定された変数ではない。
したがって:

- proxy関連項のVIFが高くなるのは偶然ではなく構造的必然である
  (「proxyはage_midの一部を再コード化したものに過ぎない」という直感の裏付け)。
- 三次項・スプライン基底を増やしすぎると、15個のsupport pointへの過剰適合になりうる。
  特に peri群 (n=48)・post群 (n=56) のサイズを踏まえ、knot数・dfは控えめに選ぶ。

- **Step 12h**: 計画行列ベースのVIF分解 (どの項が共線性の影響を受けやすいか)
- **Step 12i**: 三次モデル (age+age²+age³) への拡張、proxy追加検定の頑健性確認
- **Step 12j**: 制限三次スプライン (restricted cubic spline, patsy `cr()`) への拡張、同上
- **Step 12k**: quadratic / cubic / spline 各ベースラインでのproxy追加検定を統合したサマリー


In [ ]:
# ============================================================
# Step 12h: 計画行列ベースのVIF分解 (交互作用モデル)
# ============================================================
if 'df_female' in globals() and 'menopause_proxy' in df_female.columns:
    from patsy import dmatrix
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    ref_level = 'pre (proxy, <45)'
    vif_results = {}

    for dv in ['CHI', 'alpha1', 'alpha2']:
        df_dv = df_female.dropna(subset=[dv, 'age_mid', 'menopause_proxy']).copy()
        df_dv['age_c'] = df_dv['age_mid'] - df_dv['age_mid'].mean()

        # Step 12g の f_quad_proxy と同一の計画行列を明示的に構築 (切片含む)
        f_rhs = (f'age_c + I(age_c**2) + '
                 f'C(menopause_proxy, Treatment(reference="{ref_level}")) + '
                 f'age_c:C(menopause_proxy, Treatment(reference="{ref_level}"))')
        X = dmatrix(f_rhs, data=df_dv, return_type='dataframe')

        vifs = pd.Series(
            [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
            index=X.columns
        )
        vif_results[dv] = vifs
        print(f'\n=== {dv}: VIF (交互作用モデル計画行列, N={len(df_dv)}) ===')
        print(vifs.round(1).to_string())

    df_vif_summary = pd.DataFrame(vif_results)
    print('\n\n=== サマリー: 項別VIF (CHI, alpha1, alpha2) ===')
    print(df_vif_summary.round(1).to_string())
    print('\n注意: menopause_proxyの各ダミー・交互作用項のVIFは、この項が単独で持つ情報量ではなく')
    print('age_c (およびage_c**2) と共有する分散の大きさを表す。proxyがage_midの区分関数である以上、')
    print('主効果・交互作用項のVIFが高くなること自体は想定内であり、「VIFが高い→結果無効」ではない。')
    print('個々の係数(例: 特定群のslope差)を単独で解釈する際にのみ、この共線性を考慮する必要がある。')
    print('ネストF検定 (Step 12g) は計画行列の列空間のみに依存するため、上記のVIFの大小に影響されない。')

    vif_path = os.path.join(OUTPUT_DIR, 'female_menopause_trajectory_vif.csv')
    df_vif_summary.to_csv(vif_path)
    print(f'\n保存: {vif_path}')
else:
    print('df_female / menopause_proxy が見つからないため、VIF分解はスキップします (Step 12dを先に実行してください)')


### Step 12i: 三次モデル (age+age²+age³) による感度分析

二次モデルでは説明できなかった部分が、より柔軟な三次曲線でも同様に説明できないかを確認する。
`age_mid` が15水準の離散変数であるため三次項の追加自体に無理はないが (N=636に対し追加自由度は1)、
年齢分布の両端 (post群 n=56) では推定が不安定になりやすい点に注意する。


In [ ]:
# ============================================================
# Step 12i: 三次モデルへの拡張 (age+age²+age³) + proxy追加検定
# ============================================================
if 'df_female' in globals() and 'menopause_proxy' in df_female.columns:
    ref_level = 'pre (proxy, <45)'
    cubic_results = {}

    for dv in ['CHI', 'alpha1', 'alpha2']:
        df_dv = df_female.dropna(subset=[dv, 'age_mid', 'menopause_proxy']).copy()
        df_dv['age_c'] = df_dv['age_mid'] - df_dv['age_mid'].mean()

        # (i) 二次 vs 三次: 三次項自体が必要かどうか
        f_quad  = f'{dv} ~ age_c + I(age_c**2)'
        f_cubic = f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)'
        m_quad  = smf.ols(f_quad, data=df_dv).fit()
        m_cubic = smf.ols(f_cubic, data=df_dv).fit()
        nested_cubic_term = anova_lm(m_quad, m_cubic)
        p_cubic_term = nested_cubic_term['Pr(>F)'].iloc[1]

        # (ii) 三次ベースライン vs 三次+proxy(主効果+交互作用)
        f_cubic_proxy = (f'{dv} ~ age_c + I(age_c**2) + I(age_c**3) + '
                          f'C(menopause_proxy, Treatment(reference="{ref_level}")) + '
                          f'age_c:C(menopause_proxy, Treatment(reference="{ref_level}"))')
        m_cubic_proxy = smf.ols(f_cubic_proxy, data=df_dv).fit()
        nested_proxy = anova_lm(m_cubic, m_cubic_proxy)
        p_proxy_beyond_cubic = nested_proxy['Pr(>F)'].iloc[1]

        print(f'\n\n=== {dv}: 三次モデル感度分析 (N={len(df_dv)}) ===')
        print(f'  三次項(age**3)自体の必要性: p={p_cubic_term:.4f}  (二次→三次でRSSが有意に減るか)')
        print(f'  三次ベースライン AIC={m_cubic.aic:.2f}, adj-R2={m_cubic.rsquared_adj:.4f}')
        print(f'  三次+proxy      AIC={m_cubic_proxy.aic:.2f}, adj-R2={m_cubic_proxy.rsquared_adj:.4f}')
        print(f'  ネストF検定 (proxy追加による説明力向上, 三次ベースライン比): p={p_proxy_beyond_cubic:.4f}')

        cubic_results[dv] = {
            'p_cubic_term_needed':      p_cubic_term,
            'p_proxy_beyond_cubic':     p_proxy_beyond_cubic,
            'cubic_only_AIC':           m_cubic.aic,
            'cubic_proxy_AIC':          m_cubic_proxy.aic,
            'cubic_only_adjR2':         m_cubic.rsquared_adj,
            'cubic_proxy_adjR2':        m_cubic_proxy.rsquared_adj,
        }

    df_cubic_summary = pd.DataFrame(cubic_results).T
    df_cubic_summary.index.name = 'dv'
    print('\n\n=== サマリー: 三次モデル感度分析 (CHI, alpha1, alpha2) ===')
    print(df_cubic_summary.to_string(float_format=lambda x: f'{x:.4f}'))
    cubic_path = os.path.join(OUTPUT_DIR, 'female_menopause_trajectory_cubic_sensitivity.csv')
    df_cubic_summary.to_csv(cubic_path)
    print(f'\n保存: {cubic_path}')
else:
    print('df_female / menopause_proxy が見つからないため、三次モデル感度分析はスキップします (Step 12dを先に実行してください)')


### Step 12j: 制限三次スプライン (restricted cubic spline) による感度分析

三次多項式は年齢範囲の両端で不安定になりやすいため、より安定した制限三次スプライン
(自然三次スプライン、`patsy.cr()`) でも同じ結論になるかを確認する。
knot数は、15水準しかない `age_mid` の分布と peri/post 群のサンプルサイズ (n=48, n=56) を
踏まえ、控えめに `df=3` (内部knot1個相当) とする。


In [ ]:
# ============================================================
# Step 12j: 制限三次スプラインへの拡張 + proxy追加検定
# ============================================================
if 'df_female' in globals() and 'menopause_proxy' in df_female.columns:
    from patsy import dmatrix

    ref_level = 'pre (proxy, <45)'
    SPLINE_DF = 3  # 控えめな柔軟性 (n_peri=48, n_post=56 を踏まえた保守的な選択)
    spline_results = {}

    for dv in ['CHI', 'alpha1', 'alpha2']:
        df_dv = df_female.dropna(subset=[dv, 'age_mid', 'menopause_proxy']).copy()
        df_dv['age_c'] = df_dv['age_mid'] - df_dv['age_mid'].mean()

        n_unique_ages = df_dv['age_mid'].nunique()
        print(f'\n\n=== {dv}: 制限三次スプライン感度分析 (N={len(df_dv)}, '
              f'age_mid固有値数={n_unique_ages}) ===')

        # cr() は自然三次スプライン基底を返す (patsy)。df=SPLINE_DFで内部knot数を制御。
        spline_basis = dmatrix(f'cr(age_c, df={SPLINE_DF}) - 1', data=df_dv, return_type='dataframe')
        spline_basis.columns = [f'age_spline_{i+1}' for i in range(spline_basis.shape[1])]
        df_dv = pd.concat([df_dv.reset_index(drop=True), spline_basis.reset_index(drop=True)], axis=1)
        spline_terms = ' + '.join(spline_basis.columns)

        f_spline_only  = f'{dv} ~ {spline_terms}'
        m_spline_only  = smf.ols(f_spline_only, data=df_dv).fit()

        # spline基底 x proxy の交互作用 (各基底関数ごとにproxy群で傾きを変える)
        interaction_terms = ' + '.join(
            f'{col}:C(menopause_proxy, Treatment(reference="{ref_level}"))' for col in spline_basis.columns
        )
        f_spline_proxy = (f'{dv} ~ {spline_terms} + '
                           f'C(menopause_proxy, Treatment(reference="{ref_level}")) + '
                           f'{interaction_terms}')
        m_spline_proxy = smf.ols(f_spline_proxy, data=df_dv).fit()

        nested_spline = anova_lm(m_spline_only, m_spline_proxy)
        p_proxy_beyond_spline = nested_spline['Pr(>F)'].iloc[1]

        print(f'  スプラインのみ    AIC={m_spline_only.aic:.2f}, adj-R2={m_spline_only.rsquared_adj:.4f}')
        print(f'  スプライン+proxy  AIC={m_spline_proxy.aic:.2f}, adj-R2={m_spline_proxy.rsquared_adj:.4f}')
        print(f'  ネストF検定 (proxy追加による説明力向上, スプラインベースライン比): p={p_proxy_beyond_spline:.4f}')

        spline_results[dv] = {
            'spline_df':                  SPLINE_DF,
            'p_proxy_beyond_spline':      p_proxy_beyond_spline,
            'spline_only_AIC':            m_spline_only.aic,
            'spline_proxy_AIC':           m_spline_proxy.aic,
            'spline_only_adjR2':          m_spline_only.rsquared_adj,
            'spline_proxy_adjR2':         m_spline_proxy.rsquared_adj,
        }

    df_spline_summary = pd.DataFrame(spline_results).T
    df_spline_summary.index.name = 'dv'
    print('\n\n=== サマリー: 制限三次スプライン感度分析 (CHI, alpha1, alpha2) ===')
    print(df_spline_summary.to_string(float_format=lambda x: f'{x:.4f}'))
    spline_path = os.path.join(OUTPUT_DIR, 'female_menopause_trajectory_spline_sensitivity.csv')
    df_spline_summary.to_csv(spline_path)
    print(f'\n保存: {spline_path}')
else:
    print('df_female / menopause_proxy が見つからないため、スプライン感度分析はスキップします (Step 12dを先に実行してください)')


### Step 12k: ベースライン曲線 (二次 / 三次 / スプライン) 横断サマリー

Step 12g・12i・12j で得た「proxy追加によるネストF検定のp値」を、ベースラインの柔軟性別に
並べる。3つのベースラインすべてで結論(有意/非有意)が一致していれば、Step 12e の交互作用は
特定のモデル定式化の産物ではないと言える。


In [ ]:
# ============================================================
# Step 12k: quadratic / cubic / spline 横断サマリー
# ============================================================
needed = ['robustness_results', 'cubic_results', 'spline_results']
if all(v in globals() for v in needed):
    rows = []
    for dv in ['CHI', 'alpha1', 'alpha2']:
        rows.append({
            'dv': dv,
            'p_proxy_beyond_quadratic': robustness_results[dv]['p_proxy_beyond_quadratic'],
            'p_proxy_beyond_cubic':     cubic_results[dv]['p_proxy_beyond_cubic'],
            'p_proxy_beyond_spline':    spline_results[dv]['p_proxy_beyond_spline'],
        })
    df_cross_summary = pd.DataFrame(rows).set_index('dv')
    df_cross_summary['all_baselines_agree_ns'] = (df_cross_summary < 0.05).sum(axis=1) == 0
    print('\n=== Step 12k: ベースライン曲線横断サマリー (proxy追加のネストF検定 p値) ===')
    print(df_cross_summary.to_string(float_format=lambda x: f'{x:.4f}'))
    print('\n(all_baselines_agree_ns = True: 二次・三次・スプラインいずれのベースラインでも')
    print(' proxy追加は非有意 -> 閉経プロキシに特異的な効果ではなく、既知の非線形加齢曲線で')
    print(' 説明可能という解釈がより頑健に支持される)')

    cross_path = os.path.join(OUTPUT_DIR, 'female_menopause_trajectory_cross_baseline_summary.csv')
    df_cross_summary.to_csv(cross_path)
    print(f'\n保存: {cross_path}')
else:
    print('Step 12g・12i・12jを先に実行してください (robustness_results / cubic_results / spline_results が必要)')


In [ ]:
# ============================================================
# Step 13: 可視化ダッシュボード
# ============================================================
AGE_ORDER = list(range(1, 16))
age_labels_ordered = [AGE_GROUP_LABELS[a] for a in AGE_ORDER]

# 年齢群のグラデーションカラー (若年=青寄り, 高齢=赤寄り)
cmap = plt.cm.coolwarm
age_colors = {ag: cmap(i / (len(AGE_ORDER)-1)) for i, ag in enumerate(AGE_ORDER)}

fig = plt.figure(figsize=(16, 11), facecolor='#FFFFFF')
gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)

axes = [
    fig.add_subplot(gs[0, 0]),  # A: CHI boxplot by age group
    fig.add_subplot(gs[0, 1]),  # B: CHI trend + quadratic fit
    fig.add_subplot(gs[1, 0]),  # C: PhaseV rate by age group
    fig.add_subplot(gs[1, 1]),  # D: R^2 by age group
    fig.add_subplot(gs[2, 0]),  # E: alpha1 vs alpha2 by age group
    fig.add_subplot(gs[2, 1]),  # F: scatter age vs CHI (individual, colored by sex if available)
]
for ax in axes:
    ax.set_facecolor('#FFFFFF')
    ax.tick_params(colors='#1A1A1A')
    for spine in ax.spines.values():
        spine.set_edgecolor('#CCCCCC')
    ax.grid(True, alpha=0.15)

# --- A: CHI boxplot by age group ---
ax = axes[0]
box_data = [df_clean[df_clean['age_group']==ag]['CHI'].dropna().values
            for ag in AGE_ORDER]
bp = ax.boxplot(box_data, positions=range(len(AGE_ORDER)), widths=0.6,
                 patch_artist=True, showfliers=False,
                 medianprops=dict(color='white', lw=1.5),
                 whiskerprops=dict(color='#5A6470'), capprops=dict(color='#5A6470'))
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(age_colors[AGE_ORDER[i]])
    patch.set_alpha(0.85)
ax.axhline(0, color='#F0E68C', ls=':', lw=0.8)
ax.set_xticks(range(len(AGE_ORDER)))
ax.set_xticklabels(age_labels_ordered, rotation=45, ha='right', fontsize=7, color='#1A1A1A')
ax.set_ylabel('CHI', color='#1A1A1A')
ax.set_title('A. CHI by Age Group', color='#1A1A1A', fontsize=10)

# --- B: CHI trend with quadratic fit + Fantasia benchmark ---
ax = axes[1]
ax.scatter(df_age_summary['age_group'].map(AGE_GROUP_MIDPOINT),
           df_age_summary['CHI_mean'], color='#3A6EA5', s=40, zorder=3, label='年齢群平均')
ax.errorbar(df_age_summary['age_group'].map(AGE_GROUP_MIDPOINT),
            df_age_summary['CHI_mean'],
            yerr=df_age_summary['CHI_sd'] / np.sqrt(df_age_summary['n']),
            fmt='none', ecolor='#3A6EA5', alpha=0.5, capsize=3)
age_fit_x = np.linspace(age_num.min(), age_num.max(), 100)
age_fit_c = age_fit_x - age_num.mean()
chi_fit_y = np.polyval(coeffs, age_fit_c)
ax.plot(age_fit_x, chi_fit_y, color='#D9534F', lw=2, ls='--',
        label=f'二次回帰 (a={coeffs[0]:+.5f}, p={p_quad:.2e})')
# Okabe (2026) ECSoC Paper 2 の Fantasia ベンチマーク値を参照線として重畳
ax.scatter([27], [0.011], marker='*', s=150, color='#5CB85C', zorder=5,
           label='Fantasia Young (Paper2, n=20)')
ax.scatter([76.5], [0.152], marker='*', s=150, color='#F0AD4E', zorder=5,
           label='Fantasia Elderly (Paper2, n=20)')
ax.axhline(0, color='#F0E68C', ls=':', lw=0.8)
ax.set_xlabel('年齢 (群中央値, 歳)', color='#1A1A1A')
ax.set_ylabel('CHI', color='#1A1A1A')
ax.set_title('B. CHI Trend vs Age (+ quadratic fit)', color='#1A1A1A', fontsize=10)
ax.legend(fontsize=7, facecolor='#FFFFFF', labelcolor='#1A1A1A')

# --- C: PhaseV rate by age group ---
ax = axes[2]
ax.bar(range(len(AGE_ORDER)), df_age_summary['PhaseV_rate'] * 100,
       color=[age_colors[ag] for ag in AGE_ORDER], alpha=0.85, edgecolor='white', lw=0.5)
ax.set_xticks(range(len(AGE_ORDER)))
ax.set_xticklabels(age_labels_ordered, rotation=45, ha='right', fontsize=7, color='#1A1A1A')
ax.set_ylabel('Phase V rate (%)', color='#1A1A1A')
ax.set_title('C. Phase V (R²<0.93) Rate by Age Group', color='#1A1A1A', fontsize=10)

# --- D: R^2 by age group ---
ax = axes[3]
ax.plot(range(len(AGE_ORDER)), df_age_summary['R2_mean'], marker='o',
        color='#5CB85C', lw=1.5)
ax.axhline(PHASE_V_THR, color='#D9534F', ls=':', lw=1, label=f'PhaseV閾値 ({PHASE_V_THR})')
ax.set_xticks(range(len(AGE_ORDER)))
ax.set_xticklabels(age_labels_ordered, rotation=45, ha='right', fontsize=7, color='#1A1A1A')
ax.set_ylabel('R² (mean)', color='#1A1A1A')
ax.set_title('D. DFA Fit Quality (R²) by Age Group', color='#1A1A1A', fontsize=10)
ax.legend(fontsize=7, facecolor='#FFFFFF', labelcolor='#1A1A1A')

# --- E: alpha1 vs alpha2 trajectories across age groups ---
ax = axes[4]
ax.plot(range(len(AGE_ORDER)), df_age_summary['alpha1_mean'], marker='o',
        color='#3A6EA5', lw=1.5, label='α₁ (短距離, 4-16拍)')
ax.plot(range(len(AGE_ORDER)), df_age_summary['alpha2_mean'], marker='s',
        color='#D9534F', lw=1.5, label='α₂ (長距離, 16-64拍)')
ax.set_xticks(range(len(AGE_ORDER)))
ax.set_xticklabels(age_labels_ordered, rotation=45, ha='right', fontsize=7, color='#1A1A1A')
ax.set_ylabel('DFA exponent', color='#1A1A1A')
ax.set_title('E. α₁ / α₂ Trajectories by Age Group', color='#1A1A1A', fontsize=10)
ax.legend(fontsize=8, facecolor='#FFFFFF', labelcolor='#1A1A1A')

# --- F: individual scatter, colored by sex if available ---
ax = axes[5]
if 'sex_label' in df_clean.columns:
    for label, color in [('Male', '#3A6EA5'), ('Female', '#D9534F')]:
        sub = df_clean[df_clean['sex_label'] == label]
        ax.scatter(sub['age_mid'], sub['CHI'], s=8, alpha=0.4, color=color, label=label)
    ax.legend(fontsize=8, facecolor='#FFFFFF', labelcolor='#1A1A1A')
else:
    ax.scatter(df_clean['age_mid'], df_clean['CHI'],
               s=8, alpha=0.3, color='#5A6470')
ax.axhline(0, color='#F0E68C', ls=':', lw=0.8)
ax.set_xlabel('年齢 (群中央値, 歳)', color='#1A1A1A')
ax.set_ylabel('CHI (individual)', color='#1A1A1A')
ax.set_title('F. Individual CHI Scatter (by Sex)', color='#1A1A1A', fontsize=10)

fig.suptitle('ECSoC Autonomic Aging — CHI Developmental Profile (N={})'.format(len(df_clean)),
             color='#1A1A1A', fontsize=13)

fig_path = os.path.join(OUTPUT_DIR, 'ecsoc_aging_dashboard.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight', facecolor='#FFFFFF')
plt.show()
print(f'図保存: {fig_path}')


In [ ]:
# ============================================================
# Step 14: 最終サマリー出力
# ============================================================
print('='*70)
print('ECSoC Autonomic Aging Analysis — 結果サマリー')
print('='*70)
print()
print(f'解析対象: Autonomic Aging Database (PhysioNet)')
print(f'処理対象: {len(target_df)}名 → 処理成功(QC前): {len(df_valid_results)}名 '
      f'→ QC後解析対象: {len(df_clean)}名 (QC除外 {int(qc_flag.sum())}名, '
      f'{100*qc_flag.mean():.1f}%)')
print()
print('─── 統計検定結果 ───')
print(df_stat_summary.to_string(index=False))
print()
print('─── 年齢群別 CHI サマリー ───')
print(df_age_summary[['age_label','n','CHI_mean','CHI_sd','PhaseV_rate']]
      .to_string(index=False))
print()
print('─── 出力ファイル ───')
for f in ['all_subjects_ecsoc.csv', 'qc_clean_subjects.csv', 'age_group_summary.csv',
          'age_trend_statistics.csv', 'ecsoc_aging_dashboard.png',
          'sanity_check_single_subject.png']:
    p = os.path.join(OUTPUT_DIR, f)
    exists = '✓' if os.path.exists(p) else '✗'
    print(f'  {exists} {p}')
print()
print('─── 次のステップ (Paper 2, Section 8.1 との接続) ───')
print('1. 二次回帰でU字/J字型が支持された場合:')
print('   → Okabe(2026) Paper 2の改訂発達モデルの独立・大規模な検証となる')
print('2. 支持されなかった場合:')
print('   → Fantasia (N=40) での知見が本データセット (N>1000) では再現しないことを意味し、')
print('     Paper 2の発達モデルの再検討が必要 (サンプルサイズの効果だった可能性を含む)')
print('3. 性別層別結果に有意差がある場合:')
print('   → CHIの加齢プロファイルに性差がある可能性; Paper 3の候補トピックとして記録')
print('4. PhaseV rateが年齢とともに単調増加する場合:')
print('   → 健常加齢でもスケーリング崩壊が漸増することを示唆し、')
print('     「健常高齢者にPhaseVは存在しない」という Paper 2 の前提の再検討が必要')


## 外部検証 (Fantasia + Normal Sinus Rhythm RR Interval Database)

自コホート(Autonomic Aging DB)とは独立した2つのPhysioNetデータベースでCHIの加齢パターンを検証する。
両データベースとも Google Drive 上の zip アーカイブから展開し、データベース提供の検証済みビートアノテーション(annotator='ecg')を用いる。
自前のxqrs/gqrs検出器は使わない(自己検出パイプラインのクセが結果に混入するのを避けるため)。

In [ ]:
# ============================================================
# Step 15a: 外部検証データセットの展開 (Google Drive上のzipから)
# ============================================================
# wgetでのオンラインダウンロードではなく、Drive上に置かれたzipアーカイブを
# 展開する方式に変更 (fantasia-database-1.0.0.zip / 
# normal-sinus-rhythm-rr-interval-database-1.0.0.zip)。
#
# zipファイルの場所が不明な場合、DRIVE_ROOT配下を再帰的に検索する。
# Driveが大きいと検索に時間がかかることがあるため、既知の場所があれば
# 下のZIP_SEARCH_DIRSに優先探索パスとして指定しておくと速い。

import zipfile

EXTERNAL_ROOT = os.path.join(DRIVE_ROOT, 'PhysioNet_External')
os.makedirs(EXTERNAL_ROOT, exist_ok=True)

# 既知の置き場所があればここに追加 (例: '/content/drive/MyDrive/PhysioNet_zips')
# 空リストのままなら DRIVE_ROOT 全体を再帰検索する (フォールバック)。
ZIP_SEARCH_DIRS = []

ZIP_TARGETS = {
    'fantasia': 'fantasia-database-1.0.0.zip',
    'nsr2db':   'normal-sinus-rhythm-rr-interval-database-1.0.0.zip',
}

def find_zip(filename):
    search_roots = ZIP_SEARCH_DIRS if ZIP_SEARCH_DIRS else [DRIVE_ROOT]
    for root in search_roots:
        matches = glob.glob(os.path.join(root, '**', filename), recursive=True)
        if matches:
            return matches[0]
    return None

for db_name, zip_filename in ZIP_TARGETS.items():
    local_dir = os.path.join(EXTERNAL_ROOT, db_name)
    os.makedirs(local_dir, exist_ok=True)

    existing_hea = glob.glob(os.path.join(local_dir, '**', '*.hea'), recursive=True)
    if existing_hea:
        print(f'{db_name}: 既に{len(existing_hea)}件の.heaファイルが展開済みです。スキップします。')
        continue

    print(f'{db_name}: {zip_filename} を検索中... (DRIVE_ROOT配下の再帰検索、時間がかかる場合あり)')
    zip_path = find_zip(zip_filename)
    if zip_path is None:
        print(f'  \u26a0 見つかりませんでした。ZIP_SEARCH_DIRSに正しいフォルダを指定するか、'
              f'ファイルパスを直接確認して再実行してください。')
        continue

    print(f'  発見: {zip_path}')
    print(f'  {local_dir} に展開中...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(local_dir)

    n_hea = len(glob.glob(os.path.join(local_dir, '**', '*.hea'), recursive=True))
    print(f'  展開完了: .heaファイル {n_hea}件 (再帰検索で確認)')
    if n_hea == 0:
        print(f'  \u26a0 展開はできましたが.heaファイルが見つかりません。'
              f'zip内部の構造を確認してください:')
        for root, dirs, files in os.walk(local_dir):
            depth = root.replace(local_dir, '').count(os.sep)
            print('  ' * (depth + 1) + os.path.basename(root) + '/', f'({len(files)} files)')


In [ ]:
# ============================================================
# Step 15b: 外部検証コア処理 (アノテーション由来、検出器再実行なし)
# ============================================================
import re

def process_external_subject(record_path, annotator='ecg'):
    """データベース提供済みのビートアノテーションからCHIを計算する。
       xqrs/gqrsは使わない (検証済みアノテーションをそのまま信頼する)。"""
    try:
        hdr = wfdb.rdheader(record_path)
        fs = hdr.fs

        # ヘッダーコメントから年齢・性別を抽出 (例: "# Age: 21 Sex: F")
        age, sex = np.nan, None
        for c in hdr.comments:
            m = re.search(r'Age:\s*([\d.]+)', c)
            if m: age = float(m.group(1))
            m2 = re.search(r'Sex:\s*([MF])', c)
            if m2: sex = m2.group(1)

        ann = wfdb.rdann(record_path, annotator)
        qrs_inds = np.asarray(ann.sample, dtype=int)

        rr_raw = np.diff(qrs_inds) / fs * 1000.0
        rr_clean = rr_ectopic_filter(rr_raw)

        if len(rr_clean) < MIN_BEATS_REQUIRED:
            return {'record': os.path.basename(record_path), 'age': age, 'sex': sex,
                    'error': f'too_few_beats ({len(rr_clean)})', 'n_beats_raw': len(rr_raw)}

        dfa = dfa_ecsoc_beats(rr_clean)
        if dfa is None:
            return {'record': os.path.basename(record_path), 'age': age, 'sex': sex,
                    'error': 'dfa_failed', 'n_beats_raw': len(rr_raw), 'n_beats_clean': len(rr_clean)}

        row = {'record': os.path.basename(record_path), 'age': age, 'sex': sex, 'error': None,
               'n_beats_raw': len(rr_raw), 'n_beats_clean': len(rr_clean),
               'mean_RR_ms': float(np.mean(rr_clean)), 'SDNN_ms': float(np.std(rr_clean, ddof=1))}
        row.update(dfa)
        return row
    except Exception as e:
        return {'record': os.path.basename(record_path), 'age': np.nan, 'sex': None,
                'error': f'exception: {e}'}

# --- Fantasia (n=40, annotator='ecg', fs=250) ---
# 修正: os.listdir(直下のみ)ではなく再帰globを使う。
# wget --cut-dirs の数がURL階層と一致しない場合、ファイルは local_dir/1.0.0/ のような
# サブディレクトリに入ることがあるため、直下決め打ちは環境によって0件になる。
fantasia_dir = os.path.join(EXTERNAL_ROOT, 'fantasia')
fantasia_hea_paths = sorted(set(
    p[:-4] for p in glob.glob(os.path.join(fantasia_dir, '**', '*.hea'), recursive=True)
))
print(f'Fantasia: {len(fantasia_hea_paths)}レコード (探索ルート: {fantasia_dir})')
if len(fantasia_hea_paths) == 0:
    print('  \u26a0 .heaファイルが見つかりません。実際のディレクトリ構造を確認してください:')
    for root, dirs, files in os.walk(fantasia_dir):
        depth = root.replace(fantasia_dir, '').count(os.sep)
        print('  ' * depth + os.path.basename(root) + '/', f'({len(files)} files)')

fantasia_results = []
for rp in fantasia_hea_paths:
    rec_name = os.path.basename(rp)
    res = process_external_subject(rp, annotator='ecg')
    res['record'] = rec_name
    res['group'] = 'young' if 'y' in rec_name else 'elderly'
    fantasia_results.append(res)

df_fantasia = pd.DataFrame(fantasia_results)
if len(df_fantasia) > 0:
    print(df_fantasia[['record', 'age', 'sex', 'group', 'error', 'n_beats_clean', 'CHI']].to_string(index=False))
else:
    print('  \u26a0 df_fantasia が空です。上記のディレクトリツリーを確認し、必要ならglobパターンを調整してください。')

# --- NSR-RR (n=54, annotator='ecg', fs=128) ---
# 同じ理由で修正: 再帰globに変更
nsr_dir = os.path.join(EXTERNAL_ROOT, 'nsr2db')
nsr_hea_paths = sorted(set(
    p[:-4] for p in glob.glob(os.path.join(nsr_dir, '**', '*.hea'), recursive=True)
))
print(f'\nNSR-RR: {len(nsr_hea_paths)}レコード (探索ルート: {nsr_dir})')
if len(nsr_hea_paths) == 0:
    print('  \u26a0 .heaファイルが見つかりません。実際のディレクトリ構造を確認してください:')
    for root, dirs, files in os.walk(nsr_dir):
        depth = root.replace(nsr_dir, '').count(os.sep)
        print('  ' * depth + os.path.basename(root) + '/', f'({len(files)} files)')

nsr_results = []
for rp in nsr_hea_paths:
    rec_name = os.path.basename(rp)
    res = process_external_subject(rp, annotator='ecg')
    res['record'] = rec_name
    nsr_results.append(res)

df_nsr = pd.DataFrame(nsr_results)
if len(df_nsr) > 0:
    print(df_nsr[['record', 'age', 'sex', 'error', 'n_beats_clean', 'CHI']].to_string(index=False))
else:
    print('  \u26a0 df_nsr が空です。上記のディレクトリツリーを確認し、必要ならglobパターンを調整してください。')

# 保存
df_fantasia.to_csv(os.path.join(OUTPUT_DIR, 'external_fantasia_results.csv'), index=False)
df_nsr.to_csv(os.path.join(OUTPUT_DIR, 'external_nsr_results.csv'), index=False)


In [ ]:
# ============================================================
# Step 15c: 検証① Fantasia 若年 vs 高齢 (方向性の予測検定)
# ============================================================
from scipy import stats as sps

fy = df_fantasia[(df_fantasia['group']=='young') & (df_fantasia['error'].isna())]['CHI']
fo = df_fantasia[(df_fantasia['group']=='elderly') & (df_fantasia['error'].isna())]['CHI']

print(f'Fantasia 若年群 (21-34歳, n={len(fy)}): CHI mean={fy.mean():.4f}, sd={fy.std():.4f}')
print(f'Fantasia 高齢群 (68-85歳, n={len(fo)}): CHI mean={fo.mean():.4f}, sd={fo.std():.4f}')

u_stat, p_mw = sps.mannwhitneyu(fy, fo, alternative='two-sided')
print(f'Mann-Whitney U検定: U={u_stat:.1f}, p={p_mw:.4f}')

# 自コホートの対応する年齢群との比較 (参考)
own_young = df_clean[df_clean['age_group'].isin([2,3])]['CHI']  # 20-29歳
own_old   = df_clean[df_clean['age_group'].isin([12,13])]['CHI']  # 70-79歳
print(f'\n(参考) 自コホート 20-29歳 (n={len(own_young)}): CHI mean={own_young.mean():.4f}')
print(f'(参考) 自コホート 70-79歳 (n={len(own_old)}): CHI mean={own_old.mean():.4f}')


In [ ]:
# ============================================================
# Step 15d: 検証② NSR-RR で二次モデルを独立に再フィット
# ============================================================
d_nsr = df_nsr[df_nsr['error'].isna()].dropna(subset=['age','CHI']).copy()
x, y = d_nsr['age'].values, d_nsr['CHI'].values
n = len(y)
lin, quad = np.polyfit(x, y, 1), np.polyfit(x, y, 2)
ssr_lin = np.sum((y-np.polyval(lin,x))**2); ssr_quad = np.sum((y-np.polyval(quad,x))**2)
ss_tot = np.sum((y-y.mean())**2)
f_stat = ((ssr_lin-ssr_quad)/1)/(ssr_quad/(n-3))
p_value = 1 - sps.f.cdf(f_stat, 1, n-3)

print(f'NSR-RR (n={n}, 年齢範囲 {x.min():.0f}-{x.max():.0f}歳):')
print(f'  線形R²={1-ssr_lin/ss_tot:.4f}, 二次R²={1-ssr_quad/ss_tot:.4f}')
print(f'  二次係数={quad[0]:.6f} (自コホートの符号: 負)')
print(f'  F(1,{n-3})={f_stat:.2f}, p={p_value:.4f}')


## Step 13: CHI・alpha1・alpha2 ピーク年齢の被験者単位ペアードブートストラップ

### 目的
CHI (~44.9歳), alpha1 (~57歳), alpha2 (~65-67歳) のピーク年齢が、記述的な観察を超えて
統計的に順序づけられる (位相がずれている) と言えるかを検定する。

### 方法上のポイント
3指標は同一被験者から計算されているため、独立に3回ブートストラップするのではなく、
**1回のブートストラップ復元抽出で選ばれた同じ被験者集合に対して、CHI・alpha1・alpha2の
3つのモデルを同時にfit**する (paired/coupled bootstrap)。これにより、被験者間の個人差に
由来する相関構造を保ったまま、ピーク年齢差 (alpha1-CHI, alpha2-alpha1, alpha2-CHI) の
不確実性を評価できる。

### モデル選択 (Step 11aの結果に準拠)
- **CHI**: 三次項が非有意 (Step 11a, p=0.95) — 二次モデルの頂点年齢を使用
- **alpha1**: 三次項が有意 — 三次モデルの極大 (非対称ピーク) を使用
- **alpha2**: 三次項が有意 — 三次モデルの極大 (60代の遅いピーク) を使用。
  谷はここでは扱わない (今回は3指標間の「ピーク」順序の検定が目的のため)

三次モデルは観測範囲内に複数の極値を持ちうるため、フルサンプルでの点推定に最も近い
極大値を各ブートストラップ標本でも追跡する (曖昧性解消のためのアンカー方式)。


In [ ]:
# ============================================================
# Step 13: CHI / alpha1 / alpha2 ピーク年齢 — 被験者単位ペアードブートストラップ
# ============================================================
if 'df_clean' in globals():
    N_BOOT_PAIRED = 2000     # スプライン計算を含まないため軽量。時間に余裕があれば5000に
    RNG_SEED_PAIRED = 123

    df_joint = df_clean.dropna(subset=['CHI', 'alpha1', 'alpha2', 'age_mid']).reset_index(drop=True).copy()
    n_subj_joint = len(df_joint)
    age_min_j, age_max_j = df_joint['age_mid'].min(), df_joint['age_mid'].max()
    print(f'ペアードブートストラップ対象: N={n_subj_joint} (CHI・alpha1・alpha2すべて非欠損の被験者)')
    print(f'年齢範囲: [{age_min_j:.1f}, {age_max_j:.1f}]')

    def get_quad_vertex(boot, dv, age_mean_b):
        m = smf.ols(f'{dv} ~ age_c + I(age_c**2)', data=boot).fit()
        b2 = m.params['I(age_c ** 2)']
        c  = m.params['age_c']
        if b2 >= 0:
            return np.nan
        vertex_age = age_mean_b - c / (2*b2)
        if not (age_min_j <= vertex_age <= age_max_j):
            return np.nan
        return vertex_age

    def get_cubic_peak(boot, dv, age_mean_b, anchor_age):
        m = smf.ols(f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)', data=boot).fit()
        a  = m.params['I(age_c ** 3)']
        b2 = m.params['I(age_c ** 2)']
        c  = m.params['age_c']
        if a == 0:
            return np.nan
        roots = np.roots([3*a, 2*b2, c])
        roots = np.real(roots[np.isreal(roots)])
        cand_ages = [age_mean_b + xc for xc in roots if age_min_j <= age_mean_b + xc <= age_max_j]
        maxima = [x for x in cand_ages if (6*a*(x - age_mean_b) + 2*b2) < 0]   # 極大のみ
        if not maxima:
            return np.nan
        return min(maxima, key=lambda x: abs(x - anchor_age))  # アンカーに最も近い極大を採用

    # --- フルサンプルでの点推定 (ブートストラップのアンカーとして使用) ---
    age_mean_full = df_joint['age_mid'].mean()
    df_joint['age_c'] = df_joint['age_mid'] - age_mean_full
    anchor_chi    = get_quad_vertex(df_joint, 'CHI', age_mean_full)
    anchor_alpha1 = get_cubic_peak(df_joint, 'alpha1', age_mean_full, 57.0)   # 初期アンカーは既知の近似値
    anchor_alpha2 = get_cubic_peak(df_joint, 'alpha2', age_mean_full, 67.0)
    ANCHOR_AGE = {'CHI': anchor_chi, 'alpha1': anchor_alpha1, 'alpha2': anchor_alpha2}
    print(f'\nフルサンプル点推定 (ピーク年齢): CHI={anchor_chi:.1f}歳, alpha1={anchor_alpha1:.1f}歳, alpha2={anchor_alpha2:.1f}歳')

    rng = np.random.default_rng(RNG_SEED_PAIRED)
    records = []
    for bi in range(N_BOOT_PAIRED):
        idx = rng.integers(0, n_subj_joint, n_subj_joint)
        boot = df_joint.iloc[idx].reset_index(drop=True)
        age_mean_b = boot['age_mid'].mean()
        boot['age_c'] = boot['age_mid'] - age_mean_b

        rec = {'boot_id': bi}
        try:
            rec['CHI_peak']    = get_quad_vertex(boot, 'CHI', age_mean_b)
            rec['alpha1_peak'] = get_cubic_peak(boot, 'alpha1', age_mean_b, ANCHOR_AGE['alpha1'])
            rec['alpha2_peak'] = get_cubic_peak(boot, 'alpha2', age_mean_b, ANCHOR_AGE['alpha2'])
        except Exception:
            rec['CHI_peak'] = np.nan
            rec['alpha1_peak'] = np.nan
            rec['alpha2_peak'] = np.nan

        records.append(rec)
        if (bi + 1) % 400 == 0:
            print(f'  進捗: {bi+1}/{N_BOOT_PAIRED}')

    df_paired = pd.DataFrame(records)
    df_paired['diff_alpha1_minus_CHI']    = df_paired['alpha1_peak'] - df_paired['CHI_peak']
    df_paired['diff_alpha2_minus_alpha1'] = df_paired['alpha2_peak'] - df_paired['alpha1_peak']
    df_paired['diff_alpha2_minus_CHI']    = df_paired['alpha2_peak'] - df_paired['CHI_peak']

    paired_path = os.path.join(OUTPUT_DIR, 'peak_age_paired_bootstrap.csv')
    df_paired.to_csv(paired_path, index=False)

    n_valid = df_paired[['CHI_peak', 'alpha1_peak', 'alpha2_peak']].dropna().shape[0]
    print(f'\n\n=== Step 13 サマリー: ピーク年齢のペアードブートストラップ差 (N_BOOT={N_BOOT_PAIRED}) ===')
    print(f'3指標すべて有効なピークが得られたブートストラップ標本数: {n_valid}/{N_BOOT_PAIRED}\n')

    for col, label in [('diff_alpha1_minus_CHI', 'alpha1 peak - CHI peak'),
                        ('diff_alpha2_minus_alpha1', 'alpha2 peak - alpha1 peak'),
                        ('diff_alpha2_minus_CHI', 'alpha2 peak - CHI peak')]:
        vals = df_paired[col].dropna()
        if len(vals) == 0:
            print(f'{label}: 有効な差分なし\n')
            continue
        med = vals.median()
        lo, hi = np.percentile(vals, [2.5, 97.5])
        pct_positive = (vals > 0).mean() * 100
        print(f'{label}:')
        print(f'  中央値={med:.1f}歳, 95%CI=[{lo:.1f}, {hi:.1f}]歳, 差>0の割合={pct_positive:.1f}% (n={len(vals)})\n')

    print(f'保存: {paired_path}')
    print('\n判定の目安: 95%CIが0をまたがず、差>0(または<0)の割合が97.5%以上なら、')
    print('  その順序関係(位相のずれ)はブートストラップ上頑健に支持されたとみなせる。')
    print('  0をまたぐ、または大きく割れる場合は、「記述的な多相性」以上の主張は時期尚早。')
else:
    print('df_clean が見つからないため、Step 13はスキップします (Step 9.5を先に実行してください)')


## Figure 1 (paper): CHI・alpha1・alpha2の標準化年齢軌跡

3指標を同一軸上で比較できるよう、それぞれを全体平均・SDでz化し、年齢群ごとの平均±SEを
重ねてプロットする。Step 13で得たピーク年齢（anchor_chi/alpha1/alpha2）を縦の点線で示す。
Step 13が未実行の場合でも、このセル単独でクイックに点推定を再計算して動作するようにしてある。


In [ ]:
# ============================================================
# Figure 1 (paper): CHI・alpha1・alpha2の標準化年齢軌跡
# ============================================================
if 'df_clean' in globals():
    FIG_COLORS = {'CHI': '#3A6EA5', 'alpha1': '#5A9367', 'alpha2': '#D9534F'}
    FIG_LABELS = {'CHI': 'CHI', 'alpha1': r'$\alpha_1$', 'alpha2': r'$\alpha_2$'}
    FIG_MODEL  = {'CHI': 'quad', 'alpha1': 'cubic', 'alpha2': 'cubic'}  # 主解析と同じモデル仕様

    # --- Step13が未実行でも動くよう、このセル単独でピーク年齢を再計算できるフォールバック ---
    def _quick_quad_vertex(df, dv):
        d = df.dropna(subset=[dv, 'age_mid']).copy()
        age_mean = d['age_mid'].mean()
        d['age_c'] = d['age_mid'] - age_mean
        m = smf.ols(f'{dv} ~ age_c + I(age_c**2)', data=d).fit()
        b2, c = m.params['I(age_c ** 2)'], m.params['age_c']
        return (age_mean - c / (2*b2)) if b2 < 0 else np.nan

    def _quick_cubic_peak(df, dv):
        d = df.dropna(subset=[dv, 'age_mid']).copy()
        age_mean = d['age_mid'].mean()
        age_min_, age_max_ = d['age_mid'].min(), d['age_mid'].max()
        d['age_c'] = d['age_mid'] - age_mean
        m = smf.ols(f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)', data=d).fit()
        a, b2, c = m.params['I(age_c ** 3)'], m.params['I(age_c ** 2)'], m.params['age_c']
        roots = np.roots([3*a, 2*b2, c]) if a != 0 else np.array([])
        roots = np.real(roots[np.isreal(roots)])
        cands = [age_mean + r for r in roots if age_min_ <= age_mean + r <= age_max_]
        maxima = [x for x in cands if 6*a*(x - age_mean) + 2*b2 < 0]
        if not maxima:
            return np.nan
        return maxima[int(np.argmax([-(6*a*(x - age_mean) + 2*b2) for x in maxima]))]

    peak_ages = {}
    peak_ages['CHI']    = anchor_chi    if 'anchor_chi'    in globals() else _quick_quad_vertex(df_clean, 'CHI')
    peak_ages['alpha1'] = anchor_alpha1 if 'anchor_alpha1' in globals() else _quick_cubic_peak(df_clean, 'alpha1')
    peak_ages['alpha2'] = anchor_alpha2 if 'anchor_alpha2' in globals() else _quick_cubic_peak(df_clean, 'alpha2')

    # --- 標準化 (各指標を全体平均・SDでz化) ---
    df_fig1 = df_clean.copy()
    for dv in ['CHI', 'alpha1', 'alpha2']:
        df_fig1[f'{dv}_z'] = (df_fig1[dv] - df_fig1[dv].mean()) / df_fig1[dv].std()

    age_min_all, age_max_all = df_clean['age_mid'].min(), df_clean['age_mid'].max()
    x_grid = np.linspace(age_min_all, age_max_all, 300)

    fig, ax = plt.subplots(figsize=(9, 6), facecolor='#FFFFFF')
    ax.set_facecolor('#FFFFFF')

    # 生の年齢群平均(ノイズが大きい, 特に高齢群は小標本)は背景の薄いマーカーとしてのみ示し、
    # ピーク年齢の根拠そのものである「モデルフィット曲線」を主線として描く。
    # (以前の版では生データの折れ線を主線にしていたため、ピーク年齢の点線と
    #  視覚的な山の位置が一致しないことがあった。)
    for dv in ['CHI', 'alpha1', 'alpha2']:
        d = df_fig1.dropna(subset=[f'{dv}_z', 'age_mid']).copy()
        age_mean = d['age_mid'].mean()
        d['age_c'] = d['age_mid'] - age_mean
        xc_grid = x_grid - age_mean

        if FIG_MODEL[dv] == 'quad':
            m = smf.ols(f'{dv}_z ~ age_c + I(age_c**2)', data=d).fit()
            y_fit = np.polyval([m.params['I(age_c ** 2)'], m.params['age_c'], m.params['Intercept']], xc_grid)
        else:
            m = smf.ols(f'{dv}_z ~ age_c + I(age_c**2) + I(age_c**3)', data=d).fit()
            y_fit = np.polyval([m.params['I(age_c ** 3)'], m.params['I(age_c ** 2)'],
                                 m.params['age_c'], m.params['Intercept']], xc_grid)

        grp = d.groupby('age_group')[f'{dv}_z'].agg(['mean', 'std', 'count']).reindex(AGE_ORDER)
        se = grp['std'] / np.sqrt(grp['count'])
        xg = [AGE_GROUP_MIDPOINT[a] for a in AGE_ORDER]
        ax.errorbar(xg, grp['mean'], yerr=se, fmt='o', color=FIG_COLORS[dv], alpha=0.35,
                     markersize=4, elinewidth=1, capsize=0, zorder=2)

        ax.plot(x_grid, y_fit, color=FIG_COLORS[dv], lw=2.5, label=FIG_LABELS[dv], zorder=4)

    ax.axhline(0, color='#999999', lw=1, ls='-', alpha=0.4)
    ax.set_xlim(age_min_all - 1, age_max_all + 1)

    y_top = ax.get_ylim()[1]
    for dv in ['CHI', 'alpha1', 'alpha2']:
        if not np.isnan(peak_ages[dv]):
            ax.axvline(peak_ages[dv], color=FIG_COLORS[dv], ls=':', lw=1.6, alpha=0.8, zorder=3)
            ax.text(peak_ages[dv], y_top * 0.97, f'{peak_ages[dv]:.1f}', color=FIG_COLORS[dv],
                    fontsize=9, ha='center', va='top')

    ax.set_xlabel('Age (years)', fontsize=11, color='#1A1A1A')
    ax.set_ylabel('Standardized value (z-score)', fontsize=11, color='#1A1A1A')
    ax.set_title('Figure 1. Standardized age trajectories of CHI, ' + r'$\alpha_1$' + ', and ' + r'$\alpha_2$' +
                 '\n(markers: raw age-group mean ' + r'$\pm$' + ' SE; lines: fitted model curves; dotted: estimated peak ages)',
                 fontsize=10.5, color='#1A1A1A')
    ax.tick_params(colors='#1A1A1A')
    for spine in ax.spines.values():
        spine.set_edgecolor('#CCCCCC')
    ax.grid(True, alpha=0.15)
    ax.legend(fontsize=10, facecolor='#FFFFFF', labelcolor='#1A1A1A', loc='upper right')

    plt.tight_layout()
    fig1_path = os.path.join(OUTPUT_DIR, 'ecsoc_fig1_standardized_trajectories.png')
    plt.savefig(fig1_path, dpi=200, facecolor='#FFFFFF')
    plt.show()

    print(f'保存: {fig1_path}')
    print(f'peak ages used: CHI={peak_ages["CHI"]:.1f}, alpha1={peak_ages["alpha1"]:.1f}, alpha2={peak_ages["alpha2"]:.1f}')
else:
    print("df_clean が見つからないため、Figure1はスキップします")


## Figure 2 (paper): モデル仕様感度 (quadratic / cubic / spline)

CHI・alpha1・alpha2それぞれについて、二次・三次・restricted cubic spline (df=4) の3モデルを
同一パネルに重ねる。x軸は常に観測年齢範囲 [age_min, age_max] に固定しており(`ax.set_xlim`)、
Step 11bで起きていたような「三次係数がゼロに近い場合に変曲点が観測範囲外へ発散し、
axvline経由でx軸が壊れる」現象が起こらない設計にしてある。曲線自体も観測範囲内のグリッドでしか
評価しないため、発散した外挿値がプロットに混入することもない。


In [ ]:
# ============================================================
# Figure 2 (paper): モデル仕様感度 (quadratic / cubic / spline)
# ============================================================
if 'df_clean' in globals():
    from patsy import dmatrix, build_design_matrices
    from statsmodels.stats.anova import anova_lm

    FIG2_DF_SPLINE = 4
    _DV_TITLE = {'CHI': 'CHI', 'alpha1': r'$\alpha_1$', 'alpha2': r'$\alpha_2$'}

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), facecolor='#FFFFFF')
    fig2_summary = []

    for ax, dv in zip(axes, ['CHI', 'alpha1', 'alpha2']):
        d = df_clean.dropna(subset=[dv, 'age_mid']).copy()
        age_mean = d['age_mid'].mean()
        age_min_, age_max_ = d['age_mid'].min(), d['age_mid'].max()
        d['age_c'] = d['age_mid'] - age_mean

        m_quad = smf.ols(f'{dv} ~ age_c + I(age_c**2)', data=d).fit()
        m_cubic = smf.ols(f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)', data=d).fit()

        spline_basis = dmatrix(f'cr(age_c, df={FIG2_DF_SPLINE}) - 1', data=d, return_type='dataframe')
        spline_cols = [f'sp_{i+1}' for i in range(spline_basis.shape[1])]
        spline_basis.columns = spline_cols
        d = pd.concat([d.reset_index(drop=True), spline_basis.reset_index(drop=True)], axis=1)
        m_spline = smf.ols(f'{dv} ~ ' + ' + '.join(spline_cols), data=d).fit()

        p_spline_vs_quad = anova_lm(m_quad, m_spline)['Pr(>F)'].iloc[1]

        # --- 予測グリッドは常に観測範囲内 [age_min_, age_max_] に固定 (発散を防ぐ核心部分) ---
        x_grid = np.linspace(age_min_, age_max_, 300)
        xc_grid = x_grid - age_mean
        y_quad = np.polyval([m_quad.params['I(age_c ** 2)'], m_quad.params['age_c'],
                              m_quad.params['Intercept']], xc_grid)
        y_cubic = np.polyval([m_cubic.params['I(age_c ** 3)'], m_cubic.params['I(age_c ** 2)'],
                               m_cubic.params['age_c'], m_cubic.params['Intercept']], xc_grid)
        grid_basis = np.asarray(build_design_matrices([spline_basis.design_info], {'age_c': xc_grid})[0])
        y_spline = grid_basis @ m_spline.params[spline_cols].values
        if 'Intercept' in m_spline.params.index:
            y_spline = y_spline + m_spline.params['Intercept']

        ax.set_facecolor('#FFFFFF')
        ax.scatter(d['age_mid'], d[dv], s=8, alpha=0.12, color='#8899AA')
        grp = d.groupby('age_mid', as_index=False)[dv].mean()
        ax.scatter(grp['age_mid'], grp[dv], s=42, color='#3A3A3A', zorder=4, label='Age-group mean')
        ax.plot(x_grid, y_quad, color='#5CB85C', lw=1.6, ls='--', label='Quadratic')
        ax.plot(x_grid, y_cubic, color='#D9534F', lw=1.8, ls='-', label='Cubic')
        ax.plot(x_grid, y_spline, color='#1A1A1A', lw=2.0, label=f'Spline (df={FIG2_DF_SPLINE})')
        ax.set_xlim(age_min_ - 1, age_max_ + 1)  # 発散した極値/変曲点があってもx軸は常に観測範囲に固定

        ax.set_title(f'{_DV_TITLE[dv]}  (spline vs quad, p={p_spline_vs_quad:.3f})',
                     fontsize=10.5, color='#1A1A1A')
        ax.set_xlabel('Age (years)', color='#1A1A1A')
        if dv == 'CHI':
            ax.set_ylabel('Value', color='#1A1A1A')
        ax.tick_params(colors='#1A1A1A')
        for spine in ax.spines.values():
            spine.set_edgecolor('#CCCCCC')
        ax.grid(True, alpha=0.15)
        ax.legend(fontsize=7.5, facecolor='#FFFFFF', labelcolor='#1A1A1A')

        fig2_summary.append({'dv': dv, 'p_spline_vs_quad': p_spline_vs_quad,
                              'quad_AIC': m_quad.aic, 'cubic_AIC': m_cubic.aic, 'spline_AIC': m_spline.aic})

    fig.suptitle('Figure 2. Model-specification sensitivity (quadratic / cubic / spline)',
                  fontsize=12, color='#1A1A1A', y=1.03)
    plt.tight_layout()
    fig2_path = os.path.join(OUTPUT_DIR, 'ecsoc_fig2_model_sensitivity.png')
    plt.savefig(fig2_path, dpi=200, facecolor='#FFFFFF', bbox_inches='tight')
    plt.show()
    print(f'保存: {fig2_path}')

    df_fig2_summary = pd.DataFrame(fig2_summary)
    print(df_fig2_summary.to_string(index=False))
    fig2_csv_path = os.path.join(OUTPUT_DIR, 'fig2_model_sensitivity_summary.csv')
    df_fig2_summary.to_csv(fig2_csv_path, index=False)
    print(f'保存: {fig2_csv_path}')
else:
    print("df_clean が見つからないため、Figure2はスキップします")


## Step 15e: 外部データセットの結合とカバレッジ確認

Fantasia (二群デザイン: 若年21-34歳 / 高齢68-85歳) と NSR-RR (連続年齢, 約28-76歳) を
結合し、Step 13 (自コホート) と同じ枠組みでCHI・alpha1・alpha2のピーク年齢を検証する。

**設計上の注意**
- Fantasiaは連続年齢ではなく二群デザインのため、単独では曲線当てはめ・ピーク推定はできない
  (Step 15cの方向性検定にのみ使える)。
- NSR-RRは連続年齢だが、記録条件 (24hホルター) がFantasia (仰臥位安静・約120分) と異なるため、
  絶対水準に系統的なオフセットが生じている可能性がある。
  → データセットをダミー変数として切片にのみ含め (年齢項とは交互作用させない)、
    この水準差を吸収する。ピーク年齢の位置は切片シフトの影響を受けない。
- 結合してもalpha2ピーク(67.1歳)近傍で連続的にサンプルがあるのはNSR-RRの上限(~76歳)までで、
  その先はFantasia高齢群(68-85歳)が補う形になる。これは望ましい相補性である一方、
  67-76歳の重複域を除けば「連続でない」年齢分布であることに変わりはない。


In [ ]:
# ============================================================
# Step 15e: 外部データセットの結合とカバレッジ確認
# ============================================================
df_fantasia_valid = df_fantasia[df_fantasia['error'].isna()].dropna(
    subset=['age', 'alpha1', 'alpha2', 'CHI']).copy()
df_fantasia_valid['dataset'] = 'fantasia'

df_nsr_valid = df_nsr[df_nsr['error'].isna()].dropna(
    subset=['age', 'alpha1', 'alpha2', 'CHI']).copy()
df_nsr_valid['dataset'] = 'nsr_rr'

keep_cols = ['record', 'age', 'sex', 'dataset', 'alpha1', 'alpha2', 'CHI']
df_ext = pd.concat([df_fantasia_valid[keep_cols], df_nsr_valid[keep_cols]], ignore_index=True)

print(f'外部結合コホート: N={len(df_ext)} '
      f'(Fantasia n={len(df_fantasia_valid)}, NSR-RR n={len(df_nsr_valid)})')
print(f'年齢範囲: {df_ext["age"].min():.1f} - {df_ext["age"].max():.1f}歳')
print(df_ext.groupby('dataset')['age'].describe()[['count', 'min', 'max']])

# 年齢分布のギャップを可視化 (連続コホートでないため、これは必須の確認)
fig, ax = plt.subplots(figsize=(8, 2.8), facecolor='#FFFFFF')
for ds, color, yv in [('nsr_rr', '#3A6EA5', 1), ('fantasia', '#D9534F', 0)]:
    sub = df_ext[df_ext['dataset'] == ds]
    ax.scatter(sub['age'], [yv] * len(sub), alpha=0.6, color=color, label=ds, s=25)
ax.set_yticks([0, 1]); ax.set_yticklabels(['fantasia', 'nsr_rr'])
ax.set_xlabel('Age (years)')
ax.set_title('外部検証コホートの年齢カバレッジ (Step 15e)')
ax.axvline(44.9, color='gray', ls=':', lw=1); ax.axvline(57.3, color='gray', ls=':', lw=1)
ax.axvline(67.1, color='gray', ls=':', lw=1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'step15e_age_coverage.png'), dpi=150)
plt.show()

ext_path = os.path.join(OUTPUT_DIR, 'external_combined_fantasia_nsr.csv')
df_ext.to_csv(ext_path, index=False)
print(f'\n保存先: {ext_path}')


## Step 15f: NSR-RR単独 — 二次vs三次モデル比較 (Step 11a相当)

Fantasiaを含めず、連続年齢デザインであるNSR-RR (n≈54) のみで実施する。
サンプルサイズが自コホート(N=1,032)の1/20程度であるため、検出力は大幅に低いことを
踏まえて解釈する必要がある。「三次項が非有意」という結果が出ても、自コホートの
結果(alpha1・alpha2で三次項必要)を否定する根拠にはならない — 検出力不足と
真に効果がないことは区別できないため。


In [ ]:
# ============================================================
# Step 15f: NSR-RR単独 — 二次vs三次モデル比較 (Step 11a相当、外部コホート)
# ============================================================
d_nsr_ext = df_nsr_valid.copy()
age_mean_nsr = d_nsr_ext['age'].mean()
d_nsr_ext['age_c'] = d_nsr_ext['age'] - age_mean_nsr
age_min_nsr, age_max_nsr = d_nsr_ext['age'].min(), d_nsr_ext['age'].max()

ext_nsr_only_results = {}
for dv in ['CHI', 'alpha1', 'alpha2']:
    f_quad = f'{dv} ~ age_c + I(age_c**2)'
    f_cubic = f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)'
    m_quad = smf.ols(f_quad, data=d_nsr_ext).fit()
    m_cubic = smf.ols(f_cubic, data=d_nsr_ext).fit()
    nested = anova_lm(m_quad, m_cubic)
    p_cubic = nested['Pr(>F)'].iloc[1]

    print(f'\n=== NSR-RR単独 {dv} (N={len(d_nsr_ext)}) ===')
    print(f'  二次AIC={m_quad.aic:.2f}, 三次AIC={m_cubic.aic:.2f}, 三次項p={p_cubic:.4f}')
    ext_nsr_only_results[dv] = {'N': len(d_nsr_ext), 'p_cubic': p_cubic,
                                 'quad_AIC': m_quad.aic, 'cubic_AIC': m_cubic.aic}

print('\n自コホート: CHI=三次不要(p=0.95) / alpha1,alpha2=三次必要、との整合性を確認。')


## Step 15g: NSR-RR単独 — ピーク年齢ペアードブートストラップ (Step 13相当)

自コホートのStep 13と全く同じロジック (二次頂点 / 三次極大のアンカー方式) を、
NSR-RR単独 (連続年齢, n≈54) に適用する。データセットの水準差を扱う必要がない分、
最も解釈が単純な外部検証である。ただし alpha2 のアンカー年齢(67.1歳)はNSR-RRの
観測上限に近いため、境界外挿のリスクを明示しながら解釈する。


In [ ]:
# ============================================================
# Step 15g: NSR-RR単独 — ピーク年齢ペアードブートストラップ (Step 13相当)
# ============================================================
def get_quad_vertex_ext(boot, dv, age_mean_b, age_lo, age_hi):
    m = smf.ols(f'{dv} ~ age_c + I(age_c**2)', data=boot).fit()
    b2 = m.params['I(age_c ** 2)']; c = m.params['age_c']
    if b2 >= 0:
        return np.nan
    vertex_age = age_mean_b - c / (2 * b2)
    return vertex_age if age_lo <= vertex_age <= age_hi else np.nan

def get_cubic_peak_ext(boot, dv, age_mean_b, anchor_age, age_lo, age_hi):
    m = smf.ols(f'{dv} ~ age_c + I(age_c**2) + I(age_c**3)', data=boot).fit()
    a = m.params['I(age_c ** 3)']; b2 = m.params['I(age_c ** 2)']; c = m.params['age_c']
    if a == 0:
        return np.nan
    roots = np.roots([3 * a, 2 * b2, c])
    roots = np.real(roots[np.isreal(roots)])
    cand = [age_mean_b + xc for xc in roots if age_lo <= age_mean_b + xc <= age_hi]
    maxima = [x for x in cand if (6 * a * (x - age_mean_b) + 2 * b2) < 0]
    if not maxima:
        return np.nan
    return min(maxima, key=lambda x: abs(x - anchor_age))

if len(d_nsr_ext) >= 30:
    N_BOOT_EXT = 2000
    anchor_chi_nsr = get_quad_vertex_ext(d_nsr_ext, 'CHI', age_mean_nsr, age_min_nsr, age_max_nsr)
    anchor_alpha1_nsr = get_cubic_peak_ext(d_nsr_ext, 'alpha1', age_mean_nsr, 57.3, age_min_nsr, age_max_nsr)
    anchor_alpha2_nsr = get_cubic_peak_ext(d_nsr_ext, 'alpha2', age_mean_nsr, 67.1, age_min_nsr, age_max_nsr)
    print(f'NSR-RR フルサンプル点推定: CHI={anchor_chi_nsr}, alpha1={anchor_alpha1_nsr}, alpha2={anchor_alpha2_nsr}')
    print(f'(注: alpha2アンカー67.1歳はNSR-RRの観測上限 {age_max_nsr:.1f}歳 に近く、外挿に近いリスクがある)')

    rng_ext = np.random.default_rng(456)
    recs = []
    for bi in range(N_BOOT_EXT):
        idx = rng_ext.integers(0, len(d_nsr_ext), len(d_nsr_ext))
        boot = d_nsr_ext.iloc[idx].reset_index(drop=True)
        age_mean_b = boot['age'].mean()
        boot['age_c'] = boot['age'] - age_mean_b
        rec = {'boot_id': bi}
        try:
            rec['CHI_peak'] = get_quad_vertex_ext(boot, 'CHI', age_mean_b, age_min_nsr, age_max_nsr)
            rec['alpha1_peak'] = get_cubic_peak_ext(boot, 'alpha1', age_mean_b, anchor_alpha1_nsr, age_min_nsr, age_max_nsr)
            rec['alpha2_peak'] = get_cubic_peak_ext(boot, 'alpha2', age_mean_b, anchor_alpha2_nsr, age_min_nsr, age_max_nsr)
        except Exception:
            rec['CHI_peak'] = rec['alpha1_peak'] = rec['alpha2_peak'] = np.nan
        recs.append(rec)

    df_boot_nsr = pd.DataFrame(recs)
    df_boot_nsr['diff_alpha1_minus_CHI'] = df_boot_nsr['alpha1_peak'] - df_boot_nsr['CHI_peak']
    df_boot_nsr['diff_alpha2_minus_alpha1'] = df_boot_nsr['alpha2_peak'] - df_boot_nsr['alpha1_peak']
    df_boot_nsr['diff_alpha2_minus_CHI'] = df_boot_nsr['alpha2_peak'] - df_boot_nsr['CHI_peak']

    n_valid_nsr = df_boot_nsr[['CHI_peak', 'alpha1_peak', 'alpha2_peak']].dropna().shape[0]
    print(f'\n=== NSR-RR単独 ペアードブートストラップ結果 (有効標本 {n_valid_nsr}/{N_BOOT_EXT}) ===')
    for col, label in [('diff_alpha1_minus_CHI', 'alpha1 - CHI'),
                        ('diff_alpha2_minus_alpha1', 'alpha2 - alpha1'),
                        ('diff_alpha2_minus_CHI', 'alpha2 - CHI')]:
        vals = df_boot_nsr[col].dropna()
        if len(vals) < 50:
            print(f'{label}: 有効標本が少なすぎるため信頼区間を報告しない (n={len(vals)})')
            continue
        med = vals.median()
        ci_lo, ci_hi = np.percentile(vals, [2.5, 97.5])
        pct_pos = (vals > 0).mean() * 100
        print(f'{label}: 中央値={med:.1f}年, 95%CI=({ci_lo:.1f}, {ci_hi:.1f}), 正方向={pct_pos:.1f}%')

    df_boot_nsr.to_csv(os.path.join(OUTPUT_DIR, 'step15g_nsr_only_paired_bootstrap.csv'), index=False)
else:
    print('NSR-RRの有効標本数が少なすぎるため、このステップはスキップします。')


## Step 15h: 結合コホート (Fantasia + NSR-RR) — データセット固定効果込みモデル

NSR-RR単独では届かないalpha2ピーク(67.1歳)より先の年齢域(68-85歳)をFantasia高齢群で
補うため、両データセットを結合する。記録条件の違いによる系統的な水準差を
`C(dataset)` (切片ダミー、年齢との交互作用なし) で吸収した上で、ピーク年齢を推定する。
リサンプリングはデータセットごとに層別 (層内で復元抽出) し、退化した標本
(片方のデータセットが極端に少ない/ゼロになる)を避ける。


In [ ]:
# ============================================================
# Step 15h: 結合コホート — データセット固定効果込みモデル + 層別ペアードブートストラップ
# ============================================================
df_ext_m = df_ext.copy()
age_mean_ext = df_ext_m['age'].mean()
df_ext_m['age_c'] = df_ext_m['age'] - age_mean_ext
age_min_ext, age_max_ext = df_ext_m['age'].min(), df_ext_m['age'].max()

print(f'結合コホート N={len(df_ext_m)}, 年齢範囲 {age_min_ext:.1f}-{age_max_ext:.1f}歳')
print(df_ext_m['dataset'].value_counts())

comb_results = {}
for dv in ['CHI', 'alpha1', 'alpha2']:
    f_quad = f'{dv} ~ C(dataset) + age_c + I(age_c**2)'
    f_cubic = f'{dv} ~ C(dataset) + age_c + I(age_c**2) + I(age_c**3)'
    m_quad = smf.ols(f_quad, data=df_ext_m).fit()
    m_cubic = smf.ols(f_cubic, data=df_ext_m).fit()
    nested = anova_lm(m_quad, m_cubic)
    p_cubic = nested['Pr(>F)'].iloc[1]
    print(f'\n=== 結合コホート {dv} ===')
    print(f'  データセット固定効果: {m_quad.params.filter(like="dataset").to_dict()}')
    print(f'  二次AIC={m_quad.aic:.2f}, 三次AIC={m_cubic.aic:.2f}, 三次項p={p_cubic:.4f}')
    comb_results[dv] = {'p_cubic': p_cubic}


def get_quad_vertex_comb(boot, dv, age_mean_b, age_lo, age_hi):
    m = smf.ols(f'{dv} ~ C(dataset) + age_c + I(age_c**2)', data=boot).fit()
    b2 = m.params['I(age_c ** 2)']; c = m.params['age_c']
    if b2 >= 0:
        return np.nan
    vertex_age = age_mean_b - c / (2 * b2)
    return vertex_age if age_lo <= vertex_age <= age_hi else np.nan


def get_cubic_peak_comb(boot, dv, age_mean_b, anchor_age, age_lo, age_hi):
    m = smf.ols(f'{dv} ~ C(dataset) + age_c + I(age_c**2) + I(age_c**3)', data=boot).fit()
    a = m.params['I(age_c ** 3)']; b2 = m.params['I(age_c ** 2)']; c = m.params['age_c']
    if a == 0:
        return np.nan
    roots = np.roots([3 * a, 2 * b2, c])
    roots = np.real(roots[np.isreal(roots)])
    cand = [age_mean_b + xc for xc in roots if age_lo <= age_mean_b + xc <= age_hi]
    maxima = [x for x in cand if (6 * a * (x - age_mean_b) + 2 * b2) < 0]
    if not maxima:
        return np.nan
    return min(maxima, key=lambda x: abs(x - anchor_age))


anchor_chi_c = get_quad_vertex_comb(df_ext_m, 'CHI', age_mean_ext, age_min_ext, age_max_ext)
anchor_alpha1_c = get_cubic_peak_comb(df_ext_m, 'alpha1', age_mean_ext, 57.3, age_min_ext, age_max_ext)
anchor_alpha2_c = get_cubic_peak_comb(df_ext_m, 'alpha2', age_mean_ext, 67.1, age_min_ext, age_max_ext)
print(f'\n結合コホート フルサンプル点推定: CHI={anchor_chi_c}, alpha1={anchor_alpha1_c}, alpha2={anchor_alpha2_c}')

N_BOOT_COMB = 2000
idx_fan = df_ext_m.index[df_ext_m['dataset'] == 'fantasia'].to_numpy()
idx_nsr = df_ext_m.index[df_ext_m['dataset'] == 'nsr_rr'].to_numpy()
rng_c = np.random.default_rng(789)

recs_c = []
for bi in range(N_BOOT_COMB):
    bi_fan = rng_c.choice(idx_fan, size=len(idx_fan), replace=True)
    bi_nsr = rng_c.choice(idx_nsr, size=len(idx_nsr), replace=True)
    boot = df_ext_m.loc[np.concatenate([bi_fan, bi_nsr])].reset_index(drop=True)
    age_mean_b = boot['age'].mean()
    boot['age_c'] = boot['age'] - age_mean_b
    rec = {'boot_id': bi}
    try:
        rec['CHI_peak'] = get_quad_vertex_comb(boot, 'CHI', age_mean_b, age_min_ext, age_max_ext)
        rec['alpha1_peak'] = get_cubic_peak_comb(boot, 'alpha1', age_mean_b, anchor_alpha1_c, age_min_ext, age_max_ext)
        rec['alpha2_peak'] = get_cubic_peak_comb(boot, 'alpha2', age_mean_b, anchor_alpha2_c, age_min_ext, age_max_ext)
    except Exception:
        rec['CHI_peak'] = rec['alpha1_peak'] = rec['alpha2_peak'] = np.nan
    recs_c.append(rec)

df_boot_comb = pd.DataFrame(recs_c)
df_boot_comb['diff_alpha1_minus_CHI'] = df_boot_comb['alpha1_peak'] - df_boot_comb['CHI_peak']
df_boot_comb['diff_alpha2_minus_alpha1'] = df_boot_comb['alpha2_peak'] - df_boot_comb['alpha1_peak']
df_boot_comb['diff_alpha2_minus_CHI'] = df_boot_comb['alpha2_peak'] - df_boot_comb['CHI_peak']

n_valid_comb = df_boot_comb[['CHI_peak', 'alpha1_peak', 'alpha2_peak']].dropna().shape[0]
print(f'\n=== 結合コホート 層別ペアードブートストラップ結果 (有効標本 {n_valid_comb}/{N_BOOT_COMB}) ===')
for col, label in [('diff_alpha1_minus_CHI', 'alpha1 - CHI'),
                    ('diff_alpha2_minus_alpha1', 'alpha2 - alpha1'),
                    ('diff_alpha2_minus_CHI', 'alpha2 - CHI')]:
    vals = df_boot_comb[col].dropna()
    if len(vals) < 50:
        print(f'{label}: 有効標本が少なすぎるため信頼区間を報告しない (n={len(vals)})')
        continue
    med = vals.median()
    ci_lo, ci_hi = np.percentile(vals, [2.5, 97.5])
    pct_pos = (vals > 0).mean() * 100
    print(f'{label}: 中央値={med:.1f}年, 95%CI=({ci_lo:.1f}, {ci_hi:.1f}), 正方向={pct_pos:.1f}%')

df_boot_comb.to_csv(os.path.join(OUTPUT_DIR, 'step15h_combined_paired_bootstrap.csv'), index=False)


## Step 15i: サマリー — 自コホート vs 外部検証 (NSR-RR単独 / 結合)

自コホート(N=1,032)の点推定・95%CIと、外部検証2通り(NSR-RR単独 / Fantasia+NSR-RR結合)を
並べ、順序(CHI<alpha1<alpha2)と分離幅(12年・10年・22年)が方向・オーダーとして
再現されるかを確認する。外部コホートはNで大きく劣るため、点推定が完全に一致することは
期待せず、**方向性が一致し、CIが自コホートの点推定を含むか**を主な判断基準とする。


In [ ]:
# ============================================================
# Step 15i: サマリー比較表
# ============================================================
def summarize_boot(df_boot, label):
    row = {'source': label}
    for col, key in [('diff_alpha1_minus_CHI', 'alpha1_minus_CHI'),
                      ('diff_alpha2_minus_alpha1', 'alpha2_minus_alpha1'),
                      ('diff_alpha2_minus_CHI', 'alpha2_minus_CHI')]:
        vals = df_boot[col].dropna()
        if len(vals) < 50:
            row[key] = (np.nan, np.nan, np.nan)
        else:
            row[key] = (vals.median(), *np.percentile(vals, [2.5, 97.5]))
    return row

summary_rows = [
    {'source': '自コホート (N=1,032)',
     'alpha1_minus_CHI': (12.0, 7.0, 16.6),
     'alpha2_minus_alpha1': (9.8, 3.5, 20.7),
     'alpha2_minus_CHI': (22.1, 12.9, 34.2)},
    summarize_boot(df_boot_nsr, f'NSR-RR単独 (N={len(d_nsr_ext)})'),
    summarize_boot(df_boot_comb, f'結合 Fantasia+NSR-RR (N={len(df_ext_m)})'),
]

print('=== 自コホート vs 外部検証 比較 ===')
for row in summary_rows:
    print(f"\n{row['source']}")
    for key, jp in [('alpha1_minus_CHI', 'alpha1 - CHI'),
                    ('alpha2_minus_alpha1', 'alpha2 - alpha1'),
                    ('alpha2_minus_CHI', 'alpha2 - CHI')]:
        med, lo, hi = row[key]
        if np.isnan(med):
            print(f'  {jp}: 推定不可 (有効標本不足)')
        else:
            print(f'  {jp}: {med:.1f}年 (95%CI {lo:.1f}-{hi:.1f})')

print('\n判定基準の例: 外部検証の中央値の符号が自コホートと一致し、')
print('かつ外部検証の95%CIが自コホートの点推定を含んでいれば「方向・規模ともに整合的」。')
print('符号は一致するがCIが自コホート点推定を含まない場合は「方向は一致するが規模は要検討」。')
print('符号が逆転する場合は、順序の頑健性そのものに疑問符がつく。')
